# Forecasting the next six hours of pickups

I forecast **completed pickups**. If someone sees a high fare or a longer wait than expected and decides against the ride, that choice never becomes a pickup in my data. The [data engineering notebook](data_engineering1_organized_v3.ipynb) explains how I built the target.

The earlier experiments showed that pickup history, calendar patterns, and events can help explain ride activity. Now I want to turn that into a forecasting problem: given the information available at the end of this hour, how well can the next six hours be predicted?

I'll start with the same hour last week, then see what the fitted models add. After choosing a model on validation data, I'll look more closely at the hours and locations where it still struggles. The goal at this stage is a working offline baseline with an honest account of its weaknesses, before connecting it to a live pipeline.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import sys
import time
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import seaborn as sns
import xgboost as xgb
from markdown import Markdown
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor, Pool

# Configuring a consistent presentation-ready chart theme.
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="deep",
    font_scale=1.0,
)

plt.rcParams.update({
    # Defining figure quality.
    "figure.figsize": (10, 8),
    "figure.dpi": 440,
    "savefig.dpi": 440,
    "savefig.bbox": "tight",

    # Formatting titles and labels.
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.titlepad": 12,

    # Showing horizontal grid lines only.
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.grid.which": "major",
    "grid.color": "#A0A9B9",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.7,

    # Simplifying chart borders.
    "axes.edgecolor": "#D1D5DB",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,

    # Improving tick and legend styling.
    "xtick.color": "#4B5563",
    "ytick.color": "#4B5563",
    "legend.frameon": False,

    # Keeping a clean white background.
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

In [ ]:



def find_project_root(start: Path) -> Path:
    # Find the project root from any directory inside the repository.
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        f"Could not find the project root from {start}"
    )


PROJECT_ROOT = find_project_root(Path.cwd())

PRODUCTION_DATA_ROOT = (
    PROJECT_ROOT
    / "data/gold/production_model/hourly_zone_horizon_features"
    / "availability_delay=7h"
)

FEATURE_MANIFEST_PATH = (
    PRODUCTION_DATA_ROOT
    / "manifests/production_model_feature_manifest.parquet"
)

SPLIT_MANIFEST_PATH = (
    PRODUCTION_DATA_ROOT
    / "manifests/production_forecast_origin_split_manifest.parquet"
)

required_paths = {
    "production dataset directory": PRODUCTION_DATA_ROOT,
    "feature manifest": FEATURE_MANIFEST_PATH,
    "split manifest": SPLIT_MANIFEST_PATH,
}

path_status = {
    name: path.exists()
    for name, path in required_paths.items()
}

print("Project root:", PROJECT_ROOT)
print("\nRequired paths:")
for name, exists in path_status.items():
    print(f"{name:<30} {exists}")

assert PRODUCTION_DATA_ROOT.is_dir(), (
    f"Production dataset directory not found: {PRODUCTION_DATA_ROOT}"
)
assert FEATURE_MANIFEST_PATH.is_file(), (
    f"Feature manifest not found: {FEATURE_MANIFEST_PATH}"
)
assert SPLIT_MANIFEST_PATH.is_file(), (
    f"Split manifest not found: {SPLIT_MANIFEST_PATH}"
)

print("\nSetup validation passed: True")

## What reaches this notebook?

The [forecasting data exploration](03b_production_dataset_eda.ipynb) showed where the weekly pattern is strong and where it breaks down. The [archived weather checks](03_archived_weather_forecast_production_updated.ipynb) explain which weather runs I allow at each forecast time.

The weather pipeline already saved the combined dataset. Before fitting anything, I want to make sure this notebook is reading that same output. Otherwise a difference in results could come from a different input table rather than a different model.


In [ ]:
partition_paths = sorted(
    PRODUCTION_DATA_ROOT.glob(
        "year=*/month=*/production_features_*.parquet"
    )
)

partition_records = []
reference_schema = None

for path in partition_paths:
    parquet_file = pq.ParquetFile(path)
    schema = parquet_file.schema_arrow

    if reference_schema is None:
        reference_schema = schema

    partition_records.append({
        "year": int(path.parent.parent.name.split("=")[1]),
        "month": int(path.parent.name.split("=")[1]),
        "rows": parquet_file.metadata.num_rows,
        "columns": len(schema),
        "row_groups": parquet_file.metadata.num_row_groups,
        "size_mb": path.stat().st_size / 1024**2,
        "schema_matches": schema.equals(reference_schema),
    })

partition_audit = pd.DataFrame(partition_records)

total_rows = int(partition_audit["rows"].sum())
total_size_mb = partition_audit["size_mb"].sum()
total_row_groups = int(partition_audit["row_groups"].sum())

print("Partitions:", len(partition_paths))
print(f"Rows: {total_rows:,}")
print("Stored columns:", len(reference_schema))
print("Row groups:", total_row_groups)
print(f"Compressed size: {total_size_mb:,.2f} MB")
print(
    "All schemas identical:",
    partition_audit["schema_matches"].all(),
)

display(
    partition_audit[
        ["year", "month", "rows", "row_groups", "size_mb"]
    ].round({"size_mb": 2})
)

assert len(partition_paths) == 23, (
    f"Expected 23 partitions, found {len(partition_paths)}."
)

assert total_rows == 26_117_478, (
    f"Unexpected row count: {total_rows:,}."
)

assert len(reference_schema) == 53, (
    f"Expected 53 columns, found {len(reference_schema)}."
)

assert partition_audit["schema_matches"].all(), (
    "The monthly partitions do not have identical schemas."
)

assert not partition_audit[
    ["year", "month"]
].duplicated().any(), (
    "More than one partition was found for a year-month."
)

print("\nProduction dataset metadata validation passed: True")

### Production Dataset Check

The dimensions match the weather pipeline: 23 monthly files, about 26.1 million rows, and 53 columns. The schemas also agree across files. I can now check which of those columns the models should actually receive.


## Which columns can I use at forecast time?

The table includes paths, timestamps, and quality fields as well as predictors. Rather than select columns from memory, I'll use the saved feature manifest. That keeps later experiments from quietly receiving different inputs.


In [ ]:
# loading only the small feature manifest, not the production dataset.
feature_manifest = pd.read_parquet(
    FEATURE_MANIFEST_PATH
)

print("Manifest rows:", len(feature_manifest))
print("Manifest columns:", feature_manifest.columns.tolist())

display(feature_manifest.head())

In [ ]:
# Defining the exact structure expected from the saved manifest.
expected_manifest_columns = {
    "column_name",
    "data_type",
    "role",
    "use_in_primary_model",
    "modeling_type",
}

# Counting how the 53 stored columns are assigned to modeling roles.
role_counts = (
    feature_manifest["role"]
    .value_counts()
    .to_dict()
)

# Checking whether the manifest describes the same columns as the Parquet files.
manifest_matches_schema = (
    set(feature_manifest["column_name"])
    == set(reference_schema.names)
)

print("Role counts:", role_counts)
print(
    "Unique column names:",
    feature_manifest["column_name"].is_unique,
)
print(
    "Manifest matches Parquet schema:",
    manifest_matches_schema,
)

assert set(feature_manifest.columns) == expected_manifest_columns
assert len(feature_manifest) == 53
assert feature_manifest["column_name"].is_unique
assert manifest_matches_schema

assert role_counts == {
    "primary_feature": 30,
    "excluded_metadata": 18,
    "key": 3,
    "sensitivity_flag": 1,
    "target": 1,
}

print("\nFeature manifest structure validation passed: True")

### Feature Manifest Check

The manifest accounts for all 53 columns, with 30 marked as model features. Zone IDs and several calendar fields are categorical, so I'll carry those roles into the model matrices instead of treating their codes as measurements.


In [ ]:
# Extracting each group directly from the saved feature contract.
primary_features = feature_manifest.loc[
    feature_manifest["use_in_primary_model"],
    "column_name",
].tolist()

categorical_features = feature_manifest.loc[
    feature_manifest["use_in_primary_model"]
    & feature_manifest["modeling_type"].eq("categorical"),
    "column_name",
].tolist()

target_columns = feature_manifest.loc[
    feature_manifest["role"].eq("target"),
    "column_name",
].tolist()

key_columns = feature_manifest.loc[
    feature_manifest["role"].eq("key"),
    "column_name",
].tolist()

sensitivity_columns = feature_manifest.loc[
    feature_manifest["role"].eq("sensitivity_flag"),
    "column_name",
].tolist()

print("Primary features:", len(primary_features))
print("Categorical features:", categorical_features)
print("Target:", target_columns)
print("Keys:", key_columns)
print("Sensitivity flags:", sensitivity_columns)

In [ ]:
# Keeping only the fields approved for the primary production model.
primary_feature_manifest = feature_manifest.loc[
    feature_manifest["use_in_primary_model"],
    [
        "column_name",
        "data_type",
        "modeling_type",
    ],
].reset_index(drop=True)

display(primary_feature_manifest)

### What happens when forecast weather is missing?

The weather list needs a closer look because the model is predicting a future hour. It should receive the selected forecast values and information about their availability, not weather observed after the outcome. I'll check those roles before moving on.


In [ ]:
# Finding every weather-related field, including excluded metadata.
weather_manifest = feature_manifest.loc[
    feature_manifest["column_name"].str.contains(
        "weather|forecast",
        case=False,
        regex=True,
    ),
    [
        "column_name",
        "role",
        "use_in_primary_model",
        "modeling_type",
    ],
].reset_index(drop=True)

display(weather_manifest)

#### Weather Missingness Decision

Both general completeness flags describe information already available in the individual weather flags. I'll keep them for checks and grouped results, but leave them out of the main feature list. More columns would not necessarily give the model more information here.


In [ ]:
# Defining the expected production-model roles.
expected_categorical_features = {
    "location_id",
    "forecast_horizon",
    "hour_of_day",
    "day_of_week",
    "month",
}

expected_key_columns = {
    "forecast_origin_utc",
    "target_hour_local",
    "target_hour_utc",
}

# Checking the feature groups created from the manifest.
assert len(primary_features) == 30

assert set(categorical_features) == (
    expected_categorical_features
)

assert target_columns == ["total_pickups"]

assert set(key_columns) == expected_key_columns

assert sensitivity_columns == [
    "is_ambiguous_dst_target"
]

# Checking that non-feature roles cannot enter the model.
assert not set(target_columns) & set(primary_features)
assert not set(key_columns) & set(primary_features)
assert not set(sensitivity_columns) & set(primary_features)

print("Primary features:", len(primary_features))
print("Categorical features:", len(categorical_features))
print("Target excluded from features: True")
print("Keys excluded from features: True")
print("Sensitivity flag excluded from features: True")
print("\nFeature-role validation passed: True")

### Could observed weather have slipped in?

The earlier exploratory dataset used historical weather for the target hour. That helped describe what happened, but it would give a future-hour forecast information it could not yet know. I'll check that those columns have not carried into this dataset.


In [ ]:
# Defining the observed weather fields used in the retrospective dataset.
observed_weather_columns = {
    "temperature_f",
    "precipitation_in",
    "rain_in",
    "snowfall_in",
    "wind_speed_mph",
}

# Defining the archived forecast fields expected in the production model.
forecast_weather_columns = {
    "forecast_temperature_f",
    "forecast_precipitation_in",
    "forecast_rain_in",
    "forecast_snowfall_in",
    "forecast_wind_speed_mph",
}

# Checking both the stored schema and the approved model features.
observed_in_schema = (
    observed_weather_columns
    & set(reference_schema.names)
)

observed_in_features = (
    observed_weather_columns
    & set(primary_features)
)

forecast_features_complete = (
    forecast_weather_columns
    <= set(primary_features)
)

print(
    "Observed weather in schema:",
    sorted(observed_in_schema),
)
print(
    "Observed weather in primary features:",
    sorted(observed_in_features),
)
print(
    "All archived forecast variables included:",
    forecast_features_complete,
)

assert not observed_in_schema
assert not observed_in_features
assert forecast_features_complete

print("\nObserved-weather leakage check passed: True")

#### Observed-Weather Check

The historical weather columns are absent, and the forecast-weather versions are in the feature list. That resolves this particular concern. The separate question of when the archived forecasts originally became available remains part of the source audit.


## Can forecasts on either side of a split share an outcome?

There is a small complication with the date split: a forecast made just before midnight can predict hours after midnight. If I split only on the origin date, training and validation could still contain the same target hours. I'll use the saved boundary purge to prevent that overlap.


In [ ]:
# Loading only the saved forecast-origin split manifest.
split_manifest = pd.read_parquet(
    SPLIT_MANIFEST_PATH
)

print("Manifest rows:", len(split_manifest))
print("Manifest columns:", split_manifest.columns.tolist())

display(split_manifest.head())

In [ ]:
# Counting origins assigned to each chronological split.
split_counts = (
    split_manifest["split"]
    .value_counts()
    .to_dict()
)

# Checking the six-hour target-window arithmetic.
target_starts_correctly = (
    split_manifest["target_start_utc"]
    == split_manifest["forecast_origin_utc"]
    + np.timedelta64(1, "h")
).all()

target_ends_correctly = (
    split_manifest["target_end_utc"]
    == split_manifest["forecast_origin_utc"]
    + np.timedelta64(6, "h")
).all()

print("Origin counts:", split_counts)
print(
    "Unique forecast origins:",
    split_manifest["forecast_origin_utc"].is_unique,
)
print(
    "Rows per origin:",
    split_manifest["rows"].unique().tolist(),
)
print(
    "Target windows begin at horizon 1:",
    target_starts_correctly,
)
print(
    "Target windows end at horizon 6:",
    target_ends_correctly,
)

assert len(split_manifest) == 16_551
assert split_manifest["forecast_origin_utc"].is_unique
assert split_manifest["rows"].eq(1_578).all()
assert target_starts_correctly
assert target_ends_correctly

assert split_counts == {
    "training": 12_208,
    "test": 2_178,
    "validation": 2_153,
    "purged_boundary": 12,
}

assert split_manifest["rows"].sum() == 26_117_478

print("\nSplit-manifest structure validation passed: True")

In [ ]:
# Separating the modeling periods from the purged boundary origins.
training_origins = split_manifest.loc[
    split_manifest["split"].eq("training")
]

validation_origins = split_manifest.loc[
    split_manifest["split"].eq("validation")
]

test_origins = split_manifest.loc[
    split_manifest["split"].eq("test")
]

purged_origins = split_manifest.loc[
    split_manifest["split"].eq("purged_boundary")
]

# Comparing the final target hour in one split with the first target hour in the next.
training_validation_separated = (
    training_origins["target_end_utc"].max()
    < validation_origins["target_start_utc"].min()
)

validation_test_separated = (
    validation_origins["target_end_utc"].max()
    < test_origins["target_start_utc"].min()
)

print(
    "Training target end:",
    training_origins["target_end_utc"].max(),
)
print(
    "Validation target start:",
    validation_origins["target_start_utc"].min(),
)
print(
    "Validation target end:",
    validation_origins["target_end_utc"].max(),
)
print(
    "Test target start:",
    test_origins["target_start_utc"].min(),
)
print("Purged origins:", len(purged_origins))
print(
    "Training and validation separated:",
    training_validation_separated,
)
print(
    "Validation and test separated:",
    validation_test_separated,
)

assert len(purged_origins) == 12
assert training_validation_separated
assert validation_test_separated

print("\nTarget-window separation validation passed: True")

### Purged Split Result

After purging, there are 12,208 training origins, 2,153 validation origins, and 2,178 test origins. The target windows no longer cross the boundaries. The 12 removed origins are the price of keeping those windows separate.


### Training, Validation, and Test Periods

I'll fit on August 2024–December 2025, make model choices on January–March 2026, and evaluate the frozen setup on April–June 2026. One qualification matters: some earlier event-feature experiments used dates in that last period. It is held out from this final tuning process, but it was not unseen throughout the project.

| Split | Forecast-origin range (UTC) | Origins | Rows | Purpose |
|---|---|---:|---:|---|
| Training | `2024-08-10 07:00` to `2025-12-31 22:00` | 12,208 | 19,264,224 | Fitting model parameters |
| Purged boundary 1 | `2025-12-31 23:00` to `2026-01-01 04:00` | 6 | 9,468 | Separating training and validation targets |
| Validation | `2026-01-01 05:00` to `2026-03-31 21:00` | 2,153 | 3,397,434 | Comparing models and making tuning decisions |
| Purged boundary 2 | `2026-03-31 22:00` to `2026-04-01 03:00` | 6 | 9,468 | Separating validation and test targets |
| Test | `2026-04-01 04:00` to `2026-06-30 21:00` | 2,178 | 3,436,884 | Final evaluation after the model design is frozen |


### Why do several forecasts share one pickup count?

The validation count initially looks short for a 90-day quarter. The reason is the UTC split boundaries and the origins removed around them. I'll count the retained origins directly, rather than assume calendar days times 24 must equal the final sample.

$$
90\text{ days}
\times
24\text{ hourly origins}
=
2{,}160\text{ origins}
$$

$$
2{,}160-7
=
2{,}153\text{ validation origins}
$$

$$
263\text{ zones}
\times
6\text{ horizons}
=
1{,}578\text{ rows per origin}
$$

$$
2{,}153
\times
263
\times
6
=
3{,}397{,}434\text{ prediction rows}
$$

$$
2{,}153
\times
263
=
566{,}239\text{ predictions}
$$


## Can I fit the training data in memory?

The files take only about 127 MB on disk, which makes the data look smaller than it will be during fitting. Once decoded, the columns and model matrices need much more memory. I'll estimate that before loading everything.


In [ ]:
# Calculating the number of dataset rows represented by each split.
split_row_counts = (
    split_manifest
    .groupby("split", observed=True)["rows"]
    .sum()
    .rename("rows")
    .to_frame()
)

split_row_counts["millions"] = (
    split_row_counts["rows"] / 1_000_000
)

split_row_counts["percent_of_dataset"] = (
    split_row_counts["rows"]
    / split_row_counts["rows"].sum()
    * 100
)

display(
    split_row_counts.round({
        "millions": 3,
        "percent_of_dataset": 2,
    })
)

assert split_row_counts.loc["training", "rows"] == 19_264_224
assert split_row_counts.loc["validation", "rows"] == 3_397_434
assert split_row_counts.loc["test", "rows"] == 3_436_884
assert split_row_counts.loc["purged_boundary", "rows"] == 18_936

print("Split row-count validation passed: True")

In [ ]:
# Including model fields and columns needed for later evaluation.
memory_sample_columns = list(dict.fromkeys(
    primary_features
    + target_columns
    + key_columns
    + sensitivity_columns
    + [
        "weather_forecast_has_missing",
        "weather_forecast_complete",
    ]
))

# Reading only part of the first row group from the first training partition.
first_parquet_file = pq.ParquetFile(
    partition_paths[0]
)

memory_sample_arrow = (
    first_parquet_file
    .read_row_group(
        0,
        columns=memory_sample_columns,
    )
    .slice(0, 100_000)
)

print("Sample rows:", memory_sample_arrow.num_rows)
print("Sample columns:", memory_sample_arrow.num_columns)
print(
    "Arrow sample size (MB):",
    round(memory_sample_arrow.nbytes / 1024**2, 2),
)

assert memory_sample_arrow.num_rows == 100_000
assert memory_sample_arrow.num_columns == len(
    memory_sample_columns
)

print("\nControlled memory sample loaded: True")

In [ ]:
# Converting the controlled Arrow sample to pandas.
memory_sample_pandas = (
    memory_sample_arrow.to_pandas()
)

# Preparing fixed category sets that will stay consistent across splits.
category_values = {
    "location_id": list(range(1, 264)),
    "forecast_horizon": list(range(1, 7)),
    "hour_of_day": list(range(24)),
    "day_of_week": list(range(7)),
    "month": list(range(1, 13)),
}

memory_sample_optimized = (
    memory_sample_pandas.copy()
)

# Converting the five manifest-approved categorical features.
for column, categories in category_values.items():
    category_type = pd.CategoricalDtype(
        categories=categories
    )
    memory_sample_optimized[column] = (
        memory_sample_optimized[column]
        .astype(category_type)
    )

raw_pandas_mb = (
    memory_sample_pandas
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

optimized_pandas_mb = (
    memory_sample_optimized
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

print("Raw pandas sample (MB):", round(raw_pandas_mb, 2))
print(
    "Categorical pandas sample (MB):",
    round(optimized_pandas_mb, 2),
)
print(
    "Optimized bytes per row:",
    round(
        memory_sample_optimized
        .memory_usage(deep=True)
        .sum()
        / len(memory_sample_optimized),
        2,
    ),
)

In [ ]:
# Measuring the model matrix separately from evaluation metadata.
model_data_columns = (
    primary_features
    + target_columns
)

model_bytes_per_row = (
    memory_sample_optimized[
        model_data_columns
    ]
    .memory_usage(
        index=False,
        deep=True,
    )
    .sum()
    / len(memory_sample_optimized)
)

evaluation_bytes_per_row = (
    memory_sample_optimized
    .memory_usage(
        index=False,
        deep=True,
    )
    .sum()
    / len(memory_sample_optimized)
)

# Estimating decoded pandas memory for each retained split.
memory_estimates = []

for split_name in [
    "training",
    "validation",
    "test",
]:
    row_count = split_row_counts.loc[
        split_name,
        "rows",
    ]

    memory_estimates.append({
        "split": split_name,
        "rows": row_count,
        "model_data_gb": (
            row_count
            * model_bytes_per_row
            / 1024**3
        ),
        "with_evaluation_columns_gb": (
            row_count
            * evaluation_bytes_per_row
            / 1024**3
        ),
    })

memory_estimates = pd.DataFrame(
    memory_estimates
)

system_memory = psutil.virtual_memory()

print(
    "Model bytes per row:",
    round(model_bytes_per_row, 2),
)
print(
    "Evaluation bytes per row:",
    round(evaluation_bytes_per_row, 2),
)
print(
    "Total system RAM (GB):",
    round(system_memory.total / 1024**3, 2),
)
print(
    "Currently available RAM (GB):",
    round(system_memory.available / 1024**3, 2),
)

display(
    memory_estimates.round({
        "model_data_gb": 2,
        "with_evaluation_columns_gb": 2,
    })
)

### Memory Estimate and Loading Decision

The sample puts the training features at roughly 1.33 GB before the model creates its own structures. That seems manageable on this machine, but keeping unnecessary copies could still become expensive. I'll load the feature and evaluation columns separately and release temporary objects along the way.


## Can I train XGBoost on this machine?

Before relying on GPU training, I'll check the XGBoost build in this kernel. The GPU worked in an earlier notebook, but that does not tell me which build this environment is using now.


In [ ]:
# Reading the capabilities reported by the installed XGBoost build.
xgb_build_info = xgb.build_info()

xgb_cuda_enabled = bool(
    xgb_build_info.get("USE_CUDA", False)
)

xgb_cuda_version = xgb_build_info.get(
    "CUDA_VERSION",
    "Not reported",
)

# Asking the NVIDIA driver for the current GPU and memory status.
gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
    check=False,
)

gpu_detected = (
    gpu_query.returncode == 0
    and bool(gpu_query.stdout.strip())
)

xgb_device = (
    "cuda"
    if xgb_cuda_enabled and gpu_detected
    else "cpu"
)

print("XGBoost version:", xgb.__version__)
print("XGBoost built with CUDA:", xgb_cuda_enabled)
print("XGBoost CUDA version:", xgb_cuda_version)
print("NVIDIA GPU detected:", gpu_detected)

if gpu_detected:
    print(
        "GPU, total MB, free MB, driver:",
        gpu_query.stdout.strip(),
    )
else:
    print(
        "NVIDIA query error:",
        gpu_query.stderr.strip(),
    )

print("Selected XGBoost device:", xgb_device)

assert xgb_device in {"cuda", "cpu"}

print("\nXGBoost environment check passed: True")

### XGBoost Environment Result

The saved check finds CUDA support and the RTX 4080 Laptop GPU. That gives me a practical route for the larger fits. The reported free memory belongs to this run, so it should not be treated as a fixed requirement or a promise of the same runtime elsewhere.


## How far can last week's pickups take me?

Before training a model, I want to see how far a simple weekly pattern can take me. If the target is Monday at 8 AM, the starting prediction is the pickup count from the same zone last Monday at 8 AM.

The earlier analysis gives a reason to try this: the weekly lag follows pickups more closely than the daily lag. But it will also repeat last week's unusual conditions. If there was a concert last Monday, that higher count becomes this Monday's forecast even when no concert is scheduled.

So this baseline gives the fitted models something useful to beat. Next, I'll check its errors on the validation period and see where repeating last week works—and where it falls short.

$$
\widehat{y}_{T+h}
=
y_{T+h-168}
$$


In [ ]:
def calculate_forecast_metrics(actual, prediction):
    # Converting inputs to consistent numerical arrays.
    actual = np.asarray(
        actual,
        dtype=np.float64,
    )
    prediction = np.asarray(
        prediction,
        dtype=np.float64,
    )

    # Checking the inputs before calculating any metrics.
    assert actual.shape == prediction.shape
    assert np.isfinite(actual).all()
    assert np.isfinite(prediction).all()

    errors = prediction - actual
    absolute_errors = np.abs(errors)

    actual_total = actual.sum()
    total_sum_squares = np.sum(
        (actual - actual.mean()) ** 2
    )

    # Returning undefined ratio metrics as NaN for constant or zero-demand groups.
    wape_percent = (
        absolute_errors.sum()
        / actual_total
        * 100
        if actual_total > 0
        else np.nan
    )

    r_squared = (
        1
        - np.sum(errors**2)
        / total_sum_squares
        if total_sum_squares > 0
        else np.nan
    )

    return {
        "mae": absolute_errors.mean(),
        "rmse": np.sqrt(
            np.mean(errors**2)
        ),
        "wape_percent": wape_percent,
        "mean_bias": errors.mean(),
        "r_squared": r_squared,
    }



# Checking the function with a small result that can be verified by hand.
metric_test = calculate_forecast_metrics(
    actual=[1, 2, 3],
    prediction=[1, 2, 4],
)

assert np.isclose(metric_test["mae"], 1 / 3)
assert np.isclose(
    metric_test["rmse"],
    np.sqrt(1 / 3),
)
assert np.isclose(
    metric_test["wape_percent"],
    100 / 6,
)
assert np.isclose(metric_test["mean_bias"], 1 / 3)
assert np.isclose(metric_test["r_squared"], 0.5)

print("Metric function validation passed: True")

In [ ]:
def compare_forecast_models(
    actual,
    model_predictions,
    model_column="model",
):
    """
    Compare multiple forecast arrays against the same observations.

    Parameters
    ----------
    actual : array-like
        Observed demand values shared by every model.

    model_predictions : dict[str, array-like]
        Mapping from readable model names to aligned predictions.

    model_column : str, default="model"
        Name used for the model-label column.

    Returns
    -------
    pandas.DataFrame
        One row per model with the standard forecast metrics.
    """

    # Converting the observed values to a consistent array.
    actual = np.asarray(
        actual,
        dtype=np.float64,
    )

    if actual.ndim != 1:
        raise ValueError(
            "`actual` must be one-dimensional."
        )

    if len(actual) == 0:
        raise ValueError(
            "`actual` contains no rows."
        )

    if not model_predictions:
        raise ValueError(
            "At least one prediction array is required."
        )

    if not isinstance(model_column, str) or not model_column:
        raise ValueError(
            "`model_column` must be a nonempty string."
        )

    comparison_rows = []

    # Calculating the same metrics for every aligned model.
    for model_name, predictions in (
        model_predictions.items()
    ):
        if not isinstance(model_name, str) or not model_name:
            raise ValueError(
                "Every model must have a readable name."
            )

        predictions = np.asarray(
            predictions,
            dtype=np.float64,
        )

        if predictions.shape != actual.shape:
            raise ValueError(
                f"Prediction shape for {model_name!r} "
                "does not match the observed values."
            )

        model_metrics = calculate_forecast_metrics(
            actual=actual,
            prediction=predictions,
        )

        comparison_rows.append({
            model_column: model_name,
            **model_metrics,
        })

    comparison_results = pd.DataFrame(
        comparison_rows
    )

    # Checking that every supplied model appears exactly once.
    assert len(comparison_results) == len(
        model_predictions
    )

    assert comparison_results[
        model_column
    ].is_unique

    return comparison_results

In [ ]:
def calculate_metrics_by_group(
    evaluation_frame,
    group_columns,
    prediction_columns,
    actual_column="total_pickups",
):
    """
    Calculate forecast metrics for multiple models within each group.

    Parameters
    ----------
    evaluation_frame : pandas.DataFrame
        Evaluation data containing actual and predicted values.

    group_columns : str or list[str]
        Column or columns defining the evaluation groups.

    prediction_columns : dict[str, str]
        Mapping from readable model names to prediction columns.

    actual_column : str, default="total_pickups"
        Column containing observed demand.

    Returns
    -------
    pandas.DataFrame
        One row per group and model.
    """

    # Standardizing a single grouping column to a list.
    if isinstance(group_columns, str):
        group_columns = [group_columns]

    group_columns = list(group_columns)

    if not group_columns:
        raise ValueError(
            "At least one grouping column is required."
        )

    if not prediction_columns:
        raise ValueError(
            "At least one prediction column is required."
        )

    required_columns = [
        actual_column,
        *group_columns,
        *prediction_columns.values(),
    ]

    missing_columns = sorted(
        set(required_columns)
        - set(evaluation_frame.columns)
    )

    if missing_columns:
        raise KeyError(
            "Evaluation columns are missing: "
            f"{missing_columns}"
        )

    if evaluation_frame.empty:
        raise ValueError(
            "The evaluation frame contains no rows."
        )

    # Rejecting missing actual or prediction values.
    metric_columns = [
        actual_column,
        *prediction_columns.values(),
    ]

    if evaluation_frame[
        metric_columns
    ].isna().any().any():
        raise ValueError(
            "Actual or prediction columns contain "
            "missing values."
        )

    grouped_metric_rows = []

    # Avoiding a one-column groupby warning.
    if len(group_columns) == 1:
        grouped_evaluation = evaluation_frame.groupby(
            group_columns[0],
            observed=True,
            dropna=False,
            sort=False,
        )
    else:
        grouped_evaluation = evaluation_frame.groupby(
            group_columns,
            observed=True,
            dropna=False,
            sort=False,
        )

    for group_key, group_frame in grouped_evaluation:
        if len(group_columns) == 1:
            group_values = (group_key,)
        else:
            group_values = tuple(group_key)

        group_identifiers = dict(
            zip(
                group_columns,
                group_values,
            )
        )

        for model_name, prediction_column in (
            prediction_columns.items()
        ):
            group_metrics = calculate_forecast_metrics(
                actual=group_frame[actual_column],
                prediction=group_frame[
                    prediction_column
                ],
            )

            grouped_metric_rows.append({
                **group_identifiers,
                "Model": model_name,
                "Rows": len(group_frame),
                **group_metrics,
            })

    grouped_metrics = pd.DataFrame(
        grouped_metric_rows
    )

    expected_rows = (
        evaluation_frame[
            group_columns
        ]
        .drop_duplicates()
        .shape[0]
        * len(prediction_columns)
    )

    assert len(grouped_metrics) == expected_rows
    assert grouped_metrics["Rows"].gt(0).all()

    return grouped_metrics




In [ ]:
def build_quantile_matrix_pair(
    training_features,
    training_target,
    evaluation_features,
    evaluation_target,
    max_bin,
    enable_categorical=True,
):
    """
    Build leakage-safe paired XGBoost QuantileDMatrix objects.

    The evaluation matrix reuses quantization cuts learned only
    from the training matrix through XGBoost's `ref` argument.
    """

    if not isinstance(
        training_features,
        pd.DataFrame,
    ):
        raise TypeError(
            "`training_features` must be a DataFrame."
        )

    if not isinstance(
        evaluation_features,
        pd.DataFrame,
    ):
        raise TypeError(
            "`evaluation_features` must be a DataFrame."
        )

    if list(training_features.columns) != list(
        evaluation_features.columns
    ):
        raise ValueError(
            "Training and evaluation feature columns "
            "must match exactly and remain in the same order."
        )

    if training_features.shape[1] == 0:
        raise ValueError(
            "At least one model feature is required."
        )

    if len(training_features) != len(
        training_target
    ):
        raise ValueError(
            "Training features and target have "
            "different row counts."
        )

    if len(evaluation_features) != len(
        evaluation_target
    ):
        raise ValueError(
            "Evaluation features and target have "
            "different row counts."
        )

    if int(max_bin) <= 1:
        raise ValueError(
            "`max_bin` must be greater than one."
        )

    matrix_start = time.perf_counter()

    # Learning quantization cuts from training data only.
    training_matrix = xgb.QuantileDMatrix(
        training_features,
        label=training_target,
        max_bin=int(max_bin),
        enable_categorical=enable_categorical,
        missing=np.nan,
    )

    # Reusing training-period cuts for later evaluation data.
    evaluation_matrix = xgb.QuantileDMatrix(
        evaluation_features,
        label=evaluation_target,
        ref=training_matrix,
        max_bin=int(max_bin),
        enable_categorical=enable_categorical,
        missing=np.nan,
    )

    matrix_seconds = (
        time.perf_counter()
        - matrix_start
    )

    expected_feature_count = (
        training_features.shape[1]
    )

    assert training_matrix.num_row() == len(
        training_features
    )

    assert evaluation_matrix.num_row() == len(
        evaluation_features
    )

    assert (
        training_matrix.num_col()
        == evaluation_matrix.num_col()
        == expected_feature_count
    )

    assert (
        training_matrix.feature_names
        == evaluation_matrix.feature_names
    )

    return (
        training_matrix,
        evaluation_matrix,
        matrix_seconds,
    )



In [ ]:
def train_xgboost_candidate(
    parameters,
    training_matrix,
    evaluation_matrices,
    num_boost_round,
    early_stopping_rounds=None,
    verbose_eval=50,
):
    """
    Train one XGBoost candidate with consistent validation.

    Parameters
    ----------
    parameters : dict
        Complete XGBoost parameter dictionary.

    training_matrix : xgboost.DMatrix
        Matrix used to fit the model.

    evaluation_matrices : list[tuple]
        Ordered ``(matrix, label)`` evaluation pairs. When early
        stopping is enabled, XGBoost monitors the final pair.

    num_boost_round : int
        Maximum number of boosting rounds.

    early_stopping_rounds : int or None
        Number of rounds without improvement before stopping.

    verbose_eval : int or bool
        Frequency of XGBoost training output.

    Returns
    -------
    model : xgboost.Booster
        Fitted model.

    history : dict
        Training and evaluation metric history.

    elapsed_seconds : float
        Wall-clock training time.
    """

    if not isinstance(parameters, dict):
        raise TypeError(
            "`parameters` must be a dictionary."
        )

    if not evaluation_matrices:
        raise ValueError(
            "At least one evaluation matrix is required."
        )

    if evaluation_matrices[0][0] is not training_matrix:
        raise ValueError(
            "The training matrix must be the first "
            "evaluation matrix."
        )

    evaluation_names = [
        evaluation_name
        for _, evaluation_name
        in evaluation_matrices
    ]

    if len(evaluation_names) != len(
        set(evaluation_names)
    ):
        raise ValueError(
            "Evaluation names must be unique."
        )

    if (
        early_stopping_rounds is not None
        and len(evaluation_matrices) < 2
    ):
        raise ValueError(
            "Early stopping requires a separate "
            "validation matrix."
        )

    if int(num_boost_round) <= 0:
        raise ValueError(
            "`num_boost_round` must be positive."
        )

    training_history = {}
    training_start = time.perf_counter()

    model = xgb.train(
        params=parameters,
        dtrain=training_matrix,
        num_boost_round=int(num_boost_round),
        evals=evaluation_matrices,
        evals_result=training_history,
        early_stopping_rounds=(
            early_stopping_rounds
        ),
        verbose_eval=verbose_eval,
    )

    elapsed_seconds = (
        time.perf_counter()
        - training_start
    )

    assert model.num_boosted_rounds() > 0
    assert (
        model.num_boosted_rounds()
        <= int(num_boost_round)
    )

    assert set(training_history) == set(
        evaluation_names
    )

    for evaluation_name in evaluation_names:
        assert training_history[
            evaluation_name
        ]

    if early_stopping_rounds is not None:
        assert model.best_iteration >= 0
        assert np.isfinite(
            float(model.best_score)
        )

    return (
        model,
        training_history,
        elapsed_seconds,
    )



In [ ]:
def predict_xgboost_demand(
    model,
    prediction_matrix,
    iteration_end=None,
):
    """
    Generate raw and nonnegative XGBoost demand forecasts.

    Parameters
    ----------
    model : xgboost.Booster
        Fitted XGBoost model.

    prediction_matrix : xgboost.DMatrix
        Matrix used for prediction.

    iteration_end : int or None
        Exclusive final boosting round. Use ``best_iteration + 1``
        for early-stopped candidates. Use None for every tree in a
        frozen final model.

    Returns
    -------
    raw_predictions : numpy.ndarray
        Predictions before the nonnegative constraint.

    constrained_predictions : numpy.ndarray
        Predictions clipped at zero.

    negative_prediction_count : int
        Number of negative raw predictions.

    prediction_seconds : float
        Wall-clock inference time.
    """

    if iteration_end is not None:
        iteration_end = int(iteration_end)

        if not (
            0 < iteration_end
            <= model.num_boosted_rounds()
        ):
            raise ValueError(
                "`iteration_end` must fall within "
                "the fitted boosting rounds."
            )

    prediction_start = time.perf_counter()

    if iteration_end is None:
        raw_predictions = model.predict(
            prediction_matrix
        )
    else:
        raw_predictions = model.predict(
            prediction_matrix,
            iteration_range=(
                0,
                iteration_end,
            ),
        )

    prediction_seconds = (
        time.perf_counter()
        - prediction_start
    )

    raw_predictions = np.asarray(
        raw_predictions,
        dtype=np.float32,
    )

    if len(raw_predictions) != (
        prediction_matrix.num_row()
    ):
        raise AssertionError(
            "Prediction and matrix row counts differ."
        )

    if not np.isfinite(
        raw_predictions
    ).all():
        raise ValueError(
            "XGBoost produced nonfinite predictions."
        )

    negative_prediction_count = int(
        np.count_nonzero(
            raw_predictions < 0
        )
    )

    constrained_predictions = np.clip(
        raw_predictions,
        a_min=0,
        a_max=None,
    )

    assert (
        constrained_predictions >= 0
    ).all()

    return (
        raw_predictions,
        constrained_predictions,
        negative_prediction_count,
        prediction_seconds,
    )




### Loading Only the Validation Baseline Columns

I don't need the full model matrix to score the weekly baseline. The actual count, weekly lag, and alignment keys are enough. I'll start with that smaller table and make sure the right predictions are being compared with the right hours.


In [ ]:
# Selecting only the three monthly files that contain validation origins.
validation_partition_paths = [
    path
    for path in partition_paths
    if path.parent.parent.name == "year=2026"
    and path.parent.name in {
        "month=01",
        "month=02",
        "month=03",
    }
]

validation_start = (
    validation_origins["forecast_origin_utc"].min()
)

validation_end = (
    validation_origins["forecast_origin_utc"].max()
)

baseline_columns = [
    "forecast_origin_utc",
    "target_hour_utc",
    "location_id",
    "forecast_horizon",
    "total_pickups",
    "pickup_lag_168",
]

# Building a dataset from validation-month files only.
validation_dataset = ds.dataset(
    [
        str(path)
        for path in validation_partition_paths
    ],
    format="parquet",
)

# Filtering to the retained validation origins during the scan.
validation_filter = (
    (
        ds.field("forecast_origin_utc")
        >= validation_start.to_pydatetime()
    )
    & (
        ds.field("forecast_origin_utc")
        <= validation_end.to_pydatetime()
    )
)

validation_baseline_arrow = (
    validation_dataset.to_table(
        columns=baseline_columns,
        filter=validation_filter,
    )
)

print(
    "Validation partitions:",
    len(validation_partition_paths),
)
print(
    "Validation rows:",
    f"{validation_baseline_arrow.num_rows:,}",
)
print(
    "Loaded columns:",
    validation_baseline_arrow.num_columns,
)
print(
    "Arrow size (MB):",
    round(
        validation_baseline_arrow.nbytes
        / 1024**2,
        2,
    ),
)

assert len(validation_partition_paths) == 3
assert validation_baseline_arrow.num_rows == 3_397_434
assert validation_baseline_arrow.num_columns == 6

print("\nFiltered validation data loaded: True")

In [ ]:
# Converting only the baseline validation columns to pandas.
validation_baseline = (
    validation_baseline_arrow.to_pandas()
)

print(
    "Validation shape:",
    validation_baseline.shape,
)
print(
    "Pandas size (MB):",
    round(
        validation_baseline
        .memory_usage(deep=True)
        .sum()
        / 1024**2,
        2,
    ),
)

print("\nColumn types:")
print(validation_baseline.dtypes)

assert validation_baseline.shape == (
    3_397_434,
    6,
)

print("\nValidation conversion passed: True")

In [ ]:
# Counting the expected origin, horizon, and zone structure.
rows_per_origin = (
    validation_baseline
    .groupby(
        "forecast_origin_utc",
        observed=True,
    )
    .size()
)

rows_per_horizon = (
    validation_baseline[
        "forecast_horizon"
    ]
    .value_counts()
    .sort_index()
)

duplicate_keys = (
    validation_baseline
    .duplicated([
        "forecast_origin_utc",
        "forecast_horizon",
        "location_id",
    ])
    .sum()
)

# Checking that each target hour matches its origin and horizon.
expected_target_hours = (
    validation_baseline["forecast_origin_utc"]
    + pd.to_timedelta(
        validation_baseline[
            "forecast_horizon"
        ].astype("int64"),
        unit="h",
    )
)

target_alignment_valid = (
    validation_baseline["target_hour_utc"]
    == expected_target_hours
).all()

print(
    "Forecast origins:",
    validation_baseline[
        "forecast_origin_utc"
    ].nunique(),
)
print(
    "Location IDs:",
    validation_baseline[
        "location_id"
    ].nunique(),
)
print(
    "Rows per horizon:",
    rows_per_horizon.to_dict(),
)
print(
    "Rows per origin range:",
    (
        int(rows_per_origin.min()),
        int(rows_per_origin.max()),
    ),
)
print("Duplicate keys:", duplicate_keys)
print(
    "Missing targets:",
    validation_baseline[
        "total_pickups"
    ].isna().sum(),
)
print(
    "Missing weekly lags:",
    validation_baseline[
        "pickup_lag_168"
    ].isna().sum(),
)
print(
    "Target alignment valid:",
    target_alignment_valid,
)

assert validation_baseline[
    "forecast_origin_utc"
].nunique() == 2_153

assert set(
    validation_baseline["location_id"].unique()
) == set(range(1, 264))

assert rows_per_horizon.eq(566_239).all()
assert rows_per_origin.eq(1_578).all()
assert duplicate_keys == 0

assert validation_baseline[
    ["total_pickups", "pickup_lag_168"]
].notna().all().all()

assert target_alignment_valid

print("\nValidation baseline structure passed: True")

In [ ]:
# Using demand from the same zone and hour one week earlier.
validation_baseline[
    "seasonal_naive_prediction"
] = validation_baseline[
    "pickup_lag_168"
].astype("float32")

# Calculating the overall validation metrics.
baseline_overall_metrics = (
    calculate_forecast_metrics(
        actual=validation_baseline[
            "total_pickups"
        ],
        prediction=validation_baseline[
            "seasonal_naive_prediction"
        ],
    )
)

baseline_overall_results = pd.DataFrame([
    {
        "model": "Weekly Seasonal Naive",
        "scope": "Overall",
        **baseline_overall_metrics,
    }
])

display(
    baseline_overall_results.round(3)
)

assert (
    validation_baseline[
        "seasonal_naive_prediction"
    ]
    >= 0
).all()

print("Overall baseline evaluation passed: True")

In [ ]:
# Calculating the seasonal-naive result for each forecast horizon.
baseline_horizon_records = []

for horizon, horizon_data in (
    validation_baseline
    .groupby(
        "forecast_horizon",
        observed=True,
        sort=True,
    )
):
    horizon_metrics = (
        calculate_forecast_metrics(
            actual=horizon_data[
                "total_pickups"
            ],
            prediction=horizon_data[
                "seasonal_naive_prediction"
            ],
        )
    )

    baseline_horizon_records.append({
        "model": "Weekly Seasonal Naive",
        "forecast_horizon": int(horizon),
        "rows": len(horizon_data),
        **horizon_metrics,
    })

baseline_horizon_results = pd.DataFrame(
    baseline_horizon_records
)

display(
    baseline_horizon_results.round(3)
)

assert len(baseline_horizon_results) == 6
assert baseline_horizon_results[
    "rows"
].eq(566_239).all()

print("Horizon baseline evaluation passed: True")

### What does repeating last week get wrong?

Repeating last week misses by about 22.9 pickups on average in absolute terms. RMSE is much higher at 48.3, which points to some larger mistakes. Its mean bias is only -1.2, but positive and negative misses can cancel. I'll use the absolute errors as the main starting comparison.


## Giving every model the same rows

Now the fitted models need the full feature set. I'll keep their inputs separate from the columns used to explain errors later. The zone and target-hour keys are useful for evaluation, but that does not mean every identifying field belongs in the predictor matrix.


In [ ]:
# Separating model inputs from evaluation-only fields.
model_input_columns = primary_features.copy()
target_column = target_columns[0]

diagnostic_columns = [
    "weather_forecast_has_missing",
    "weather_forecast_complete",
]

evaluation_key_columns = (
    key_columns
    + sensitivity_columns
    + diagnostic_columns
)

# Combining the columns once while preserving their order.
model_loading_columns = list(dict.fromkeys(
    model_input_columns
    + [target_column]
    + evaluation_key_columns
))

# Checking the loading contract against the saved schema.
missing_loading_columns = (
    set(model_loading_columns)
    - set(reference_schema.names)
)

model_evaluation_overlap = (
    set(model_input_columns)
    & set(evaluation_key_columns)
)

print("Model input columns:", len(model_input_columns))
print("Total columns to load:", len(model_loading_columns))
print(
    "Missing loading columns:",
    sorted(missing_loading_columns),
)
print(
    "Model/evaluation overlap:",
    sorted(model_evaluation_overlap),
)
print("Target column:", target_column)
print(
    "Evaluation-only columns:",
    evaluation_key_columns,
)

assert len(model_input_columns) == 30
assert len(model_loading_columns) == 37
assert not missing_loading_columns
assert not model_evaluation_overlap
assert target_column not in model_input_columns

print("\nModel loading contract passed: True")

In [ ]:
# Selecting monthly partitions that contain retained training origins.
training_partition_paths = [
    path
    for path in partition_paths
    if (
        path.parent.parent.name == "year=2024"
        and path.parent.name >= "month=08"
    )
    or path.parent.parent.name == "year=2025"
]

training_start = (
    training_origins["forecast_origin_utc"].min()
)

training_end = (
    training_origins["forecast_origin_utc"].max()
)

# Building a dataset from training-period files only.
training_dataset = ds.dataset(
    [
        str(path)
        for path in training_partition_paths
    ],
    format="parquet",
)

training_filter = (
    (
        ds.field("forecast_origin_utc")
        >= training_start.to_pydatetime()
    )
    & (
        ds.field("forecast_origin_utc")
        <= training_end.to_pydatetime()
    )
)

# Reading only the approved model and evaluation columns.
load_start_time = time.perf_counter()

training_arrow = training_dataset.to_table(
    columns=model_loading_columns,
    filter=training_filter,
)

training_load_seconds = (
    time.perf_counter()
    - load_start_time
)

print(
    "Training partitions:",
    len(training_partition_paths),
)
print(
    "Training rows:",
    f"{training_arrow.num_rows:,}",
)
print(
    "Loaded columns:",
    training_arrow.num_columns,
)
print(
    "Arrow size (GB):",
    round(training_arrow.nbytes / 1024**3, 2),
)
print(
    "Load time (seconds):",
    round(training_load_seconds, 2),
)

assert len(training_partition_paths) == 17
assert training_arrow.num_rows == 19_264_224
assert training_arrow.num_columns == 37

print("\nFiltered training data loaded: True")

In [ ]:
# Reading the full validation column set with the same origin filter.
validation_load_start = time.perf_counter()

validation_arrow = (
    validation_dataset.to_table(
        columns=model_loading_columns,
        filter=validation_filter,
    )
)

validation_load_seconds = (
    time.perf_counter()
    - validation_load_start
)

print(
    "Validation rows:",
    f"{validation_arrow.num_rows:,}",
)
print(
    "Loaded columns:",
    validation_arrow.num_columns,
)
print(
    "Arrow size (GB):",
    round(validation_arrow.nbytes / 1024**3, 2),
)
print(
    "Load time (seconds):",
    round(validation_load_seconds, 2),
)

assert validation_arrow.num_rows == 3_397_434
assert validation_arrow.num_columns == 37

print("\nFull validation data loaded: True")

In [ ]:
# Counting Arrow nulls without creating pandas copies.
training_null_counts = {
    column: training_arrow[column].null_count
    for column in model_loading_columns
}

validation_null_counts = {
    column: validation_arrow[column].null_count
    for column in model_loading_columns
}

training_missing = {
    column: count
    for column, count in training_null_counts.items()
    if count > 0
}

validation_missing = {
    column: count
    for column, count in validation_null_counts.items()
    if count > 0
}

# Checking that only archived forecast values are allowed to be missing.
unexpected_training_missing = (
    set(training_missing)
    - forecast_weather_columns
)

unexpected_validation_missing = (
    set(validation_missing)
    - forecast_weather_columns
)

print("Training missing values:")
print(training_missing)

print("\nValidation missing values:")
print(validation_missing)

print(
    "\nUnexpected training columns:",
    sorted(unexpected_training_missing),
)
print(
    "Unexpected validation columns:",
    sorted(unexpected_validation_missing),
)

assert not unexpected_training_missing
assert not unexpected_validation_missing

assert training_null_counts[target_column] == 0
assert validation_null_counts[target_column] == 0

assert all(
    training_null_counts[column] == 0
    for column in [
        "pickup_lag_24",
        "pickup_lag_168",
        "pickup_lag_336",
    ]
)

assert all(
    validation_null_counts[column] == 0
    for column in [
        "pickup_lag_24",
        "pickup_lag_168",
        "pickup_lag_336",
    ]
)

print("\nSplit missingness validation passed: True")

In [ ]:
# Pairing each weather variable with its availability flag.
weather_flag_pairs = [
    (
        "forecast_temperature_f",
        "temperature_forecast_available",
    ),
    (
        "forecast_precipitation_in",
        "precipitation_forecast_available",
    ),
    (
        "forecast_rain_in",
        "rain_forecast_available",
    ),
    (
        "forecast_snowfall_in",
        "snowfall_forecast_available",
    ),
    (
        "forecast_wind_speed_mph",
        "wind_forecast_available",
    ),
]

weather_flag_records = []

# Checking the variable-specific flags in both retained splits.
for split_name, arrow_table in {
    "training": training_arrow,
    "validation": validation_arrow,
}.items():

    for weather_column, flag_column in weather_flag_pairs:
        variable_available = pc.invert(
            pc.is_null(
                arrow_table[weather_column]
            )
        )

        flag_matches = pc.all(
            pc.equal(
                arrow_table[flag_column],
                variable_available,
            )
        ).as_py()

        weather_flag_records.append({
            "split": split_name,
            "weather_variable": weather_column,
            "flag_matches": flag_matches,
        })

weather_flag_validation = pd.DataFrame(
    weather_flag_records
)

display(weather_flag_validation)

assert weather_flag_validation[
    "flag_matches"
].all()

print(
    "Variable-specific weather flags passed: True"
)

In [ ]:
general_weather_records = []

# Checking whether each row has any missing weather value.
for split_name, arrow_table in {
    "training": training_arrow,
    "validation": validation_arrow,
}.items():

    weather_columns = [
        pair[0]
        for pair in weather_flag_pairs
    ]

    missing_any = pc.is_null(
        arrow_table[
            weather_columns[0]
        ]
    )

    for weather_column in weather_columns[1:]:
        missing_any = pc.or_kleene(
            missing_any,
            pc.is_null(
                arrow_table[weather_column]
            ),
        )

    missing_flag_matches = pc.all(
        pc.equal(
            arrow_table[
                "weather_forecast_has_missing"
            ],
            missing_any,
        )
    ).as_py()

    complete_flag_matches = pc.all(
        pc.equal(
            arrow_table[
                "weather_forecast_complete"
            ],
            pc.invert(missing_any),
        )
    ).as_py()

    incomplete_rows = pc.sum(
        pc.cast(
            missing_any,
            pa.int64(),
        )
    ).as_py()

    general_weather_records.append({
        "split": split_name,
        "incomplete_rows": incomplete_rows,
        "missing_flag_matches": (
            missing_flag_matches
        ),
        "complete_flag_matches": (
            complete_flag_matches
        ),
    })

general_weather_validation = pd.DataFrame(
    general_weather_records
)

display(general_weather_validation)

assert general_weather_validation[
    "missing_flag_matches"
].all()

assert general_weather_validation[
    "complete_flag_matches"
].all()

print("General weather flags passed: True")

### Keeping missing weather visible

The incomplete-weather rows are still present in training, while validation has none. I'll let models that support `NaN` handle those values directly. This also tells me something about the evaluation: good validation results alone will not show how the model behaves through a weather-data outage.


## Preparing the Model Data

The loaded tables are ready to become model matrices. I'll keep the feature order and categorical types consistent across periods, then retain a separate view for scoring by hour, horizon, and location.


In [ ]:
training_row_count = training_arrow.num_rows

# Converting only the approved model features.
X_train = training_arrow.select(
    model_input_columns
).to_pandas(
    split_blocks=True
)

# Keeping the target separate from the feature matrix.
y_train = training_arrow[
    target_column
].to_pandas()

assert X_train.shape == (
    training_row_count,
    len(model_input_columns),
)

assert len(y_train) == training_row_count

assert list(X_train.columns) == model_input_columns

assert y_train.name == target_column

training_matrix_mb = (
    X_train.memory_usage(deep=True).sum()
    / 1024**2
)

training_target_mb = (
    y_train.memory_usage(deep=True)
    / 1024**2
)

print(f"Training matrix: {X_train.shape}")
print(f"Training target rows: {len(y_train):,}")
print(f"Feature matrix size (MB): {training_matrix_mb:,.2f}")
print(f"Target size (MB): {training_target_mb:,.2f}")
print()
print("Training conversion passed: True")

In [ ]:
validation_row_count = validation_arrow.num_rows

# Converting validation features in the same order as training.
X_validation = validation_arrow.select(
    model_input_columns
).to_pandas(
    split_blocks=True
)

# Keeping the validation target separate.
y_validation = validation_arrow[
    target_column
].to_pandas()

assert X_validation.shape == (
    validation_row_count,
    len(model_input_columns),
)

assert len(y_validation) == validation_row_count

assert list(X_validation.columns) == list(
    X_train.columns
)

assert y_validation.name == target_column

validation_matrix_mb = (
    X_validation.memory_usage(deep=True).sum()
    / 1024**2
)

validation_target_mb = (
    y_validation.memory_usage(deep=True)
    / 1024**2
)

print(f"Validation matrix: {X_validation.shape}")
print(
    "Validation target rows: "
    f"{len(y_validation):,}"
)
print(
    "Feature matrix size (MB): "
    f"{validation_matrix_mb:,.2f}"
)
print(
    "Target size (MB): "
    f"{validation_target_mb:,.2f}"
)
print()
print("Validation conversion passed: True")

In [ ]:
# Defining fields kept only for later error analysis.
evaluation_only_columns = [
    "forecast_origin_utc",
    "target_hour_local",
    "target_hour_utc",
    "is_ambiguous_dst_target",
    "weather_forecast_has_missing",
    "weather_forecast_complete",
]

# Keeping diagnostic fields outside the model matrix.
validation_evaluation = validation_arrow.select(
    evaluation_only_columns
).to_pandas(
    split_blocks=True
)

assert validation_evaluation.shape == (
    validation_row_count,
    len(evaluation_only_columns),
)

assert list(
    validation_evaluation.columns
) == evaluation_only_columns

assert not set(
    validation_evaluation.columns
).intersection(
    X_validation.columns
)

validation_evaluation_mb = (
    validation_evaluation
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

print(
    "Evaluation frame: "
    f"{validation_evaluation.shape}"
)
print(
    "Evaluation frame size (MB): "
    f"{validation_evaluation_mb:,.2f}"
)
print()
print("Validation evaluation frame passed: True")

In [ ]:
expected_category_levels = {
    "location_id": set(range(1, 264)),
    "forecast_horizon": set(range(1, 7)),
    "hour_of_day": set(range(24)),
    "day_of_week": set(range(7)),
    "month": set(range(1, 13)),
}

category_range_records = []

# Checking complete training and valid validation levels.
for feature_name, expected_levels in (
    expected_category_levels.items()
):
    training_levels = set(
        X_train[feature_name].unique()
    )
    validation_levels = set(
        X_validation[feature_name].unique()
    )

    category_range_records.append({
        "feature": feature_name,
        "train_range": (
            f"{min(training_levels)}–"
            f"{max(training_levels)}"
        ),
        "train_levels": len(training_levels),
        "validation_range": (
            f"{min(validation_levels)}–"
            f"{max(validation_levels)}"
        ),
        "validation_levels": len(
            validation_levels
        ),
        "training_complete": (
            training_levels == expected_levels
        ),
        "validation_valid": (
            validation_levels
            .issubset(expected_levels)
        ),
    })

category_range_validation = pd.DataFrame(
    category_range_records
)

display(category_range_validation)

assert category_range_validation[
    "training_complete"
].all()

assert category_range_validation[
    "validation_valid"
].all()

print("Categorical range validation passed: True")

In [ ]:
# Applying the same category levels to both splits.
for feature_name, expected_levels in (
    expected_category_levels.items()
):
    category_levels = sorted(expected_levels)

    X_train[feature_name] = pd.Categorical(
        X_train[feature_name],
        categories=category_levels,
    )

    X_validation[feature_name] = pd.Categorical(
        X_validation[feature_name],
        categories=category_levels,
    )

# Checking category types and level consistency.
for feature_name in categorical_features:
    assert isinstance(
        X_train[feature_name].dtype,
        pd.CategoricalDtype,
    )

    assert isinstance(
        X_validation[feature_name].dtype,
        pd.CategoricalDtype,
    )

    assert X_train[
        feature_name
    ].cat.categories.equals(
        X_validation[
            feature_name
        ].cat.categories
    )

    assert not X_train[
        feature_name
    ].isna().any()

    assert not X_validation[
        feature_name
    ].isna().any()

category_dtype_summary = pd.DataFrame({
    "feature": categorical_features,
    "training_dtype": [
        str(X_train[column].dtype)
        for column in categorical_features
    ],
    "validation_dtype": [
        str(X_validation[column].dtype)
        for column in categorical_features
    ],
    "levels": [
        len(
            X_train[column].cat.categories
        )
        for column in categorical_features
    ],
})

display(category_dtype_summary)

print("Categorical encoding passed: True")

In [ ]:
# Releasing Arrow tables after completing the conversions.
del training_arrow
del validation_arrow

gc.collect()

available_ram_gb = (
    psutil.virtual_memory().available
    / 1024**3
)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Notebook process RAM (GB): "
    f"{process_ram_gb:,.2f}"
)
print(
    "Available system RAM (GB): "
    f"{available_ram_gb:,.2f}"
)

assert available_ram_gb > 10

print()
print("Arrow memory cleanup passed: True")

### Model Data Is Ready

The matrices now contain about 19.26 million training rows and 3.40 million validation rows. The feature order matches, the categorical columns remain categorical, and missing weather has not been silently filled. That gives me a consistent starting point for the model comparisons.


In [ ]:
smoke_training_rows = 200_000
smoke_validation_rows = 50_000
random_seed = 42

forecast_weather_columns = [
    "forecast_temperature_f",
    "forecast_precipitation_in",
    "forecast_rain_in",
    "forecast_snowfall_in",
    "forecast_wind_speed_mph",
]

# Sampling reproducible rows for the GPU smoke test.
X_train_smoke = X_train.sample(
    n=smoke_training_rows,
    random_state=random_seed,
)

y_train_smoke = y_train.loc[
    X_train_smoke.index
]

X_validation_smoke = X_validation.sample(
    n=smoke_validation_rows,
    random_state=random_seed,
)

y_validation_smoke = y_validation.loc[
    X_validation_smoke.index
]

# Checking that the sample preserves production behavior.
training_missing_weather = int(
    X_train_smoke[
        forecast_weather_columns
    ].isna().any(axis=1).sum()
)

assert len(X_train_smoke) == smoke_training_rows
assert len(X_validation_smoke) == (
    smoke_validation_rows
)
assert training_missing_weather > 0

assert X_train_smoke.columns.equals(
    X_validation_smoke.columns
)

assert not y_train_smoke.isna().any()
assert not y_validation_smoke.isna().any()

print(
    "Smoke-test training rows: "
    f"{len(X_train_smoke):,}"
)
print(
    "Smoke-test validation rows: "
    f"{len(X_validation_smoke):,}"
)
print(
    "Training rows with missing weather: "
    f"{training_missing_weather:,}"
)
print()
print("Smoke-test samples passed: True")

In [ ]:
# Defining a small model for compatibility testing.
xgb_smoke_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=25,
    max_depth=6,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=random_seed,
    eval_metric=["rmse", "mae"],
)

smoke_start_time = time.perf_counter()

# Fitting categories and missing values on the GPU.
xgb_smoke_model.fit(
    X_train_smoke,
    y_train_smoke,
    eval_set=[
        (
            X_validation_smoke,
            y_validation_smoke,
        )
    ],
    verbose=False,
)

smoke_training_seconds = (
    time.perf_counter()
    - smoke_start_time
)

smoke_predictions = xgb_smoke_model.predict(
    X_validation_smoke
)

booster_feature_types = (
    xgb_smoke_model
    .get_booster()
    .feature_types
)

categorical_type_count = (
    booster_feature_types.count("c")
)

assert len(smoke_predictions) == (
    smoke_validation_rows
)
assert np.isfinite(smoke_predictions).all()
assert categorical_type_count == len(
    categorical_features
)

print(
    "Smoke-test training time (seconds): "
    f"{smoke_training_seconds:,.2f}"
)
print(
    "Finite predictions: "
    f"{np.isfinite(smoke_predictions).all()}"
)
print(
    "Categorical features recognized: "
    f"{categorical_type_count}"
)
print()
print("XGBoost GPU smoke test passed: True")

In [ ]:
# Creating an explicit prediction matrix.
smoke_validation_dmatrix = xgb.DMatrix(
    X_validation_smoke,
    label=y_validation_smoke,
    enable_categorical=True,
    missing=np.nan,
)

# Predicting without relying on CPU inplace prediction.
smoke_dmatrix_predictions = (
    xgb_smoke_model
    .get_booster()
    .predict(smoke_validation_dmatrix)
)

predictions_match = np.allclose(
    smoke_predictions,
    smoke_dmatrix_predictions,
    rtol=1e-6,
    atol=1e-6,
)

assert np.isfinite(
    smoke_dmatrix_predictions
).all()

assert predictions_match

print(
    "Finite DMatrix predictions: "
    f"{np.isfinite(smoke_dmatrix_predictions).all()}"
)
print(
    "Predictions match earlier results: "
    f"{predictions_match}"
)
print()
print("Explicit prediction path passed: True")

### GPU Compatibility Check

The small GPU test works, including rows with missing weather. All predictions are finite, and XGBoost recognizes the categorical fields. I can move to the full fit without first discovering a basic compatibility issue on millions of rows.


## What does the first XGBoost model add?

The weekly baseline cannot respond to a holiday, different weather, or an event happening this week. XGBoost can combine those inputs with pickup history. I'll begin with a squared-error model and see whether that extra flexibility improves the later validation predictions.


In [ ]:
xgb_max_bin = 256

# Releasing temporary smoke-test objects.
del X_train_smoke
del y_train_smoke
del X_validation_smoke
del y_validation_smoke
del xgb_smoke_model
del smoke_validation_dmatrix
del smoke_predictions
del smoke_dmatrix_predictions

gc.collect()
# Building paired matrices with training-derived cuts.
(
    xgb_training_matrix,
    xgb_validation_matrix,
    matrix_build_seconds,
) = build_quantile_matrix_pair(
    training_features=X_train,
    training_target=y_train,
    evaluation_features=X_validation,
    evaluation_target=y_validation,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)


assert xgb_training_matrix.num_row() == len(
    y_train
)
assert xgb_validation_matrix.num_row() == len(
    y_validation
)
assert xgb_training_matrix.num_col() == len(
    model_input_columns
)
assert xgb_validation_matrix.num_col() == len(
    model_input_columns
)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Training matrix rows: "
    f"{xgb_training_matrix.num_row():,}"
)
print(
    "Validation matrix rows: "
    f"{xgb_validation_matrix.num_row():,}"
)
print(
    "Matrix columns: "
    f"{xgb_training_matrix.num_col()}"
)
print(
    "Build time (seconds): "
    f"{matrix_build_seconds:,.2f}"
)
print(
    "Notebook process RAM (GB): "
    f"{process_ram_gb:,.2f}"
)
print()
print("Quantized matrices passed: True")

In [ ]:
initial_xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": ["mae", "rmse"],
    "tree_method": "hist",
    "device": "cuda",
    "max_bin": xgb_max_bin,
    "max_depth": 8,
    "learning_rate": 0.05,
    "min_child_weight": 20,
    "subsample": 0.80,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.10,
    "reg_lambda": 5.00,
    "seed": random_seed,
}

# initial_xgb_params = {
#     "objective": "reg:squarederror",
#     "eval_metric": ["mae", "rmse"],
#     "tree_method": "hist",
#     "device": "cuda",
#     "max_bin": xgb_max_bin,
#     "max_depth": 8,
#     "learning_rate": 0.05,
#     "min_child_weight": 20,
#     "subsample": 0.80,
#     "colsample_bytree": 0.85,
#     "reg_alpha": 0.07,
#     "reg_lambda": 10.00,
#     "seed": random_seed,
# }

maximum_boosting_rounds = 2_000
early_stopping_rounds = 100

# Training the first full production model.
(
    initial_xgb_model,
    initial_xgb_history,
    initial_training_seconds,
) = train_xgboost_candidate(
    parameters=initial_xgb_params,
    training_matrix=xgb_training_matrix,
    evaluation_matrices=[
        (
            xgb_training_matrix,
            "training",
        ),
        (
            xgb_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert initial_xgb_model.best_iteration >= 0
assert initial_xgb_model.best_score > 0

print()
print(
    "Best boosting round: "
    f"{initial_xgb_model.best_iteration + 1:,}"
)
print(
    "Best validation RMSE: "
    f"{float(initial_xgb_model.best_score):,.4f}"
)
print(
    "Training time (minutes): "
    f"{initial_training_seconds / 60:,.2f}"
)
print()
print("Initial XGBoost training passed: True")

### What can I change about how the trees learn?

The first fit uses 1,999 boosting rounds. Each new tree corrects part of the error left by earlier trees. The table below records the settings so the comparisons that follow have a clear reference point.

| Parameter | Value | Practical role |
|---|---:|---|
| `objective` | `reg:squarederror` | Trains the model by penalizing squared prediction errors. Larger errors receive more weight. |
| `eval_metric` | MAE and RMSE | Tracks both typical absolute error and sensitivity to larger errors. RMSE is listed last, so early stopping monitors validation RMSE. |
| `tree_method` | `hist` | Groups continuous values into bins before finding tree splits. This is much faster for a dataset of this size. |
| `device` | `cuda` | Trains the model with the RTX 4080 GPU. |
| `max_bin` | 256 | Allows up to 256 histogram bins for continuous features. More bins can represent finer splits but use more memory. |
| `max_depth` | 8 | Allows each tree to capture moderately complex interactions between zone, time, lags, weather, and events. |
| `learning_rate` | 0.05 | Limits how much each new tree changes the combined prediction. The lower rate makes learning more gradual. |
| `min_child_weight` | 20 | Requires enough training weight in a new branch. This discourages splits based on very small groups of rows. |
| `subsample` | 0.80 | Uses 80% of training rows for each tree, adding regularization and reducing computation. |
| `colsample_bytree` | 0.85 | Makes each tree consider 85% of the available features. This prevents every tree from depending on the same predictors. |
| `reg_alpha` | 0.10 | Adds L1 regularization, which can reduce weak or unnecessary leaf effects. |
| `reg_lambda` | 5.00 | Adds L2 regularization, shrinking extreme leaf values and making the model more stable. |
| `seed` | 42 | Keeps row and feature sampling reproducible. |
| `maximum_boosting_rounds` | 2,000 | Places an upper limit on the number of trees. |
| `early_stopping_rounds` | 100 | Stops training when validation RMSE fails to improve for 100 consecutive rounds. |


In [ ]:
best_iteration_end = (
    initial_xgb_model.best_iteration + 1
)

# Predicting at the best validation boosting round.
(
    initial_xgb_predictions,
    initial_xgb_predictions_constrained,
    negative_prediction_count,
    prediction_seconds,
) = predict_xgboost_demand(
    model=initial_xgb_model,
    prediction_matrix=xgb_validation_matrix,
    iteration_end=best_iteration_end,
)

negative_prediction_percent = (
    negative_prediction_count
    / len(initial_xgb_predictions)
    * 100
)

assert len(initial_xgb_predictions) == len(
    y_validation
)
assert np.isfinite(
    initial_xgb_predictions
).all()

print(
    "Validation predictions: "
    f"{len(initial_xgb_predictions):,}"
)
print(
    "Prediction time (seconds): "
    f"{prediction_seconds:,.2f}"
)
print(
    "Minimum prediction: "
    f"{initial_xgb_predictions.min():,.3f}"
)
print(
    "Negative predictions: "
    f"{negative_prediction_count:,}"
)
print(
    "Negative prediction rate: "
    f"{negative_prediction_percent:.4f}%"
)
print()


In [ ]:
adjusted_prediction_count = int(
    np.count_nonzero(
        initial_xgb_predictions_constrained
        != initial_xgb_predictions
    )
)

negative_prediction_mask = (
    initial_xgb_predictions < 0
)

negative_group_actual_mean = float(
    y_validation[
        negative_prediction_mask
    ].mean()
)

negative_group_zero_percent = float(
    (
        y_validation[
            negative_prediction_mask
        ] == 0
    ).mean()
    * 100
)

assert adjusted_prediction_count == (
    negative_prediction_count
)
assert (
    initial_xgb_predictions_constrained.min()
    >= 0
)
assert np.isfinite(
    initial_xgb_predictions_constrained
).all()

print(
    "Predictions adjusted to zero: "
    f"{adjusted_prediction_count:,}"
)
print(
    "Minimum constrained prediction: "
    f"{initial_xgb_predictions_constrained.min():.3f}"
)
print(
    "Mean actual demand in adjusted rows: "
    f"{negative_group_actual_mean:,.3f}"
)
print(
    "Zero-demand share in adjusted rows: "
    f"{negative_group_zero_percent:.2f}%"
)
print()
print("Nonnegative prediction constraint passed: True")

### What to do with negative pickup forecasts

Some predictions are negative, mostly in hours with zero or very few pickups. A negative pickup count has no practical meaning, so I'll set those forecasts to zero. This affects about 2.22% of validation rows, and I'll apply the same rule consistently rather than change it between comparisons.


In [ ]:
weekly_naive_predictions = (
    X_validation[
        "pickup_lag_168"
    ].to_numpy()
)

# Comparing the baseline and both XGBoost outputs.
initial_model_comparison = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions={
            "Weekly Seasonal Naive": (
                weekly_naive_predictions
            ),
            "XGBoost Raw": (
                initial_xgb_predictions
            ),
            "XGBoost Constrained": (
                initial_xgb_predictions_constrained
            ),
        },
    )
)

display(
    initial_model_comparison.round(3)
)

raw_rmse = initial_model_comparison.loc[
    initial_model_comparison["model"]
    == "XGBoost Raw",
    "rmse",
].iloc[0]

constrained_rmse = (
    initial_model_comparison.loc[
        initial_model_comparison["model"]
        == "XGBoost Constrained",
        "rmse",
    ].iloc[0]
)

assert constrained_rmse <= raw_rmse

print(
    "Constrained RMSE does not exceed "
    "raw RMSE: True"
)

### Did the first model beat last week?

The first model improves substantially on repeating last week. MAE falls from 22.9 to 16.3 pickups, and RMSE from 48.3 to 33.2. That is a useful start, but the pooled score could hide a weak result for certain hours or horizons. I'll check those next.


In [ ]:
# Preparing aligned validation data for horizon-level metrics.
validation_horizon_frame = pd.DataFrame({
    "forecast_horizon": (
        X_validation[
            "forecast_horizon"
        ]
        .astype("int8")
        .to_numpy()
    ),
    "total_pickups": np.asarray(
        y_validation
    ),
    "weekly_naive_prediction": (
        weekly_naive_predictions
    ),
    "xgboost_prediction": (
        initial_xgb_predictions_constrained
    ),
})

# Calculating both model results within each forecast horizon.
initial_horizon_metrics = (
    calculate_metrics_by_group(
        evaluation_frame=(
            validation_horizon_frame
        ),
        group_columns="forecast_horizon",
        prediction_columns={
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
            "XGBoost Constrained": (
                "xgboost_prediction"
            ),
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Restoring the original model-first reporting order.
horizon_model_order = {
    "Weekly Seasonal Naive": 0,
    "XGBoost Constrained": 1,
}

initial_horizon_metrics[
    "_model_order"
] = initial_horizon_metrics[
    "model"
].map(horizon_model_order)

initial_horizon_metrics = (
    initial_horizon_metrics
    .sort_values(
        [
            "_model_order",
            "forecast_horizon",
        ]
    )
    .drop(columns="_model_order")
    .reset_index(drop=True)
)

# Checking the expected validation structure.
assert len(initial_horizon_metrics) == 12

assert set(
    initial_horizon_metrics[
        "forecast_horizon"
    ].astype(int)
) == set(range(1, 7))

assert initial_horizon_metrics[
    "rows"
].eq(566_239).all()

assert (
    initial_horizon_metrics
    .groupby(
        "model",
        observed=True,
    )["rows"]
    .sum()
    .eq(len(y_validation))
    .all()
)

display(
    initial_horizon_metrics.round(3)
)

print(
    "Horizon metric validation passed:",
    True,
)

# Releasing the temporary metric frame.
del validation_horizon_frame

### Does the gain hold six hours ahead?

The improvement holds from one through six hours ahead. The scores are quite similar across horizons, which is worth understanding rather than automatically calling good or suspicious. Much of the model's information comes from daily and weekly patterns that remain relevant throughout this short window.


In [ ]:
training_rmse_history = np.asarray(
    initial_xgb_history[
        "training"
    ]["rmse"],
    dtype=float,
)

validation_rmse_history = np.asarray(
    initial_xgb_history[
        "validation"
    ]["rmse"],
    dtype=float,
)

validation_mae_history = np.asarray(
    initial_xgb_history[
        "validation"
    ]["mae"],
    dtype=float,
)

trained_rounds = len(
    validation_rmse_history
)

best_round_index = (
    initial_xgb_model.best_iteration
)

checkpoint_rounds = sorted(set([
    1,
    100,
    500,
    1_000,
    1_500,
    best_round_index + 1,
    trained_rounds,
]))

checkpoint_rounds = [
    round_number
    for round_number in checkpoint_rounds
    if round_number <= trained_rounds
]

learning_curve_summary = pd.DataFrame({
    "round": checkpoint_rounds,
    "validation_mae": [
        validation_mae_history[
            round_number - 1
        ]
        for round_number in checkpoint_rounds
    ],
    "validation_rmse": [
        validation_rmse_history[
            round_number - 1
        ]
        for round_number in checkpoint_rounds
    ],
})

display(
    learning_curve_summary.round(4)
)

rmse_gain_last_100 = (
    validation_rmse_history[
        best_round_index - 100
    ]
    - validation_rmse_history[
        best_round_index
    ]
)

rmse_gain_last_500 = (
    validation_rmse_history[
        best_round_index - 500
    ]
    - validation_rmse_history[
        best_round_index
    ]
)

final_training_rmse = (
    training_rmse_history[
        best_round_index
    ]
)

final_validation_rmse = (
    validation_rmse_history[
        best_round_index
    ]
)

print(
    "RMSE gain over final 100 rounds: "
    f"{rmse_gain_last_100:.4f}"
)
print(
    "RMSE gain over final 500 rounds: "
    f"{rmse_gain_last_500:.4f}"
)
print(
    "Training RMSE at best round: "
    f"{final_training_rmse:.4f}"
)
print(
    "Validation RMSE at best round: "
    f"{final_validation_rmse:.4f}"
)

### Would more trees be worth it?

The curve is still near its best at the round limit, but the last 100 rounds improve RMSE by only 0.0057. Simply adding more trees does not look like the most promising next move. I'll keep this fit as the reference and test whether a different objective handles counts more naturally.


## Can a count objective keep the accuracy?

A tree model can produce a negative forecast even though a pickup count cannot be negative. I try a Poisson objective to see whether a model built for counts can handle that constraint and still compete on the same validation rows.

Squared error gave useful predictions, but it also produced negatives that needed clipping. A Poisson objective avoids that issue by construction. I'll test whether it can keep the accuracy while giving a more natural prediction range.

| Parameter | Value | Practical role |
|---|---:|---|
| `objective` | `count:poisson` | Models a positive expected count instead of an unrestricted continuous value. |
| `max_delta_step` | 0.7 | Limits how sharply each boosting update can change the Poisson prediction, improving numerical stability. |


In [ ]:
poisson_xgb_params = (
    initial_xgb_params.copy()
)

poisson_xgb_params.update({
    "objective": "count:poisson",
    "max_delta_step": 0.7,
})

# Training the count-based production model.
(
    poisson_xgb_model,
    poisson_xgb_history,
    poisson_training_seconds,
) = train_xgboost_candidate(
    parameters=poisson_xgb_params,
    training_matrix=xgb_training_matrix,
    evaluation_matrices=[
        (
            xgb_training_matrix,
            "training",
        ),
        (
            xgb_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert poisson_xgb_model.best_iteration >= 0
assert poisson_xgb_model.best_score > 0

print()
print(
    "Best boosting round: "
    f"{poisson_xgb_model.best_iteration + 1:,}"
)
print(
    "Best validation RMSE: "
    f"{float(poisson_xgb_model.best_score):,.4f}"
)
print(
    "Training time (minutes): "
    f"{poisson_training_seconds / 60:,.2f}"
)
print()
print("Poisson XGBoost training passed: True")

In [ ]:
poisson_best_iteration_end = (
    poisson_xgb_model.best_iteration + 1
)

# Predicting positive expected demand counts.
(
    poisson_xgb_predictions,
    poisson_xgb_predictions_constrained,
    poisson_negative_count,
    poisson_prediction_seconds,
) = predict_xgboost_demand(
    model=poisson_xgb_model,
    prediction_matrix=xgb_validation_matrix,
    iteration_end=(
        poisson_best_iteration_end
    ),
)

assert len(poisson_xgb_predictions) == len(
    y_validation
)
assert np.isfinite(
    poisson_xgb_predictions
).all()
assert poisson_negative_count == 0
assert poisson_xgb_predictions.min() >= 0

print(
    "Validation predictions: "
    f"{len(poisson_xgb_predictions):,}"
)
print(
    "Prediction time (seconds): "
    f"{poisson_prediction_seconds:,.2f}"
)
print(
    "Minimum prediction: "
    f"{poisson_xgb_predictions.min():.6f}"
)
print(
    "Negative predictions: "
    f"{poisson_negative_count:,}"
)
print()
print("Poisson prediction constraint passed: True")

In [ ]:
# Comparing production-safe prediction approaches.
xgb_objective_comparison = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions={
            "Weekly Seasonal Naive": (
                weekly_naive_predictions
            ),
            "XGBoost Squared Error": (
                initial_xgb_predictions_constrained
            ),
            "XGBoost Poisson": (
                poisson_xgb_predictions
            ),
        },
    )
)

display(
    xgb_objective_comparison.round(3)
)

squared_error_rmse = (
    xgb_objective_comparison.loc[
        xgb_objective_comparison["model"]
        == "XGBoost Squared Error",
        "rmse",
    ].iloc[0]
)

poisson_rmse = (
    xgb_objective_comparison.loc[
        xgb_objective_comparison["model"]
        == "XGBoost Poisson",
        "rmse",
    ].iloc[0]
)

assert np.isclose(
    poisson_rmse,
    float(poisson_xgb_model.best_score),
    atol=0.001,
)

print(
    "Lower-RMSE XGBoost objective: "
    + (
        "Squared Error"
        if squared_error_rmse < poisson_rmse
        else "Poisson"
    )
)

### What did I gain by using a count objective?

Poisson solves the range problem, but RMSE rises from 33.2 to 34.9 in this comparison. That is more accuracy than I want to give up just to avoid clipping. I'll retain squared error for now and compare another model family.


## Would CatBoost handle these inputs better?

Before settling on XGBoost, I want another model that can work with the categorical fields and missing weather. CatBoost is a reasonable candidate. I'll start with a practical configuration rather than spend a large search budget before seeing whether it is competitive.


In [ ]:
catboost_training_pool_start = (
    time.perf_counter()
)

# Building the full training pool.
catboost_training_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=categorical_features,
)

catboost_training_pool_seconds = (
    time.perf_counter()
    - catboost_training_pool_start
)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

available_ram_gb = (
    psutil.virtual_memory().available
    / 1024**3
)

assert catboost_training_pool.num_row() == (
    len(y_train)
)
assert catboost_training_pool.num_col() == (
    len(model_input_columns)
)
assert len(
    catboost_training_pool
    .get_cat_feature_indices()
) == len(categorical_features)
assert available_ram_gb > 10

print(
    "Training pool rows: "
    f"{catboost_training_pool.num_row():,}"
)
print(
    "Training pool columns: "
    f"{catboost_training_pool.num_col()}"
)
print(
    "Build time (seconds): "
    f"{catboost_training_pool_seconds:,.2f}"
)
print(
    "Notebook process RAM (GB): "
    f"{process_ram_gb:,.2f}"
)
print(
    "Available system RAM (GB): "
    f"{available_ram_gb:,.2f}"
)
print()


In [ ]:
catboost_validation_pool_start = (
    time.perf_counter()
)

# Building the full validation pool.
catboost_validation_pool = Pool(
    data=X_validation,
    label=y_validation,
    cat_features=categorical_features,
)

catboost_validation_pool_seconds = (
    time.perf_counter()
    - catboost_validation_pool_start
)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

available_ram_gb = (
    psutil.virtual_memory().available
    / 1024**3
)

assert catboost_validation_pool.num_row() == (
    len(y_validation)
)
assert catboost_validation_pool.num_col() == (
    catboost_training_pool.num_col()
)
assert (
    catboost_validation_pool
    .get_feature_names()
    == catboost_training_pool
    .get_feature_names()
)
assert available_ram_gb > 10

print(
    "Validation pool rows: "
    f"{catboost_validation_pool.num_row():,}"
)
print(
    "Validation pool columns: "
    f"{catboost_validation_pool.num_col()}"
)
print(
    "Build time (seconds): "
    f"{catboost_validation_pool_seconds:,.2f}"
)
print(
    "Notebook process RAM (GB): "
    f"{process_ram_gb:,.2f}"
)
print(
    "Available system RAM (GB): "
    f"{available_ram_gb:,.2f}"
)
print()
print("Full CatBoost validation pool passed: True")

### Initial CatBoost Configuration

The CatBoost pools fit in memory, so I can use the full training period. The settings below give me a starting model with regularization. Then I'll check whether a few targeted changes close the validation gap.

| Parameter | Value | Practical role |
|---|---:|---|
| `loss_function` | `RMSE` | Penalizes larger pickups errors more strongly and aligns with one of the primary validation metrics. |
| `eval_metric` | `RMSE` | Uses validation RMSE to select the best tree count and trigger early stopping. |
| `iterations` | 2,000 | Sets a generous upper limit on the number of boosting trees. |
| `learning_rate` | 0.05 | Adds each tree gradually, reducing the risk of making overly aggressive corrections. |
| `depth` | 8 | Allows interactions among location, time, lags, weather, and events while keeping tree complexity controlled. |
| `l2_leaf_reg` | 5.0 | Shrinks extreme leaf values and adds protection against overfitting. |
| `random_strength` | 1.0 | Adds randomness when selecting similar candidate splits. |
| `bootstrap_type` | `Bernoulli` | Samples training rows independently for each tree. |
| `subsample` | 0.80 | Uses 80% of rows per tree, adding regularization and reducing GPU work. |
| `border_count` | 128 | Divides numeric features into up to 128 candidate ranges for GPU split selection. |
| `one_hot_max_size` | 32 | One-hot encodes smaller categorical variables while allowing CatBoost statistics for the 263-level `location_id`. |
| `max_ctr_complexity` | 1 | Limits categorical target-statistic combinations to control memory use and complexity. |
| `nan_mode` | `Min` | Keeps missing weather values as missing and consistently assigns them to one side of candidate splits. |
| `grow_policy` | `SymmetricTree` | Builds balanced trees with the same split at each level, which is efficient for GPU inference. |
| `task_type` | `GPU` | Uses the RTX 4080 for training. |
| `devices` | `0` | Selects the first detected GPU. |
| `random_seed` | 42 | Keeps sampling choices as reproducible as possible. |
| `od_wait` | 100 | Stops after 100 rounds without validation improvement. |


In [ ]:
initial_catboost_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 2_000,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,
    "border_count": 128,
    "one_hot_max_size": 32,
    "max_ctr_complexity": 1,
    "nan_mode": "Min",
    "grow_policy": "SymmetricTree",
    "task_type": "GPU",
    "devices": "0",
    "random_seed": random_seed,
    "od_type": "Iter",
    "od_wait": 100,
    "allow_writing_files": False,
    "verbose": 100,
}

initial_catboost_model = CatBoostRegressor(
    **initial_catboost_params
)

catboost_training_start = (
    time.perf_counter()
)

# Training the first direct CatBoost model.
initial_catboost_model.fit(
    catboost_training_pool,
    eval_set=catboost_validation_pool,
    use_best_model=True,
)

catboost_training_seconds = (
    time.perf_counter()
    - catboost_training_start
)

catboost_best_iteration = (
    initial_catboost_model
    .get_best_iteration()
)

catboost_best_rmse = (
    initial_catboost_model
    .get_best_score()
    ["validation"]["RMSE"]
)

assert catboost_best_iteration >= 0
assert catboost_best_rmse > 0
assert (
    initial_catboost_model.tree_count_
    == catboost_best_iteration + 1
)

print()
print(
    "Best boosting round: "
    f"{catboost_best_iteration + 1:,}"
)
print(
    "Best validation RMSE: "
    f"{catboost_best_rmse:,.4f}"
)
print(
    "Trees retained: "
    f"{initial_catboost_model.tree_count_:,}"
)
print(
    "Training time (minutes): "
    f"{catboost_training_seconds / 60:,.2f}"
)
print()
print("Initial CatBoost training passed: True")

In [ ]:
catboost_prediction_start = (
    time.perf_counter()
)

# Predicting with the retained CatBoost trees.
initial_catboost_predictions = (
    initial_catboost_model.predict(
        catboost_validation_pool
    )
)

catboost_prediction_seconds = (
    time.perf_counter()
    - catboost_prediction_start
)

catboost_negative_count = int(
    (initial_catboost_predictions < 0).sum()
)

catboost_negative_percent = (
    catboost_negative_count
    / len(initial_catboost_predictions)
    * 100
)

assert len(initial_catboost_predictions) == (
    len(y_validation)
)
assert np.isfinite(
    initial_catboost_predictions
).all()

print(
    "Validation predictions: "
    f"{len(initial_catboost_predictions):,}"
)
print(
    "Prediction time (seconds): "
    f"{catboost_prediction_seconds:,.2f}"
)
print(
    "Minimum prediction: "
    f"{initial_catboost_predictions.min():,.3f}"
)
print(
    "Negative predictions: "
    f"{catboost_negative_count:,}"
)
print(
    "Negative prediction rate: "
    f"{catboost_negative_percent:.4f}%"
)
print()
print("Initial CatBoost predictions passed: True")

In [ ]:
# Applying the same production demand constraint.
initial_catboost_predictions_constrained = (
    np.clip(
        initial_catboost_predictions,
        a_min=0,
        a_max=None,
    )
)

catboost_adjusted_count = int(
    np.count_nonzero(
        initial_catboost_predictions_constrained
        != initial_catboost_predictions
    )
)

catboost_negative_mask = (
    initial_catboost_predictions < 0
)

catboost_negative_actual_mean = float(
    y_validation[
        catboost_negative_mask
    ].mean()
)

catboost_negative_zero_percent = float(
    (
        y_validation[
            catboost_negative_mask
        ] == 0
    ).mean()
    * 100
)

assert catboost_adjusted_count == (
    catboost_negative_count
)
assert (
    initial_catboost_predictions_constrained
    .min()
    >= 0
)
assert np.isfinite(
    initial_catboost_predictions_constrained
).all()

print(
    "Predictions adjusted to zero: "
    f"{catboost_adjusted_count:,}"
)
print(
    "Mean actual demand in adjusted rows: "
    f"{catboost_negative_actual_mean:,.3f}"
)
print(
    "Zero-demand share in adjusted rows: "
    f"{catboost_negative_zero_percent:.2f}%"
)
print()
print("CatBoost demand constraint passed: True")

In [ ]:
# Comparing the baseline and direct tree models.
direct_model_comparison = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions={
            "Weekly Seasonal Naive": (
                weekly_naive_predictions
            ),
            "XGBoost": (
                initial_xgb_predictions_constrained
            ),
            "CatBoost": (
                initial_catboost_predictions_constrained
            ),
        },
    )
)


display(
    direct_model_comparison.round(3)
)

catboost_raw_metrics = (
    calculate_forecast_metrics(
        y_validation,
        initial_catboost_predictions,
    )
)

catboost_constrained_rmse = (
    direct_model_comparison.loc[
        direct_model_comparison["model"]
        == "CatBoost",
        "rmse",
    ].iloc[0]
)

assert catboost_constrained_rmse <= (
    catboost_raw_metrics["rmse"]
)

print(
    "CatBoost constraint does not "
    "increase RMSE: True"
)

In [ ]:
# Extracting the available CatBoost learning history.
catboost_evaluation_history = (
    initial_catboost_model
    .get_evals_result()
)

catboost_training_rmse = np.asarray(
    catboost_evaluation_history[
        "learn"
    ]["RMSE"],
    dtype=float,
)

catboost_validation_rmse = np.asarray(
    catboost_evaluation_history[
        "validation"
    ]["RMSE"],
    dtype=float,
)

catboost_trained_rounds = len(
    catboost_validation_rmse
)

catboost_best_round = (
    initial_catboost_model.get_best_iteration()
    + 1
)

# Keeping only checkpoints available in this run.
catboost_checkpoint_rounds = sorted(set([
    1,
    100,
    500,
    1_000,
    catboost_best_round,
    catboost_trained_rounds,
]))

catboost_checkpoint_rounds = [
    round_number
    for round_number in catboost_checkpoint_rounds
    if 1 <= round_number <= catboost_trained_rounds
]

catboost_learning_summary = pd.DataFrame({
    "round": catboost_checkpoint_rounds,
    "training_rmse": [
        catboost_training_rmse[
            round_number - 1
        ]
        for round_number in catboost_checkpoint_rounds
    ],
    "validation_rmse": [
        catboost_validation_rmse[
            round_number - 1
        ]
        for round_number in catboost_checkpoint_rounds
    ],
})

display(
    catboost_learning_summary.style
    .format({
        "round": "{:,.0f}",
        "training_rmse": "{:.4f}",
        "validation_rmse": "{:.4f}",
    })
    .hide(axis="index")
)

# Calculating gains only when enough rounds exist.
catboost_gain_last_100 = (
    catboost_validation_rmse[-101]
    - catboost_validation_rmse[-1]
    if catboost_trained_rounds > 100
    else np.nan
)

catboost_gain_last_500 = (
    catboost_validation_rmse[-501]
    - catboost_validation_rmse[-1]
    if catboost_trained_rounds > 500
    else np.nan
)

assert catboost_trained_rounds > 0
assert 1 <= catboost_best_round <= catboost_trained_rounds

print(
    "Recorded boosting rounds:",
    f"{catboost_trained_rounds:,}",
)
print(
    "Best boosting round:",
    f"{catboost_best_round:,}",
)
print(
    "RMSE gain over final 100 recorded rounds:",
    f"{catboost_gain_last_100:.4f}",
)
print(
    "RMSE gain over final 500 recorded rounds:",
    f"{catboost_gain_last_500:.4f}",
)
print(
    "Final training-validation gap:",
    f"{(
        catboost_validation_rmse[-1]
        - catboost_training_rmse[-1]
    ):.4f}",
)

### CatBoost Tuning Decision

CatBoost is still improving at the round boundary, but the late gains are small compared with its gap from XGBoost. I'll try a higher learning rate and depth before deciding whether more training is worthwhile.


In [ ]:
catboost_lr_params = (
    initial_catboost_params.copy()
)

catboost_lr_params.update({
    "learning_rate": 0.075,
})

catboost_lr_model = CatBoostRegressor(
    **catboost_lr_params
)

catboost_lr_start = time.perf_counter()

# Testing a faster CatBoost learning rate.
catboost_lr_model.fit(
    catboost_training_pool,
    eval_set=catboost_validation_pool,
    use_best_model=True,
)

catboost_lr_seconds = (
    time.perf_counter()
    - catboost_lr_start
)

catboost_lr_best_iteration = (
    catboost_lr_model.get_best_iteration()
)

catboost_lr_best_rmse = (
    catboost_lr_model
    .get_best_score()
    ["validation"]["RMSE"]
)

assert catboost_lr_best_iteration >= 0
assert catboost_lr_best_rmse > 0
assert (
    catboost_lr_model.tree_count_
    == catboost_lr_best_iteration + 1
)

print()
print(
    "Best boosting round: "
    f"{catboost_lr_best_iteration + 1:,}"
)
print(
    "Best validation RMSE: "
    f"{catboost_lr_best_rmse:,.4f}"
)
print(
    "RMSE change from initial: "
    f"{(
        catboost_lr_best_rmse
        - catboost_best_rmse
    ):+.4f}"
)
print(
    "Training time (minutes): "
    f"{catboost_lr_seconds / 60:,.2f}"
)
print()
print("CatBoost learning-rate test passed: True")

In [ ]:
catboost_lr_prediction_start = (
    time.perf_counter()
)

# Predicting with the learning-rate candidate.
catboost_lr_predictions = (
    catboost_lr_model.predict(
        catboost_validation_pool
    )
)

catboost_lr_prediction_seconds = (
    time.perf_counter()
    - catboost_lr_prediction_start
)

catboost_lr_negative_count = int(
    (catboost_lr_predictions < 0).sum()
)

# Applying the production lower bound.
catboost_lr_predictions_constrained = (
    np.clip(
        catboost_lr_predictions,
        a_min=0,
        a_max=None,
    )
)

assert len(catboost_lr_predictions) == (
    len(y_validation)
)
assert np.isfinite(
    catboost_lr_predictions
).all()
assert (
    catboost_lr_predictions_constrained.min()
    >= 0
)

print(
    "Prediction time (seconds): "
    f"{catboost_lr_prediction_seconds:,.2f}"
)
print(
    "Minimum raw prediction: "
    f"{catboost_lr_predictions.min():,.3f}"
)
print(
    "Negative predictions: "
    f"{catboost_lr_negative_count:,}"
)
print(
    "Negative prediction rate: "
    f"{(
        catboost_lr_negative_count
        / len(catboost_lr_predictions)
        * 100
    ):.4f}%"
)
print()
print("Tuned CatBoost predictions passed: True")

In [ ]:
# Comparing both CatBoost specifications and the XGBoost reference.
catboost_tuning_comparison = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions={
            "CatBoost LR 0.050": (
                initial_catboost_predictions_constrained
            ),
            "CatBoost LR 0.075": (
                catboost_lr_predictions_constrained
            ),
            "XGBoost Reference": (
                initial_xgb_predictions_constrained
            ),
        },
    )
)


display(
    catboost_tuning_comparison.round(3)
)

initial_catboost_wape = (
    catboost_tuning_comparison.loc[
        catboost_tuning_comparison["model"]
        == "CatBoost LR 0.050",
        "wape_percent",
    ].iloc[0]
)

tuned_catboost_wape = (
    catboost_tuning_comparison.loc[
        catboost_tuning_comparison["model"]
        == "CatBoost LR 0.075",
        "wape_percent",
    ].iloc[0]
)

print(
    "WAPE change from higher learning rate: "
    f"{(
        tuned_catboost_wape
        - initial_catboost_wape
    ):+.3f} percentage points"
)

### Did smaller corrections help CatBoost?

The higher learning rate helps: RMSE falls from about 37.0 to 36.7, with small improvements in WAPE and bias too. The gap remains, though. Next I'll see whether more depth helps it represent the zone and calendar interactions.


In [ ]:
# Defining the controlled depth-10 candidate.
catboost_depth10_params = catboost_lr_params.copy()
catboost_depth10_params.update({
    "depth": 10,
})

catboost_depth10_model = CatBoostRegressor(
    **catboost_depth10_params
)

# Training the higher-capacity candidate.
catboost_depth10_start = time.perf_counter()

catboost_depth10_model.fit(
    catboost_training_pool,
    eval_set=catboost_validation_pool,
    use_best_model=True,
)

catboost_depth10_minutes = (
    time.perf_counter()
    - catboost_depth10_start
) / 60

# Collecting the validation result.
catboost_depth10_best_iteration = (
    catboost_depth10_model.get_best_iteration()
)
catboost_depth10_best_round = (
    catboost_depth10_best_iteration + 1
)
catboost_depth10_best_rmse = (
    catboost_depth10_model
    .get_best_score()["validation"]["RMSE"]
)

catboost_depth10_rmse_change = (
    catboost_depth10_best_rmse
    - catboost_lr_best_rmse
)

assert catboost_depth10_best_round >= 1
assert np.isfinite(catboost_depth10_best_rmse)

print(
    "Best boosting round: "
    f"{catboost_depth10_best_round:,}"
)
print(
    "Best validation RMSE: "
    f"{catboost_depth10_best_rmse:,.4f}"
)
print(
    "RMSE change from depth 8: "
    f"{catboost_depth10_rmse_change:+.4f}"
)
print(
    "Training time (minutes): "
    f"{catboost_depth10_minutes:,.2f}"
)
print()
print("CatBoost depth test completed: True")

In [ ]:
# Generating validation predictions from the depth-10 model.
catboost_depth10_prediction_start = time.perf_counter()

catboost_depth10_predictions = (
    catboost_depth10_model.predict(
        catboost_validation_pool
    )
)

catboost_depth10_prediction_seconds = (
    time.perf_counter()
    - catboost_depth10_prediction_start
)

# Checking the nonnegative demand constraint.
catboost_depth10_negative_count = int(
    (catboost_depth10_predictions < 0).sum()
)

catboost_depth10_negative_rate = (
    100
    * catboost_depth10_negative_count
    / len(catboost_depth10_predictions)
)

catboost_depth10_predictions_constrained = np.clip(
    catboost_depth10_predictions,
    a_min=0,
    a_max=None,
)

assert len(catboost_depth10_predictions) == len(
    y_validation
)
assert np.isfinite(
    catboost_depth10_predictions
).all()
assert (
    catboost_depth10_predictions_constrained >= 0
).all()

print(
    "Prediction time (seconds): "
    f"{catboost_depth10_prediction_seconds:,.2f}"
)
print(
    "Minimum raw prediction: "
    f"{catboost_depth10_predictions.min():,.3f}"
)
print(
    "Negative predictions: "
    f"{catboost_depth10_negative_count:,}"
)
print(
    "Negative prediction rate: "
    f"{catboost_depth10_negative_rate:,.4f}%"
)
print()
print("CatBoost depth-10 predictions passed: True")

In [ ]:
# Comparing the retained direct-model candidates.
direct_model_comparison = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions={
            "Weekly Seasonal Naive": (
                weekly_naive_predictions
            ),
            "XGBoost": (
                initial_xgb_predictions_constrained
            ),
            "CatBoost Depth 8": (
                catboost_lr_predictions_constrained
            ),
            "CatBoost Depth 10": (
                catboost_depth10_predictions_constrained
            ),
        },
    )
)

display(
    direct_model_comparison.round(3)
)

# Calculating the remaining gap to XGBoost.
xgboost_rmse = direct_model_comparison.loc[
    direct_model_comparison["model"] == "XGBoost",
    "rmse",
].iloc[0]

depth10_rmse = direct_model_comparison.loc[
    direct_model_comparison["model"]
    == "CatBoost Depth 10",
    "rmse",
].iloc[0]

print(
    "Depth-10 CatBoost RMSE gap to XGBoost: "
    f"{depth10_rmse - xgboost_rmse:+.3f}"
)

### Selecting the CatBoost specification

Depth 10 lowers RMSE again, to about 36.0. I'll keep this as the direct CatBoost candidate. It is better than the earlier CatBoost versions, but it still does not catch the direct XGBoost reference.

$$
e_t
=
y_t
-
y_{t-168}
$$

$$
\widehat{y}_t
=
y_{t-168}
+
\widehat{e}_t
$$


In [ ]:
# Extracting the production-available weekly baselines.
training_weekly_baseline = (
    X_train["pickup_lag_168"]
    .to_numpy(dtype=np.float32, copy=False)
)

validation_weekly_baseline = (
    X_validation["pickup_lag_168"]
    .to_numpy(dtype=np.float32, copy=False)
)

# Calculating the residual targets.
y_train_residual = (
    np.asarray(y_train, dtype=np.float32)
    - training_weekly_baseline
)

y_validation_residual = (
    np.asarray(y_validation, dtype=np.float32)
    - validation_weekly_baseline
)

# Checking completeness and row alignment.
assert len(y_train_residual) == len(X_train)
assert len(y_validation_residual) == len(X_validation)

assert np.isfinite(training_weekly_baseline).all()
assert np.isfinite(validation_weekly_baseline).all()
assert np.isfinite(y_train_residual).all()
assert np.isfinite(y_validation_residual).all()

assert np.allclose(
    validation_weekly_baseline,
    weekly_naive_predictions,
)

assert np.allclose(
    training_weekly_baseline
    + y_train_residual,
    np.asarray(y_train),
)

residual_summary = pd.DataFrame({
    "split": ["Training", "Validation"],
    "mean_residual": [
        y_train_residual.mean(),
        y_validation_residual.mean(),
    ],
    "median_residual": [
        np.median(y_train_residual),
        np.median(y_validation_residual),
    ],
    "residual_std": [
        y_train_residual.std(),
        y_validation_residual.std(),
    ],
    "minimum_residual": [
        y_train_residual.min(),
        y_validation_residual.min(),
    ],
    "maximum_residual": [
        y_train_residual.max(),
        y_validation_residual.max(),
    ],
})



# Calculating the baseline bias directly.
weekly_naive_mean_bias = np.mean(
    weekly_naive_predictions
    - np.asarray(y_validation)
)

assert np.isclose(
            y_validation_residual.mean(),
            -weekly_naive_mean_bias,
            atol=1e-3,
        )


In [ ]:
# Creating deterministic residual samples for visualization.
residual_chart_sample_size = 250_000

training_residual_indices = np.linspace(
    0,
    len(y_train_residual) - 1,
    num=min(
        residual_chart_sample_size,
        len(y_train_residual),
    ),
    dtype=np.int64,
)

validation_residual_indices = np.linspace(
    0,
    len(y_validation_residual) - 1,
    num=min(
        residual_chart_sample_size,
        len(y_validation_residual),
    ),
    dtype=np.int64,
)

residual_chart_data = pd.concat(
    [
        pd.DataFrame({
            "split": "Training",
            "residual": y_train_residual[
                training_residual_indices
            ],
        }),
        pd.DataFrame({
            "split": "Validation",
            "residual": y_validation_residual[
                validation_residual_indices
            ],
        }),
    ],
    ignore_index=True,
)

# Calculating full-period mean residuals by target hour.
training_residual_hours = (
    X_train["hour_of_day"]
    .to_numpy(dtype=np.int32, copy=False)
)

validation_residual_hours = (
    X_validation["hour_of_day"]
    .to_numpy(dtype=np.int32, copy=False)
)

training_hour_counts = np.bincount(
    training_residual_hours,
    minlength=24,
)

validation_hour_counts = np.bincount(
    validation_residual_hours,
    minlength=24,
)

training_hourly_residual_mean = (
    np.bincount(
        training_residual_hours,
        weights=y_train_residual,
        minlength=24,
    )
    / training_hour_counts
)

validation_hourly_residual_mean = (
    np.bincount(
        validation_residual_hours,
        weights=y_validation_residual,
        minlength=24,
    )
    / validation_hour_counts
)

hourly_residual_data = pd.DataFrame({
    "split": np.repeat(
        ["Training", "Validation"],
        24,
    ),
    "hour_of_day": np.tile(
        np.arange(24),
        2,
    ),
    "mean_residual": np.concatenate([
        training_hourly_residual_mean,
        validation_hourly_residual_mean,
    ]),
})

assert residual_chart_data.shape[0] <= 500_000
assert hourly_residual_data.shape[0] == 48
assert residual_chart_data["residual"].notna().all()
assert hourly_residual_data["mean_residual"].notna().all()

In [ ]:
# Limiting the histogram calculation to the central display range.
residual_display_limits = np.percentile(
    residual_chart_data["residual"],
    [1, 99],
)

residual_palette = {
    "Training": "#4C78A8",
    "Validation": "#E07A5F",
}

# Comparing residual distributions and hourly bias.
# Creating vertically stacked charts for narrower screens.
fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 10),
)
sns.histplot(
    data=residual_chart_data,
    x="residual",
    hue="split",
    hue_order=["Training", "Validation"],
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    bins=80,
    binrange=tuple(residual_display_limits),
    palette=residual_palette,
    linewidth=1.7,
    ax=axes[0],
)

axes[0].axvline(
    0,
    color="#444444",
    linestyle="--",
    linewidth=1,
)
axes[0].axvline(
    y_train_residual.mean(),
    color=residual_palette["Training"],
    linestyle=":",
    linewidth=1.5,
)
axes[0].axvline(
    y_validation_residual.mean(),
    color=residual_palette["Validation"],
    linestyle=":",
    linewidth=1.5,
)
axes[0].set_title(
    "Weekly-Baseline Residual Distribution"
)
axes[0].set_xlabel(
    "Actual demand − weekly baseline"
)
axes[0].set_ylabel(
    "Density"
)

sns.lineplot(
    data=hourly_residual_data,
    x="hour_of_day",
    y="mean_residual",
    hue="split",
    hue_order=["Training", "Validation"],
    marker="o",
    palette=residual_palette,
    linewidth=2,
    ax=axes[1],
)

axes[1].axhline(
    0,
    color="#444444",
    linestyle="--",
    linewidth=1,
)
axes[1].set_xticks(
    range(0, 24, 2)
)
axes[1].set_title(
    "Mean Weekly-Baseline Residual by Hour"
)
axes[1].set_xlabel(
    "Target hour of day"
)
axes[1].set_ylabel(
    "Mean residual"
)

for axis in axes:
    axis.grid(
        axis="y",
        alpha=0.20,
    )
    sns.despine(
        ax=axis,
    )

    legend = axis.get_legend()
    if legend is not None:
        legend.set_title("Split")

fig.suptitle(
    "Where Does the Weekly Baseline Need Correction?",
    fontsize=15,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
plt.tight_layout()
fig.subplots_adjust(
    hspace=0.45
)
plt.show()
plt.show()

display(
    residual_summary.round(3)
)

# Checking the residual sign relationship. 
# result = (  

#         print(
#             "Weekly baseline mean bias: "
#             f"{weekly_naive_mean_bias:,.3f}"
#         ),
#         print(
#             "Validation mean residual: "
#             f"{y_validation_residual.mean():,.3f}"
#         ),
#         print(
#             "Bias-residual relationship valid: True"
#         ),
#         print(),
#         print("Hybrid residual targets passed: True")
#         )

### Predicting a correction to last week

There is another way to use the weekly pattern. Instead of predicting pickups from scratch, the model can predict how much to add to or subtract from last week. I'll calculate that residual target and see whether it makes the learning problem easier.

$$
e_t
=
y_t
-
y_{t-168}
$$


In [ ]:
# Defining the categorical columns explicitly.
catboost_categorical_columns = [
    "location_id",
    "forecast_horizon",
    "hour_of_day",
    "day_of_week",
    "month",
]

assert all(
    column in X_train.columns
    for column in catboost_categorical_columns
)

# Building the residual-target CatBoost pools.
hybrid_pool_start = time.perf_counter()

catboost_hybrid_training_pool = Pool(
    data=X_train,
    label=y_train_residual,
    cat_features=catboost_categorical_columns,
)

catboost_hybrid_validation_pool = Pool(
    data=X_validation,
    label=y_validation_residual,
    cat_features=catboost_categorical_columns,
)

hybrid_pool_seconds = (
    time.perf_counter()
    - hybrid_pool_start
)

# Releasing the direct-target pools after successful construction.
del catboost_training_pool
del catboost_validation_pool

gc.collect()

assert (
    catboost_hybrid_training_pool.num_row()
    == len(y_train_residual)
)
assert (
    catboost_hybrid_validation_pool.num_row()
    == len(y_validation_residual)
)
assert (
    catboost_hybrid_training_pool.num_col()
    == X_train.shape[1]
)
assert (
    catboost_hybrid_validation_pool.num_col()
    == X_validation.shape[1]
)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Hybrid training pool rows: "
    f"{catboost_hybrid_training_pool.num_row():,}"
)
print(
    "Hybrid validation pool rows: "
    f"{catboost_hybrid_validation_pool.num_row():,}"
)
print(
    "Pool columns: "
    f"{catboost_hybrid_training_pool.num_col():,}"
)
print(
    "Build time (seconds): "
    f"{hybrid_pool_seconds:,.2f}"
)
print(
    "Notebook process RAM (GB): "
    f"{process_ram_gb:,.2f}"
)
print()


### Training the CatBoost residual-correction model

The predictors stay the same, but CatBoost now learns the difference between actual pickups and the weekly baseline. After prediction, I'll add that correction back to last week's count. This keeps the comparison focused on the target formulation.


In [ ]:
# Reusing the retained CatBoost specification.
catboost_hybrid_params = (
    catboost_depth10_params.copy()
)

catboost_hybrid_model = CatBoostRegressor(
    **catboost_hybrid_params
)

# Training on earlier-period residuals.
catboost_hybrid_start = time.perf_counter()

catboost_hybrid_model.fit(
    catboost_hybrid_training_pool,
    eval_set=catboost_hybrid_validation_pool,
    use_best_model=True,
)

catboost_hybrid_minutes = (
    time.perf_counter()
    - catboost_hybrid_start
) / 60

# Collecting later-period validation results.
catboost_hybrid_best_iteration = (
    catboost_hybrid_model.get_best_iteration()
)

catboost_hybrid_best_round = (
    catboost_hybrid_best_iteration + 1
)

catboost_hybrid_best_rmse = (
    catboost_hybrid_model
    .get_best_score()["validation"]["RMSE"]
)

baseline_residual_rmse = np.sqrt(
    np.mean(
        np.square(y_validation_residual)
    )
)

hybrid_residual_rmse_change = (
    catboost_hybrid_best_rmse
    - baseline_residual_rmse
)

assert catboost_hybrid_best_round >= 1
assert np.isfinite(
    catboost_hybrid_best_rmse
)

print(
    "Best boosting round: "
    f"{catboost_hybrid_best_round:,}"
)
print(
    "Best validation residual RMSE: "
    f"{catboost_hybrid_best_rmse:,.4f}"
)
print(
    "Uncorrected baseline residual RMSE: "
    f"{baseline_residual_rmse:,.4f}"
)
print(
    "Residual RMSE change: "
    f"{hybrid_residual_rmse_change:+.4f}"
)
print(
    "Training time (minutes): "
    f"{catboost_hybrid_minutes:,.2f}"
)
print()


### Interpreting the hybrid training result

The correction model improves on assuming that every residual is zero. But an improvement over the weekly baseline is only part of the question. I'll reconstruct the pickup forecasts and compare them with the direct models on the same validation rows.


In [ ]:
# Predicting residual corrections for the later validation period.
hybrid_prediction_start = time.perf_counter()

catboost_hybrid_residual_predictions = (
    catboost_hybrid_model.predict(
        catboost_hybrid_validation_pool
    )
)

hybrid_prediction_seconds = (
    time.perf_counter()
    - hybrid_prediction_start
)

# Combining the weekly baseline and predicted correction.
catboost_hybrid_predictions_raw = (
    validation_weekly_baseline
    + catboost_hybrid_residual_predictions
)

# Applying the demand constraint after combination.
catboost_hybrid_predictions_constrained = np.clip(
    catboost_hybrid_predictions_raw,
    a_min=0,
    a_max=None,
)

hybrid_negative_count = int(
    (catboost_hybrid_predictions_raw < 0).sum()
)

hybrid_negative_rate = (
    100
    * hybrid_negative_count
    / len(catboost_hybrid_predictions_raw)
)

hybrid_raw_rmse = np.sqrt(
    np.mean(
        np.square(
            np.asarray(y_validation)
            - catboost_hybrid_predictions_raw
        )
    )
)

# Checking prediction alignment and numerical validity.
assert len(
    catboost_hybrid_residual_predictions
) == len(y_validation)

assert np.isfinite(
    catboost_hybrid_residual_predictions
).all()

assert np.isfinite(
    catboost_hybrid_predictions_raw
).all()

assert (
    catboost_hybrid_predictions_constrained >= 0
).all()

assert np.isclose(
    hybrid_raw_rmse,
    catboost_hybrid_best_rmse,
    atol=1e-3,
)

print(
    "Prediction time (seconds): "
    f"{hybrid_prediction_seconds:,.2f}"
)
print(
    "Mean predicted residual correction: "
    f"{catboost_hybrid_residual_predictions.mean():,.3f}"
)
print(
    "Raw hybrid RMSE: "
    f"{hybrid_raw_rmse:,.4f}"
)
print(
    "Minimum raw demand prediction: "
    f"{catboost_hybrid_predictions_raw.min():,.3f}"
)
print(
    "Negative demand predictions: "
    f"{hybrid_negative_count:,}"
)
print(
    "Negative prediction rate: "
    f"{hybrid_negative_rate:,.4f}%"
)
print()


In [ ]:
# Comparing raw and constrained hybrid predictions.
hybrid_model_comparison = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions={
            "Weekly Seasonal Naive": (
                weekly_naive_predictions
            ),
            "XGBoost": (
                initial_xgb_predictions_constrained
            ),
            "Direct CatBoost": (
                catboost_depth10_predictions_constrained
            ),
            "Hybrid CatBoost Raw": (
                catboost_hybrid_predictions_raw
            ),
            "Hybrid CatBoost Constrained": (
                catboost_hybrid_predictions_constrained
            ),
        },
    )
)

display(
    hybrid_model_comparison.round(3)
)

# Measuring the effect of the demand constraint.
hybrid_raw_metrics = (
    hybrid_model_comparison.loc[
        hybrid_model_comparison["model"]
        == "Hybrid CatBoost Raw"
    ].iloc[0]
)

hybrid_constrained_metrics = (
    hybrid_model_comparison.loc[
        hybrid_model_comparison["model"]
        == "Hybrid CatBoost Constrained"
    ].iloc[0]
)

print(
    "RMSE change from clipping: "
    f"{(
        hybrid_constrained_metrics['rmse']
        - hybrid_raw_metrics['rmse']
    ):+.4f}"
)
print(
    "WAPE change from clipping: "
    f"{(
        hybrid_constrained_metrics['wape_percent']
        - hybrid_raw_metrics['wape_percent']
    ):+.4f} percentage points"
)

### Is it better to forecast the count or the correction?

The hybrid improves over the weekly baseline, but the direct XGBoost candidate is still stronger here. So residual correction is useful as an alternative, not an automatic upgrade. I'll now look at whether the overall ranking changes across operating conditions.


In [ ]:
# Preparing aligned validation arrays used by later diagnostics.
validation_actual = np.asarray(
    y_validation
)

validation_horizons = (
    X_validation[
        "forecast_horizon"
    ]
    .astype("int8")
    .to_numpy()
)

# Preparing a lightweight frame for grouped metric calculation.
validation_horizon_comparison_frame = pd.DataFrame({
    "forecast_horizon": validation_horizons,
    "total_pickups": validation_actual,
    "weekly_naive_prediction": (
        weekly_naive_predictions
    ),
    "xgboost_prediction": (
        initial_xgb_predictions_constrained
    ),
    "direct_catboost_prediction": (
        catboost_depth10_predictions_constrained
    ),
    "hybrid_catboost_prediction": (
        catboost_hybrid_predictions_constrained
    ),
})

# Calculating every model within each forecast horizon.
horizon_model_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=(
            validation_horizon_comparison_frame
        ),
        group_columns="forecast_horizon",
        prediction_columns={
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
            "XGBoost": (
                "xgboost_prediction"
            ),
            "Direct CatBoost": (
                "direct_catboost_prediction"
            ),
            "Hybrid CatBoost": (
                "hybrid_catboost_prediction"
            ),
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Restoring the original model-first reporting order.
horizon_comparison_model_order = {
    "Weekly Seasonal Naive": 0,
    "XGBoost": 1,
    "Direct CatBoost": 2,
    "Hybrid CatBoost": 3,
}

horizon_model_comparison[
    "_model_order"
] = horizon_model_comparison[
    "model"
].map(horizon_comparison_model_order)

horizon_model_comparison = (
    horizon_model_comparison
    .sort_values(
        [
            "_model_order",
            "forecast_horizon",
        ]
    )
    .drop(columns="_model_order")
    .reset_index(drop=True)
)

# Checking complete and balanced horizon coverage.
horizon_row_counts = (
    horizon_model_comparison[
        [
            "forecast_horizon",
            "rows",
        ]
    ]
    .drop_duplicates()
    .sort_values("forecast_horizon")
)

assert len(horizon_row_counts) == 6
assert horizon_row_counts["rows"].nunique() == 1
assert horizon_model_comparison.shape[0] == 24

assert horizon_model_comparison[
    [
        "mae",
        "rmse",
        "wape_percent",
        "mean_bias",
        "r_squared",
    ]
].notna().all().all()

assert (
    horizon_model_comparison
    .groupby(
        "model",
        observed=True,
    )["rows"]
    .sum()
    .eq(len(validation_actual))
    .all()
)

display(
    horizon_model_comparison.round(3)
)

print(
    "Rows per model-horizon group: "
    f"{horizon_row_counts['rows'].iloc[0]:,}"
)

print()

print(
    "Horizon model comparison passed:",
    True,
)

# Releasing only the temporary grouped-metric frame.
del validation_horizon_comparison_frame

### Comparing performance across forecast horizons

Every horizon has the same number of validation rows. Among these four candidates, XGBoost has the lowest MAE, RMSE, and WAPE throughout the six-hour window. I'll keep looking for exceptions rather than assume that ranking holds everywhere.


In [ ]:
# Preparing the target-hour values.
validation_hours = (
    X_validation[
        "hour_of_day"
    ]
    .astype("int8")
    .to_numpy()
)

# Preserving the aligned prediction arrays for later diagnostics.
hour_model_predictions = {
    "Weekly Seasonal Naive": (
        weekly_naive_predictions
    ),
    "XGBoost": (
        initial_xgb_predictions_constrained
    ),
    "Direct CatBoost": (
        catboost_depth10_predictions_constrained
    ),
    "Hybrid CatBoost": (
        catboost_hybrid_predictions_constrained
    ),
}

# Preparing a lightweight frame for hourly metric calculation.
validation_hour_comparison_frame = pd.DataFrame({
    "hour_of_day": validation_hours,
    "total_pickups": validation_actual,
    "weekly_naive_prediction": (
        weekly_naive_predictions
    ),
    "xgboost_prediction": (
        initial_xgb_predictions_constrained
    ),
    "direct_catboost_prediction": (
        catboost_depth10_predictions_constrained
    ),
    "hybrid_catboost_prediction": (
        catboost_hybrid_predictions_constrained
    ),
})

# Calculating every model within each local target hour.
hour_model_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=(
            validation_hour_comparison_frame
        ),
        group_columns="hour_of_day",
        prediction_columns={
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
            "XGBoost": (
                "xgboost_prediction"
            ),
            "Direct CatBoost": (
                "direct_catboost_prediction"
            ),
            "Hybrid CatBoost": (
                "hybrid_catboost_prediction"
            ),
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Restoring the original model-first reporting order.
hour_comparison_model_order = {
    model_name: position
    for position, model_name in enumerate(
        hour_model_predictions
    )
}

hour_model_comparison[
    "_model_order"
] = hour_model_comparison[
    "model"
].map(hour_comparison_model_order)

hour_model_comparison = (
    hour_model_comparison
    .sort_values(
        [
            "_model_order",
            "hour_of_day",
        ]
    )
    .drop(columns="_model_order")
    .reset_index(drop=True)
)

# Checking complete hourly coverage for every model.
assert hour_model_comparison.shape[0] == 96

assert set(
    hour_model_comparison[
        "hour_of_day"
    ].astype(int)
) == set(range(24))

assert hour_model_comparison[
    [
        "wape_percent",
        "mean_bias",
    ]
].notna().all().all()

assert (
    hour_model_comparison
    .groupby(
        "model",
        observed=True,
    )["rows"]
    .sum()
    .eq(len(validation_actual))
    .all()
)

# Releasing only the temporary grouped-metric frame.
del validation_hour_comparison_frame

# Defining consistent model colors.
hour_model_palette = {
    "Weekly Seasonal Naive": "#9C9C9C",
    "XGBoost": "#4C78A8",
    "Direct CatBoost": "#59A14F",
    "Hybrid CatBoost": "#E07A5F",
}

# Comparing hourly relative error and bias.
fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 10),
)

sns.lineplot(
    data=hour_model_comparison,
    x="hour_of_day",
    y="wape_percent",
    hue="model",
    marker="o",
    linewidth=2,
    palette=hour_model_palette,
    ax=axes[0],
)

axes[0].set_title(
    "Validation WAPE by Target Hour"
)
axes[0].set_xlabel("")
axes[0].set_ylabel("WAPE (%)")
axes[0].set_xticks(range(0, 24, 2))

sns.lineplot(
    data=hour_model_comparison,
    x="hour_of_day",
    y="mean_bias",
    hue="model",
    marker="o",
    linewidth=2,
    palette=hour_model_palette,
    ax=axes[1],
)

axes[1].axhline(
    0,
    color="#444444",
    linestyle="--",
    linewidth=1,
)
axes[1].set_title(
    "Validation Mean Bias by Target Hour"
)
axes[1].set_xlabel(
    "Target hour of day"
)
axes[1].set_ylabel(
    "Mean bias"
)
axes[1].set_xticks(range(0, 24, 2))

for axis in axes:
    axis.grid(
        axis="y",
        alpha=0.20,
    )
    sns.despine(
        ax=axis,
    )

    legend = axis.get_legend()
    if legend is not None:
        legend.set_title("Model")

fig.suptitle(
    "How Does Model Performance Change Across the Day?",
    fontsize=15,
    fontweight="bold",
    y=1.01,
)

plt.tight_layout()
fig.subplots_adjust(
    hspace=0.35
)
plt.show()



### Where in the day does the model help most?

XGBoost also leads these candidates on WAPE at every hour. The early morning remains difficult in relative terms, when pickup totals are small. I'll read those percentages alongside absolute errors so a quiet hour does not look more important just because its denominator is smaller.


In [ ]:
# Calculating mean demand by zone from the earlier training period.
training_location_ids = (
    X_train["location_id"]
    .to_numpy(dtype=np.int32, copy=False)
)

training_actual = np.asarray(
    y_train,
    dtype=np.float32,
)

zone_training_counts = np.bincount(
    training_location_ids,
    minlength=264,
)

zone_training_totals = np.bincount(
    training_location_ids,
    weights=training_actual,
    minlength=264,
)

zone_training_activity = pd.Series(
    zone_training_totals[1:] / zone_training_counts[1:],
    index=np.arange(1, 264),
    name="training_mean_demand",
)

# Defining activity groups from training demand only.
activity_labels = [
    "Low",
    "Medium-Low",
    "Medium-High",
    "High",
]

zone_activity_groups = pd.qcut(
    zone_training_activity,
    q=4,
    labels=activity_labels,
)

zone_activity_lookup = pd.DataFrame({
    "training_mean_demand": zone_training_activity,
    "activity_level": zone_activity_groups,
})

# Mapping the fixed training-defined groups to validation rows.
validation_activity_levels = (
    X_validation["location_id"]
    .map(zone_activity_groups)
)

assert zone_training_counts[1:].min() > 0
assert zone_activity_lookup.shape[0] == 263
assert validation_activity_levels.notna().all()
assert set(validation_activity_levels.unique()) == set(activity_labels)

display(
    zone_activity_lookup
    .groupby("activity_level", observed=True)
    .agg(
        zones=("training_mean_demand", "size"),
        minimum_training_mean=("training_mean_demand", "min"),
        median_training_mean=("training_mean_demand", "median"),
        maximum_training_mean=("training_mean_demand", "max"),
    )
    .round(2)
)



In [ ]:
# Defining the fixed activity order.
activity_order = [
    "Low",
    "Medium-Low",
    "Medium-High",
    "High",
]

# Preparing aligned validation data by training-defined activity level.
validation_activity_frame = pd.DataFrame({
    "activity_level": (
        validation_activity_levels
        .astype(str)
        .to_numpy()
    ),
    "total_pickups": validation_actual,
    "weekly_naive_prediction": (
        weekly_naive_predictions
    ),
    "xgboost_prediction": (
        initial_xgb_predictions_constrained
    ),
    "direct_catboost_prediction": (
        catboost_depth10_predictions_constrained
    ),
    "hybrid_catboost_prediction": (
        catboost_hybrid_predictions_constrained
    ),
})

validation_activity_frame[
    "activity_level"
] = pd.Categorical(
    validation_activity_frame[
        "activity_level"
    ],
    categories=activity_order,
    ordered=True,
)

# Calculating every model within each activity group.
activity_model_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=(
            validation_activity_frame
        ),
        group_columns="activity_level",
        prediction_columns={
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
            "XGBoost": (
                "xgboost_prediction"
            ),
            "Direct CatBoost": (
                "direct_catboost_prediction"
            ),
            "Hybrid CatBoost": (
                "hybrid_catboost_prediction"
            ),
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Preserving the established activity and model order.
activity_model_order = {
    model_name: position
    for position, model_name in enumerate(
        hour_model_predictions
    )
}

activity_model_comparison[
    "_model_order"
] = activity_model_comparison[
    "model"
].map(activity_model_order)

activity_model_comparison[
    "activity_level"
] = pd.Categorical(
    activity_model_comparison[
        "activity_level"
    ],
    categories=activity_order,
    ordered=True,
)

activity_model_comparison = (
    activity_model_comparison
    .sort_values(
        [
            "_model_order",
            "activity_level",
        ]
    )
    .drop(columns="_model_order")
    .reset_index(drop=True)
)

# Checking complete activity-group coverage.
assert activity_model_comparison.shape[0] == 16
assert activity_model_comparison["rows"].gt(0).all()

assert activity_model_comparison[
    [
        "mae",
        "rmse",
        "wape_percent",
        "mean_bias",
    ]
].notna().all().all()

assert (
    activity_model_comparison
    .groupby(
        "model",
        observed=True,
    )["rows"]
    .sum()
    .eq(len(validation_actual))
    .all()
)

# Releasing only the temporary grouped-metric frame.
del validation_activity_frame

# Defining short descriptions for the validation metrics.
from IPython.display import Markdown as NotebookMarkdown

# Defining short descriptions for the validation metrics.
display(NotebookMarkdown(
    """
**How to read these tables**

- **MAE:** average prediction error in pickups.
- **RMSE:** gives more weight to unusually large errors.
- **WAPE:** total absolute error relative to total demand.
- **Mean bias:** negative values indicate underprediction; positive values indicate overprediction.

The activity groups were defined from the earlier training period.  
The metrics below come only from the later validation period.

Each table is sorted from lowest to highest RMSE's of models.
"""
))

# Displaying one comparison table for each activity group.
for activity_level in activity_order:
    group_zone_summary = (
        zone_activity_lookup.loc[
            zone_activity_lookup["activity_level"]
            == activity_level,
            "training_mean_demand",
        ]
    )

    group_results = (
        activity_model_comparison.loc[
            activity_model_comparison["activity_level"]
            == activity_level,
            [
                "model",
                "mae",
                "rmse",
                "wape_percent",
                "mean_bias",
            ],
        ]
        .sort_values("rmse")
        .rename(columns={
            "model": "Model",
            "mae": "MAE",
            "rmse": "RMSE",
            "wape_percent": "WAPE (%)",
            "mean_bias": "Mean Bias",
        })
        .reset_index(drop=True)
    )

    zones_in_group = int(group_zone_summary.shape[0])
    minimum_mean = group_zone_summary.min()
    maximum_mean = group_zone_summary.max()

    display(NotebookMarkdown(
        f"""
### {activity_level}-activity zones

This group contains **{zones_in_group} zones** with average training
demand from **{minimum_mean:.2f} to {maximum_mean:.2f} pickups per hour**.
Lower MAE and RMSE indicate better absolute accuracy, while WAPE makes
the error relative to this group's demand volume.
"""
    ))

    display(
        group_results.style
        .format({
            "MAE": "{:.3f}",
            "RMSE": "{:.3f}",
            "WAPE (%)": "{:.3f}",
            "Mean Bias": "{:+.3f}",
        })
        .hide(axis="index")
    )



In [ ]:
# Preparing the ordered activity comparison data.
activity_chart_data = (
    activity_model_comparison
    .sort_values(["activity_level", "model"])
    .copy()
)

# Comparing relative and absolute errors by activity level.
fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 11),
)

sns.lineplot(
    data=activity_chart_data,
    x="activity_level",
    y="wape_percent",
    hue="model",
    marker="o",
    markersize=8,
    linewidth=2.2,
    palette=hour_model_palette,
    ax=axes[0],
    hue_order=[
        "XGBoost",
        "Direct CatBoost",
        "Hybrid CatBoost",
        "Weekly Seasonal Naive",
    ],
)

axes[0].set_title(
    "Relative Error by Zone Activity Level",
    loc="left",
)
axes[0].set_xlabel("")
axes[0].set_ylabel("WAPE (%)")

sns.lineplot(
    data=activity_chart_data,
    x="activity_level",
    y="rmse",
    hue="model",
    marker="o",
    markersize=8,
    linewidth=2.2,
    palette=hour_model_palette,
    ax=axes[1],
    hue_order=[
        "XGBoost",
        "Direct CatBoost",
        "Hybrid CatBoost",
        "Weekly Seasonal Naive",
    ],
)

axes[1].set_title(
    "Absolute Error by Zone Activity Level",
    loc="left",
)
axes[1].set_xlabel(
    "Training-defined zone activity level"
)
axes[1].set_ylabel("RMSE")

# Formatting both charts consistently.
for axis in axes:
    axis.grid(
        axis="y",
        alpha=0.20,
    )
    sns.despine(ax=axis)

axes[0].legend(
    title="Model",
    frameon=True,
)

second_legend = axes[1].get_legend()

if second_legend is not None:
    second_legend.remove()

fig.suptitle(
    "Does Model Performance Remain Stable Across Zone Activity Levels?",
    fontsize=15,
    fontweight="bold",
    x=0.06,
    ha="left",
    y=1.01,
)

fig.text(
    0.06,
    0.975,
    (
        "Activity groups use earlier training demand; "
        "errors use the later validation period."
    ),
    fontsize=10.5,
    color="#555555",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)

fig.subplots_adjust(
    hspace=0.35
)

plt.show()

### Does it help in both quiet and busy zones?

Busy zones have larger errors measured in pickups, while quieter zones can have larger percentage errors. Both matter, but they answer different questions. Comparing RMSE and WAPE together gives a more useful picture than choosing one ranking for all locations.


In [ ]:
# Calculating each zone's training-only 95th-percentile threshold.
training_demand_series = pd.Series(
    training_actual,
    copy=False,
)

training_location_series = pd.Series(
    training_location_ids,
    copy=False,
)

zone_extreme_thresholds = (
    training_demand_series
    .groupby(training_location_series)
    .quantile(0.95)
    .rename("training_p95_demand")
)

# Mapping the fixed thresholds to later validation rows.
validation_location_ids = (
    X_validation["location_id"]
    .to_numpy(dtype=np.int32, copy=False)
)

validation_extreme_thresholds = (
    pd.Series(validation_location_ids)
    .map(zone_extreme_thresholds)
    .to_numpy()
)

validation_is_extreme = (
    validation_actual
    > validation_extreme_thresholds
)

# Checking the training-defined segmentation.
assert zone_extreme_thresholds.shape[0] == 263
assert zone_extreme_thresholds.notna().all()
assert np.isfinite(validation_extreme_thresholds).all()
assert validation_is_extreme.any()
assert (~validation_is_extreme).any()

extreme_definition_summary = pd.DataFrame({
    "Measure": [
        "Zones with training thresholds",
        "Minimum zone-specific p95",
        "Median zone-specific p95",
        "Maximum zone-specific p95",
        "Ordinary validation rows",
        "Extreme validation rows",
        "Extreme validation share (%)",
    ],
    "Value": [
        zone_extreme_thresholds.shape[0],
        zone_extreme_thresholds.min(),
        zone_extreme_thresholds.median(),
        zone_extreme_thresholds.max(),
        (~validation_is_extreme).sum(),
        validation_is_extreme.sum(),
        validation_is_extreme.mean() * 100,
    ],
})

display(
    extreme_definition_summary.style
    .format({"Value": "{:,.2f}"})
    .hide(axis="index")
)

More validation rows exceed the training 95th percentile than I expected: 8.66%, rather than roughly 5%. This could reflect higher later pickups, but sparse zones with zero thresholds might also distort the count. I'll check that before interpreting the increase.


In [ ]:
# Mapping zone-specific thresholds back to training rows.
training_extreme_thresholds = (
    training_location_series
    .map(zone_extreme_thresholds)
    .to_numpy()
)

training_is_extreme = (
    training_actual
    > training_extreme_thresholds
)

# Identifying zones whose training p95 equals zero.
zero_threshold_zones = (
    zone_extreme_thresholds[
        zone_extreme_thresholds == 0
    ]
    .index
    .to_numpy()
)

validation_has_zero_threshold = np.isin(
    validation_location_ids,
    zero_threshold_zones,
)

# Summarizing the extreme-demand definition.
extreme_threshold_audit = pd.DataFrame({
    "Measure": [
        "Training extreme share (%)",
        "Validation extreme share (%)",
        "Zones with p95 equal to zero",
        "Validation rows from zero-threshold zones",
        "Extreme share within zero-threshold zones (%)",
        "Extreme share within other zones (%)",
    ],
    "Value": [
        training_is_extreme.mean() * 100,
        validation_is_extreme.mean() * 100,
        len(zero_threshold_zones),
        validation_has_zero_threshold.sum(),
        validation_is_extreme[
            validation_has_zero_threshold
        ].mean() * 100,
        validation_is_extreme[
            ~validation_has_zero_threshold
        ].mean() * 100,
    ],
})

display(
    extreme_threshold_audit.style
    .format({"Value": "{:,.3f}"})
    .hide(axis="index")
)

The sparse zones are not driving it. Their share above the threshold is small, while the other zones show the increase. I'll keep the training thresholds fixed; redefining them from validation would hide the change I am trying to examine.


In [ ]:
# Defining ordinary and extreme validation segments.
demand_segment_masks = {
    "Ordinary demand": ~validation_is_extreme,
    "Zone-relative extreme demand": (
        validation_is_extreme
    ),
}

# Preparing aligned validation data by demand segment.
validation_extreme_frame = pd.DataFrame({
    "demand_segment": np.where(
        validation_is_extreme,
        "Zone-relative extreme demand",
        "Ordinary demand",
    ),
    "total_pickups": validation_actual,
    "weekly_naive_prediction": (
        weekly_naive_predictions
    ),
    "xgboost_prediction": (
        initial_xgb_predictions_constrained
    ),
    "direct_catboost_prediction": (
        catboost_depth10_predictions_constrained
    ),
    "hybrid_catboost_prediction": (
        catboost_hybrid_predictions_constrained
    ),
})

# Calculating every model within each demand segment.
extreme_model_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=validation_extreme_frame,
        group_columns="demand_segment",
        prediction_columns={
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
            "XGBoost": (
                "xgboost_prediction"
            ),
            "Direct CatBoost": (
                "direct_catboost_prediction"
            ),
            "Hybrid CatBoost": (
                "hybrid_catboost_prediction"
            ),
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Checking complete demand-segment coverage.
assert extreme_model_comparison.shape[0] == 8
assert extreme_model_comparison["rows"].gt(0).all()

assert extreme_model_comparison[
    [
        "mae",
        "rmse",
        "wape_percent",
        "mean_bias",
    ]
].notna().all().all()

assert (
    extreme_model_comparison
    .groupby(
        "model",
        observed=True,
    )["rows"]
    .sum()
    .eq(len(validation_actual))
    .all()
)

# Releasing the temporary grouped-metric frame.
del validation_extreme_frame

# Displaying one table for each demand segment.
for segment_name in demand_segment_masks:
    segment_table = (
        extreme_model_comparison.loc[
            extreme_model_comparison[
                "demand_segment"
            ]
            == segment_name,
            [
                "model",
                "rows",
                "mae",
                "rmse",
                "wape_percent",
                "mean_bias",
            ],
        ]
        .sort_values("rmse")
        .rename(columns={
            "model": "Model",
            "rows": "Rows",
            "mae": "MAE",
            "rmse": "RMSE",
            "wape_percent": "WAPE (%)",
            "mean_bias": "Mean Bias",
        })
        .reset_index(drop=True)
    )

    display(NotebookMarkdown(
        f"### {segment_name}"
    ))

    display(
        segment_table.style
        .format({
            "Rows": "{:,.0f}",
            "MAE": "{:.3f}",
            "RMSE": "{:.3f}",
            "WAPE (%)": "{:.3f}",
            "Mean Bias": "{:+.3f}",
        })
        .hide(axis="index")
    )

Here the result is less clear-cut. XGBoost has the lowest extreme-period RMSE, but direct CatBoost has slightly lower MAE and WAPE. All models underpredict these hours. I would not call one universally best from such small differences across competing metrics.


In [ ]:
# Defining the presentation order.
segment_order = [
    "Ordinary demand",
    "Zone-relative extreme demand",
]

model_order = [
    "XGBoost",
    "Direct CatBoost",
    "Hybrid CatBoost",
    "Weekly Seasonal Naive",
]

# Defining consistent model colors.
activity_model_palette = {
    "XGBoost": "#2F6B9A",
    "Direct CatBoost": "#78A96B",
    "Hybrid CatBoost": "#D98C68",
    "Weekly Seasonal Naive": "#A6A6A6",
}

# Comparing extreme-demand accuracy and bias.
fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 10),
)

sns.barplot(
    data=extreme_model_comparison,
    x="demand_segment",
    y="rmse",
    hue="model",
    order=segment_order,
    hue_order=model_order,
    palette=activity_model_palette,
    errorbar=None,
    ax=axes[0],
)

axes[0].set_title(
    "RMSE Increases Substantially During Extreme Demand",
    loc="left",
)
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE")

sns.barplot(
    data=extreme_model_comparison,
    x="demand_segment",
    y="mean_bias",
    hue="model",
    order=segment_order,
    hue_order=model_order,
    palette=activity_model_palette,
    errorbar=None,
    ax=axes[1],
    
)

axes[1].axhline(
    0,
    color="#4B5563",
    linestyle="--",
    linewidth=1,
)

axes[1].set_title(
    "All Models Underpredict Zone-Relative Extremes",
    loc="left",
)
axes[1].set_xlabel("Validation demand segment")
axes[1].set_ylabel("Mean bias")

# Adding values directly to the bars.
for axis in axes:
    for container in axis.containers:
        axis.bar_label(
            container,
            fmt="%.1f",
            padding=3,
            fontsize=9,
        )
        axis.grid(False)

axes[0].legend(
    title="Model",
    loc="upper left",
    frameon=False,
)

second_legend = axes[1].get_legend()
if second_legend is not None:
    second_legend.remove()

fig.suptitle(
    "How Do the Models Perform When Demand Becomes Unusually High?",
    fontsize=15,
    fontweight="bold",
    x=0.06,
    ha="left",
    y=1.01,
)

fig.text(
    0.06,
    0.975,
    "Extreme demand exceeds each zone's training-only 95th-percentile threshold.",
    fontsize=10.5,
    color="#555555",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.95]
)
fig.subplots_adjust(
    hspace=0.35
)

plt.show()

### How do the models handle unusually busy hours?

The weekly pattern helps on ordinary hours, but unusually high activity remains a weakness. The extreme group is defined using actual outcomes, so some negative bias is expected from that selection. I still want to know whether the same tendency appears when grouping by forecasts, which are available before the outcome.


In [ ]:
# Calculating training zero-demand frequency by zone.
training_zero_counts = np.bincount(
    training_location_ids,
    weights=(training_actual == 0),
    minlength=264,
)

zone_training_zero_percent = pd.Series(
    (
        training_zero_counts[1:]
        / zone_training_counts[1:]
    ) * 100,
    index=np.arange(1, 264),
    name="training_zero_percent",
)

# Identifying structurally sparse zones from training only.
high_zero_zone_ids = (
    zone_training_zero_percent[
        zone_training_zero_percent >= 90
    ]
    .index
    .to_numpy()
)

validation_is_high_zero_zone = np.isin(
    validation_location_ids,
    high_zero_zone_ids,
)

# Summarizing their later validation behavior.
high_zero_zone_summary = pd.DataFrame({
    "location_id": high_zero_zone_ids,
    "training_zero_percent": (
        zone_training_zero_percent.loc[
            high_zero_zone_ids
        ].to_numpy()
    ),
    "validation_zero_percent": [
        (
            validation_actual[
                validation_location_ids == location_id
            ] == 0
        ).mean() * 100
        for location_id in high_zero_zone_ids
    ],
    "validation_mean_demand": [
        validation_actual[
            validation_location_ids == location_id
        ].mean()
        for location_id in high_zero_zone_ids
    ],
})

assert len(high_zero_zone_ids) > 0
assert validation_is_high_zero_zone.any()
assert (~validation_is_high_zero_zone).any()
assert high_zero_zone_summary[
    [
        "training_zero_percent",
        "validation_zero_percent",
        "validation_mean_demand",
    ]
].notna().all().all()

display(
    high_zero_zone_summary.style
    .format({
        "location_id": "{:.0f}",
        "training_zero_percent": "{:.2f}",
        "validation_zero_percent": "{:.2f}",
        "validation_mean_demand": "{:.3f}",
    })
    .hide(axis="index")
)

print(
    "Training-defined high-zero zones:",
    len(high_zero_zone_ids),
)
print(
    "Validation rows in these zones:",
    f"{validation_is_high_zero_zone.sum():,}",
)

The seven sparse zones remain nearly always zero in validation. WAPE can be unstable with so little activity, so the next comparison will emphasize pickup-unit errors and bias instead of treating a large percentage as a large operational loss.


In [ ]:
# Defining the training-based zone segments.
zero_zone_segment_masks = {
    "At least 90% zero-demand hours": (
        validation_is_high_zero_zone
    ),
    "Other zones": (
        ~validation_is_high_zero_zone
    ),
}

# Preparing aligned validation data by zone segment.
validation_zero_zone_frame = pd.DataFrame({
    "zone_segment": np.where(
        validation_is_high_zero_zone,
        "At least 90% zero-demand hours",
        "Other zones",
    ),
    "total_pickups": validation_actual,
    "weekly_naive_prediction": (
        weekly_naive_predictions
    ),
    "xgboost_prediction": (
        initial_xgb_predictions_constrained
    ),
    "direct_catboost_prediction": (
        catboost_depth10_predictions_constrained
    ),
    "hybrid_catboost_prediction": (
        catboost_hybrid_predictions_constrained
    ),
})

# Calculating every model within each zone segment.
zero_zone_model_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=validation_zero_zone_frame,
        group_columns="zone_segment",
        prediction_columns={
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
            "XGBoost": (
                "xgboost_prediction"
            ),
            "Direct CatBoost": (
                "direct_catboost_prediction"
            ),
            "Hybrid CatBoost": (
                "hybrid_catboost_prediction"
            ),
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Calculating total actual demand once per zone segment.
zero_zone_actual_demand = (
    validation_zero_zone_frame
    .groupby(
        "zone_segment",
        sort=False,
    )["total_pickups"]
    .sum()
)

zero_zone_model_comparison[
    "actual_demand"
] = zero_zone_model_comparison[
    "zone_segment"
].map(zero_zone_actual_demand)

# Checking complete zone-segment coverage.
assert zero_zone_model_comparison.shape[0] == 8
assert zero_zone_model_comparison["rows"].gt(0).all()

assert zero_zone_model_comparison[
    [
        "actual_demand",
        "mae",
        "rmse",
        "wape_percent",
        "mean_bias",
    ]
].notna().all().all()

assert (
    zero_zone_model_comparison
    .groupby("model")["rows"]
    .sum()
    .eq(len(validation_actual))
    .all()
)

# Releasing the temporary grouped-metric objects.
del validation_zero_zone_frame
del zero_zone_actual_demand

# Displaying the segments separately for readability.
for segment_name in zero_zone_segment_masks:
    segment_table = (
        zero_zone_model_comparison.loc[
            zero_zone_model_comparison["zone_segment"]
            == segment_name,
            [
                "model",
                "rows",
                "actual_demand",
                "mae",
                "rmse",
                "wape_percent",
                "mean_bias",
            ],
        ]
        .sort_values("rmse")
        .rename(columns={
            "model": "Model",
            "rows": "Rows",
            "actual_demand": "Total Actual Demand",
            "mae": "MAE",
            "rmse": "RMSE",
            "wape_percent": "WAPE (%)",
            "mean_bias": "Mean Bias",
        })
        .reset_index(drop=True)
    )

    display(NotebookMarkdown(
        f"### {segment_name}"
    ))

    display(
        segment_table.style
        .format({
            "Rows": "{:,.0f}",
            "Total Actual Demand": "{:,.0f}",
            "MAE": "{:.4f}",
            "RMSE": "{:.4f}",
            "WAPE (%)": "{:.2f}",
            "Mean Bias": "{:+.4f}",
        })
        .hide(axis="index")
    )

This is a clear exception to the overall result. In these seven zones, repeating last week has RMSE about 0.17, compared with 0.93 for XGBoost. The global model often predicts a little activity where almost none occurs. That suggests a simple fallback is worth testing later.


In [ ]:
# Creating a sparse-zone-aware production prediction.
sparse_aware_xgb_predictions = (
    initial_xgb_predictions_constrained.copy()
)

sparse_aware_xgb_predictions[
    validation_is_high_zero_zone
] = weekly_naive_predictions[
    validation_is_high_zero_zone
]

# Comparing the routed forecast with the original XGBoost model.
sparse_routing_comparison = pd.DataFrame([
    {
        "model": "XGBoost",
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "Sparse-aware XGBoost",
        **calculate_forecast_metrics(
            validation_actual,
            sparse_aware_xgb_predictions,
        ),
    },
])

# Calculating the practical size of the change.
sparse_routing_comparison["changed_rows"] = [
    0,
    int(validation_is_high_zero_zone.sum()),
]

assert sparse_aware_xgb_predictions.shape == (
    initial_xgb_predictions_constrained.shape
)
assert np.array_equal(
    sparse_aware_xgb_predictions[
        ~validation_is_high_zero_zone
    ],
    initial_xgb_predictions_constrained[
        ~validation_is_high_zero_zone
    ],
)

display(
    sparse_routing_comparison[
        [
            "model",
            "changed_rows",
            "mae",
            "rmse",
            "wape_percent",
            "mean_bias",
            "r_squared",
        ]
    ]
    .rename(columns={
        "model": "Model",
        "changed_rows": "Changed Rows",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "Changed Rows": "{:,.0f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

### Where the simpler forecast still wins

I'll keep these zones in the reported evaluation so the weakness remains visible. A training-defined sparse-zone rule might help, but it needs a separate later comparison. The pooled improvement is not a reason to ignore a group where the simpler forecast works better.


## Would a recent trend help beyond the weekly lag?

Validation pickups are higher than training pickups, so I want to see whether an explicit trend feature helps the model adapt. I'll start with the difference between the two weekly lags. Both source hours precede issuance, though the availability of the underlying cleaned counts still matters.

$$
\text{weekly trend difference}
=
\texttt{pickup\_lag\_168}
-
\texttt{pickup\_lag\_336}
$$


In [ ]:
# Calculating the leakage-safe weekly trend difference.
training_weekly_trend_difference = (
    X_train["pickup_lag_168"]
    .to_numpy(dtype=np.float32, copy=False)
    - X_train["pickup_lag_336"]
    .to_numpy(dtype=np.float32, copy=False)
)

validation_weekly_trend_difference = (
    X_validation["pickup_lag_168"]
    .to_numpy(dtype=np.float32, copy=False)
    - X_validation["pickup_lag_336"]
    .to_numpy(dtype=np.float32, copy=False)
)

# Checking chronological availability at every horizon.
available_horizons = np.arange(1, 7)

lag_168_age_at_origin = (
    168 - available_horizons
)

lag_336_age_at_origin = (
    336 - available_horizons
)

assert lag_168_age_at_origin.min() == 162
assert lag_336_age_at_origin.min() == 330
assert np.isfinite(
    training_weekly_trend_difference
).all()
assert np.isfinite(
    validation_weekly_trend_difference
).all()

# Comparing the feature distribution across periods.
weekly_trend_summary = pd.DataFrame({
    "Period": [
        "Earlier training period",
        "Later validation period",
    ],
    "Mean": [
        training_weekly_trend_difference.mean(),
        validation_weekly_trend_difference.mean(),
    ],
    "Median": [
        np.median(training_weekly_trend_difference),
        np.median(validation_weekly_trend_difference),
    ],
    "Standard Deviation": [
        training_weekly_trend_difference.std(),
        validation_weekly_trend_difference.std(),
    ],
    "1st Percentile": [
        np.percentile(
            training_weekly_trend_difference,
            1,
        ),
        np.percentile(
            validation_weekly_trend_difference,
            1,
        ),
    ],
    "99th Percentile": [
        np.percentile(
            training_weekly_trend_difference,
            99,
        ),
        np.percentile(
            validation_weekly_trend_difference,
            99,
        ),
    ],
})

display(
    weekly_trend_summary.style
    .format({
        "Mean": "{:.3f}",
        "Median": "{:.3f}",
        "Standard Deviation": "{:.3f}",
        "1st Percentile": "{:.1f}",
        "99th Percentile": "{:.1f}",
    })
    .hide(axis="index")
)

In [ ]:
# Defining the single experimental feature.
weekly_trend_feature = (
    "weekly_trend_difference"
)

assert weekly_trend_feature not in X_train.columns
assert weekly_trend_feature not in X_validation.columns



try:
    # Adding the feature temporarily to the existing matrices.
    X_train[weekly_trend_feature] = (
        training_weekly_trend_difference
    )

    X_validation[weekly_trend_feature] = (
        validation_weekly_trend_difference
    )

    # Building paired matrices with training-derived cuts.
    (
        xgb_trend_training_matrix,
        xgb_trend_validation_matrix,
        trend_matrix_seconds,
    ) = build_quantile_matrix_pair(
        training_features=X_train,
        training_target=y_train,
        evaluation_features=X_validation,
        evaluation_target=y_validation,
        max_bin=xgb_max_bin,
        enable_categorical=True,
    )

finally:
    # Restoring the original 30-feature DataFrames.
    X_train.drop(
        columns=weekly_trend_feature,
        inplace=True,
        errors="ignore",
    )

    X_validation.drop(
        columns=weekly_trend_feature,
        inplace=True,
        errors="ignore",
    )


assert xgb_trend_training_matrix.num_col() == 31
assert xgb_trend_validation_matrix.num_col() == 31
assert xgb_trend_training_matrix.num_row() == len(y_train)
assert xgb_trend_validation_matrix.num_row() == len(
    y_validation
)
assert weekly_trend_feature not in X_train.columns
assert weekly_trend_feature not in X_validation.columns

print(
    "Trend training matrix:",
    f"{xgb_trend_training_matrix.num_row():,} rows ×",
    f"{xgb_trend_training_matrix.num_col()} columns",
)
print(
    "Trend validation matrix:",
    f"{xgb_trend_validation_matrix.num_row():,} rows ×",
    f"{xgb_trend_validation_matrix.num_col()} columns",
)
print(
    "Build time (seconds):",
    f"{trend_matrix_seconds:,.2f}",
)

The new matrices are ready, and the original feature frames have been restored. I'll fit the same XGBoost setup with one intended change: adding the weekly trend difference. That makes the resulting comparison easier to interpret.


In [ ]:
# Reusing the retained XGBoost configuration.
trend_xgb_params = initial_xgb_params.copy()

# Training with the single additional trend feature.
(
    trend_xgb_model,
    trend_xgb_history,
    trend_training_seconds,
) = train_xgboost_candidate(
    parameters=trend_xgb_params,
    training_matrix=xgb_trend_training_matrix,
    evaluation_matrices=[
        (
            xgb_trend_training_matrix,
            "training",
        ),
        (
            xgb_trend_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert trend_xgb_model.best_iteration >= 0
assert trend_xgb_model.best_score > 0

print()
print(
    "Best boosting round:",
    f"{trend_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{float(trend_xgb_model.best_score):,.4f}",
)
print(
    "Training time (minutes):",
    f"{trend_training_seconds / 60:,.2f}",
)

In [ ]:
trend_best_iteration_end = (
    trend_xgb_model.best_iteration + 1
)

# Predicting and applying the nonnegative demand constraint.
(
    trend_xgb_predictions,
    trend_xgb_predictions_constrained,
    trend_negative_prediction_count,
    trend_prediction_seconds,
) = predict_xgboost_demand(
    model=trend_xgb_model,
    prediction_matrix=(
        xgb_trend_validation_matrix
    ),
    iteration_end=(
        trend_best_iteration_end
    ),
)


assert len(trend_xgb_predictions_constrained) == len(
    y_validation
)
assert np.isfinite(
    trend_xgb_predictions_constrained
).all()
assert (
    trend_xgb_predictions_constrained >= 0
).all()

# Comparing the original and trend-feature models.
trend_feature_comparison = pd.DataFrame([
    {
        "model": "Original XGBoost",
        "features": 30,
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "XGBoost + weekly trend difference",
        "features": 31,
        **calculate_forecast_metrics(
            validation_actual,
            trend_xgb_predictions_constrained,
        ),
    },
])

display(
    trend_feature_comparison
    .rename(columns={
        "model": "Model",
        "features": "Features",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "Features": "{:.0f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

### Why I dropped the weekly trend difference

The difference feature is available under the implemented timing rule, but it does not help validation enough to keep. The model already sees both weekly counts, so expressing their difference explicitly may add little. I'll try a relative-change version before closing this question.

$$
\texttt{weekly\_trend\_difference}
=
\texttt{pickup\_lag\_168}
-
\texttt{pickup\_lag\_336}
$$


In [ ]:
# Releasing objects from the rejected trend experiment.
del xgb_trend_training_matrix
del xgb_trend_validation_matrix
del trend_xgb_model
del trend_xgb_history
del trend_xgb_predictions
del trend_xgb_predictions_constrained
del training_weekly_trend_difference
del validation_weekly_trend_difference

gc.collect()

# Checking memory after cleanup.
process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

### Testing a stable weekly-lag ratio

A 20-pickup change means something different in a quiet zone and a busy one. A log-ratio expresses change relative to level and can handle zeros with `log1p`. I'll test that version using the same comparison setup.

$$
\texttt{weekly\_log\_ratio}
=
\log\left(
\frac{1+\texttt{pickup\_lag\_168}}
     {1+\texttt{pickup\_lag\_336}}
\right)
$$

$$
\texttt{weekly\_log\_ratio}
=
\log(1+\texttt{pickup\_lag\_168})
-
\log(1+\texttt{pickup\_lag\_336})
$$


In [ ]:
# Extracting the two production-available weekly lags.
training_lag_168 = (
    X_train["pickup_lag_168"]
    .to_numpy(dtype=np.float32, copy=False)
)

training_lag_336 = (
    X_train["pickup_lag_336"]
    .to_numpy(dtype=np.float32, copy=False)
)

validation_lag_168 = (
    X_validation["pickup_lag_168"]
    .to_numpy(dtype=np.float32, copy=False)
)

validation_lag_336 = (
    X_validation["pickup_lag_336"]
    .to_numpy(dtype=np.float32, copy=False)
)

assert (training_lag_168 >= 0).all()
assert (training_lag_336 >= 0).all()
assert (validation_lag_168 >= 0).all()
assert (validation_lag_336 >= 0).all()

# Calculating the stable relative weekly change.
training_weekly_log_ratio = (
    np.log1p(training_lag_168)
    - np.log1p(training_lag_336)
).astype(np.float32)

validation_weekly_log_ratio = (
    np.log1p(validation_lag_168)
    - np.log1p(validation_lag_336)
).astype(np.float32)

assert np.isfinite(training_weekly_log_ratio).all()
assert np.isfinite(validation_weekly_log_ratio).all()

# Summarizing the transformation and zero handling.
weekly_log_ratio_summary = pd.DataFrame({
    "Period": [
        "Earlier training period",
        "Later validation period",
    ],
    "Mean": [
        training_weekly_log_ratio.mean(),
        validation_weekly_log_ratio.mean(),
    ],
    "Median": [
        np.median(training_weekly_log_ratio),
        np.median(validation_weekly_log_ratio),
    ],
    "Standard Deviation": [
        training_weekly_log_ratio.std(),
        validation_weekly_log_ratio.std(),
    ],
    "1st Percentile": [
        np.percentile(training_weekly_log_ratio, 1),
        np.percentile(validation_weekly_log_ratio, 1),
    ],
    "99th Percentile": [
        np.percentile(training_weekly_log_ratio, 99),
        np.percentile(validation_weekly_log_ratio, 99),
    ],
    "Both Lags Zero (%)": [
        (
            (training_lag_168 == 0)
            & (training_lag_336 == 0)
        ).mean() * 100,
        (
            (validation_lag_168 == 0)
            & (validation_lag_336 == 0)
        ).mean() * 100,
    ],
})

display(
    weekly_log_ratio_summary.style
    .format({
        "Mean": "{:.4f}",
        "Median": "{:.4f}",
        "Standard Deviation": "{:.4f}",
        "1st Percentile": "{:.4f}",
        "99th Percentile": "{:.4f}",
        "Both Lags Zero (%)": "{:.3f}",
    })
    .hide(axis="index")
)

The values stay finite, including the both-zero cases, and the distribution is centered near zero. The transformation is behaving as intended. The next fit will tell me whether that mathematical convenience translates into better predictions.


In [ ]:
# Defining the ratio feature name.
weekly_ratio_feature = "weekly_log_ratio"

assert weekly_ratio_feature not in X_train.columns
assert weekly_ratio_feature not in X_validation.columns



try:
    # Adding the ratio feature temporarily.
    X_train[weekly_ratio_feature] = (
        training_weekly_log_ratio
    )

    X_validation[weekly_ratio_feature] = (
        validation_weekly_log_ratio
    )

        # Building paired matrices with training-derived cuts.
    (
        xgb_ratio_training_matrix,
        xgb_ratio_validation_matrix,
        ratio_matrix_seconds,
    ) = build_quantile_matrix_pair(
        training_features=X_train,
        training_target=y_train,
        evaluation_features=X_validation,
        evaluation_target=y_validation,
        max_bin=xgb_max_bin,
        enable_categorical=True,
    )

finally:
    # Restoring the original 30-feature DataFrames.
    X_train.drop(
        columns=weekly_ratio_feature,
        inplace=True,
        errors="ignore",
    )

    X_validation.drop(
        columns=weekly_ratio_feature,
        inplace=True,
        errors="ignore",
    )


assert xgb_ratio_training_matrix.num_row() == len(y_train)
assert xgb_ratio_validation_matrix.num_row() == len(
    y_validation
)
assert xgb_ratio_training_matrix.num_col() == 31
assert xgb_ratio_validation_matrix.num_col() == 31
assert weekly_ratio_feature not in X_train.columns
assert weekly_ratio_feature not in X_validation.columns

print(
    "Ratio training matrix:",
    f"{xgb_ratio_training_matrix.num_row():,} rows ×",
    f"{xgb_ratio_training_matrix.num_col()} columns",
)
print(
    "Ratio validation matrix:",
    f"{xgb_ratio_validation_matrix.num_row():,} rows ×",
    f"{xgb_ratio_validation_matrix.num_col()} columns",
)
print(
    "Build time (seconds):",
    f"{ratio_matrix_seconds:,.2f}",
)

In [ ]:
# Reusing the retained XGBoost configuration.
ratio_xgb_params = initial_xgb_params.copy()

# Training with the single additional ratio feature.
(
    ratio_xgb_model,
    ratio_xgb_history,
    ratio_training_seconds,
) = train_xgboost_candidate(
    parameters=ratio_xgb_params,
    training_matrix=xgb_ratio_training_matrix,
    evaluation_matrices=[
        (
            xgb_ratio_training_matrix,
            "training",
        ),
        (
            xgb_ratio_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert ratio_xgb_model.best_iteration >= 0
assert ratio_xgb_model.best_score > 0

print()
print(
    "Best boosting round:",
    f"{ratio_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{float(ratio_xgb_model.best_score):,.4f}",
)
print(
    "Training time (minutes):",
    f"{ratio_training_seconds / 60:,.2f}",
)

In [ ]:
ratio_best_iteration_end = (
    ratio_xgb_model.best_iteration + 1
)

# Predicting and applying the nonnegative demand constraint.
(
    ratio_xgb_predictions,
    ratio_xgb_predictions_constrained,
    ratio_negative_prediction_count,
    ratio_prediction_seconds,
) = predict_xgboost_demand(
    model=ratio_xgb_model,
    prediction_matrix=(
        xgb_ratio_validation_matrix
    ),
    iteration_end=(
        ratio_best_iteration_end
    ),
)


assert len(ratio_xgb_predictions_constrained) == len(
    y_validation
)
assert np.isfinite(
    ratio_xgb_predictions_constrained
).all()
assert (
    ratio_xgb_predictions_constrained >= 0
).all()

# Comparing the original and ratio-feature models.
ratio_feature_comparison = pd.DataFrame([
    {
        "model": "Original XGBoost",
        "features": 30,
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "XGBoost + weekly log ratio",
        "features": 31,
        **calculate_forecast_metrics(
            validation_actual,
            ratio_xgb_predictions_constrained,
        ),
    },
])

display(
    ratio_feature_comparison
    .rename(columns={
        "model": "Model",
        "features": "Features",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "Features": "{:.0f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

### Why I dropped the weekly log ratio

The log-ratio does not earn a place in the model either. I'll leave both trend transformations out and retain the original inputs. This is a useful reminder that a feature can make sense on paper without helping a fitted tree model.

$$
\texttt{weekly\_log\_ratio}
=
\log(1+\texttt{pickup\_lag\_168})
-
\log(1+\texttt{pickup\_lag\_336})
$$


## Would I switch from XGBoost to LightGBM?

XGBoost has looked strongest among the direct candidates so far. I'll now compare LightGBM with the same forecasting inputs to see whether another boosting implementation changes that conclusion.


In [ ]:
# Releasing objects from the rejected ratio experiment.
del xgb_ratio_training_matrix
del xgb_ratio_validation_matrix
del ratio_xgb_model
del ratio_xgb_history
del ratio_xgb_predictions
del ratio_xgb_predictions_constrained
del training_weekly_log_ratio
del validation_weekly_log_ratio

gc.collect()

# Checking memory before the next model family.
process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Checking whether LightGBM is available.
import importlib.util

lightgbm_spec = importlib.util.find_spec(
    "lightgbm"
)

if lightgbm_spec is None:
    print("LightGBM installed: False")
else:
    import lightgbm as lgb

    print("LightGBM installed: True")
    print("LightGBM version:", lgb.__version__)

In [ ]:


gc.collect()

lightgbm_dataset_start = time.perf_counter()

# Building the full earlier-period training dataset.
lightgbm_training_data = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=catboost_categorical_columns,
    free_raw_data=False,
)

# Building the full later-period validation dataset.
lightgbm_validation_data = lgb.Dataset(
    X_validation,
    label=y_validation,
    categorical_feature=catboost_categorical_columns,
    reference=lightgbm_training_data,
    free_raw_data=False,
)

# Constructing shared histogram bins before training.
lightgbm_training_data.construct()
lightgbm_validation_data.construct()

lightgbm_dataset_seconds = (
    time.perf_counter()
    - lightgbm_dataset_start
)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

assert lightgbm_training_data.num_data() == len(y_train)
assert lightgbm_validation_data.num_data() == len(
    y_validation
)
assert lightgbm_training_data.num_feature() == 30
assert lightgbm_validation_data.num_feature() == 30

print(
    "Training rows:",
    f"{lightgbm_training_data.num_data():,}",
)
print(
    "Validation rows:",
    f"{lightgbm_validation_data.num_data():,}",
)
print(
    "Features:",
    lightgbm_training_data.num_feature(),
)
print(
    "Dataset build time (seconds):",
    f"{lightgbm_dataset_seconds:.2f}",
)
print(
    "Notebook process RAM (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Defining the controlled full LightGBM specification.
initial_lightgbm_params = {
    "objective": "regression",
    "metric": ["rmse", "l1"],
    "device_type": "cpu",
    "num_threads": 14,
    "force_col_wise": True,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "max_depth": 8,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.80,
    "bagging_freq": 1,
    "lambda_l1": 0.10,
    "lambda_l2": 5.00,
    "max_bin": 255,
    "seed": random_seed,
    "feature_fraction_seed": random_seed,
    "bagging_seed": random_seed,
    "data_random_seed": random_seed,
    "deterministic": True,
    "first_metric_only": True,
    "verbosity": -1,
}

initial_lightgbm_history = {}
lightgbm_training_start = time.perf_counter()

# Training on the earlier period only.
initial_lightgbm_model = lgb.train(
    params=initial_lightgbm_params,
    train_set=lightgbm_training_data,
    num_boost_round=2_000,
    valid_sets=[
        lightgbm_training_data,
        lightgbm_validation_data,
    ],
    valid_names=[
        "training",
        "validation",
    ],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=False,
        ),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(
            initial_lightgbm_history
        ),
    ],
)

lightgbm_training_seconds = (
    time.perf_counter()
    - lightgbm_training_start
)

assert initial_lightgbm_model.best_iteration > 0

print()
print(
    "Best boosting round:",
    f"{initial_lightgbm_model.best_iteration:,}",
)
print(
    "Best validation RMSE:",
    f"{initial_lightgbm_model.best_score['validation']['rmse']:.4f}",
)
print(
    "Training time (minutes):",
    f"{lightgbm_training_seconds / 60:.2f}",
)

In [ ]:
# Predicting at the retained LightGBM boosting round.
lightgbm_prediction_start = time.perf_counter()

initial_lightgbm_predictions = (
    initial_lightgbm_model.predict(
        X_validation,
        num_iteration=(
            initial_lightgbm_model.best_iteration
        ),
    )
)

lightgbm_prediction_seconds = (
    time.perf_counter()
    - lightgbm_prediction_start
)

# Applying the same nonnegative production constraint.
lightgbm_negative_predictions = int(
    (initial_lightgbm_predictions < 0).sum()
)

initial_lightgbm_predictions_constrained = np.clip(
    initial_lightgbm_predictions,
    a_min=0,
    a_max=None,
)

assert len(initial_lightgbm_predictions_constrained) == len(
    y_validation
)
assert np.isfinite(
    initial_lightgbm_predictions_constrained
).all()
assert (
    initial_lightgbm_predictions_constrained >= 0
).all()

# Comparing the two leading direct models.
leading_booster_comparison = pd.DataFrame([
    {
        "model": "XGBoost",
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "LightGBM",
        **calculate_forecast_metrics(
            validation_actual,
            initial_lightgbm_predictions_constrained,
        ),
    },
])

display(
    leading_booster_comparison
    .rename(columns={
        "model": "Model",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

print(
    "Negative LightGBM predictions:",
    f"{lightgbm_negative_predictions:,}",
)
print(
    "Prediction time (seconds):",
    f"{lightgbm_prediction_seconds:.2f}",
)

In [ ]:
# Extracting the full LightGBM learning history.
lightgbm_training_rmse = np.asarray(
    initial_lightgbm_history["training"]["rmse"],
    dtype=float,
)

lightgbm_validation_rmse = np.asarray(
    initial_lightgbm_history["validation"]["rmse"],
    dtype=float,
)

lightgbm_trained_rounds = len(
    lightgbm_validation_rmse
)

lightgbm_checkpoint_rounds = [
    500,
    1_000,
    1_500,
    1_900,
    2_000,
]

lightgbm_learning_summary = pd.DataFrame({
    "Round": lightgbm_checkpoint_rounds,
    "Training RMSE": [
        lightgbm_training_rmse[
            round_number - 1
        ]
        for round_number in lightgbm_checkpoint_rounds
    ],
    "Validation RMSE": [
        lightgbm_validation_rmse[
            round_number - 1
        ]
        for round_number in lightgbm_checkpoint_rounds
    ],
})

rmse_gain_last_100 = (
    lightgbm_validation_rmse[-101]
    - lightgbm_validation_rmse[-1]
)

rmse_gain_last_500 = (
    lightgbm_validation_rmse[-501]
    - lightgbm_validation_rmse[-1]
)

display(
    lightgbm_learning_summary.style
    .format({
        "Round": "{:,.0f}",
        "Training RMSE": "{:.4f}",
        "Validation RMSE": "{:.4f}",
    })
    .hide(axis="index")
)

print(
    "Validation RMSE gain over final 100 rounds:",
    f"{rmse_gain_last_100:.4f}",
)
print(
    "Validation RMSE gain over final 500 rounds:",
    f"{rmse_gain_last_500:.4f}",
)

### A small RMSE gain comes with trade-offs

LightGBM slightly improves RMSE, from 33.2236 to 33.0286, but the other metrics and bias do not all improve. I'll retain XGBoost for its overall trade-off and computational practicality. The choice is not a claim that it has the lowest error under every criterion.


In [ ]:
# Releasing LightGBM training objects while preserving predictions.
lightgbm_objects_to_release = [
    "initial_lightgbm_model",
    "lightgbm_training_data",
    "lightgbm_validation_data",
    "initial_lightgbm_history",
    "initial_lightgbm_predictions",
    "lightgbm_training_rmse",
    "lightgbm_validation_rmse",
    "lightgbm_learning_summary",
]

for object_name in lightgbm_objects_to_release:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

# Confirming that predictions remain available for later analysis.
assert len(
    initial_lightgbm_predictions_constrained
) == len(y_validation)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "LightGBM predictions preserved:",
    True,
)
print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

## Which XGBoost settings are worth keeping?

The initial XGBoost setup is already useful, so I'll make a few focused changes rather than start a large search. I'll compare depth, child-node restrictions, and regularization, keeping the data and scoring rules fixed.


In [ ]:
# Creating the depth-7 controlled specification.
depth7_xgb_params = initial_xgb_params.copy()

depth7_xgb_params.update({
    "max_depth": 7,
})

# Training the controlled depth-7 specification.
(
    depth7_xgb_model,
    depth7_xgb_history,
    depth7_training_seconds,
) = train_xgboost_candidate(
    parameters=depth7_xgb_params,
    training_matrix=xgb_training_matrix,
    evaluation_matrices=[
        (
            xgb_training_matrix,
            "training",
        ),
        (
            xgb_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert depth7_xgb_model.best_iteration >= 0
assert depth7_xgb_model.best_score > 0

print()
print(
    "Best boosting round:",
    f"{depth7_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{float(depth7_xgb_model.best_score):.4f}",
)
print(
    "Training time (minutes):",
    f"{depth7_training_seconds / 60:.2f}",
)

In [ ]:
depth7_best_iteration_end = (
    depth7_xgb_model.best_iteration + 1
)

# Predicting and applying the nonnegative demand constraint.
(
    depth7_xgb_predictions,
    depth7_xgb_predictions_constrained,
    depth7_negative_prediction_count,
    depth7_prediction_seconds,
) = predict_xgboost_demand(
    model=depth7_xgb_model,
    prediction_matrix=xgb_validation_matrix,
    iteration_end=(
        depth7_best_iteration_end
    ),
)


assert len(depth7_xgb_predictions_constrained) == len(
    y_validation
)
assert np.isfinite(
    depth7_xgb_predictions_constrained
).all()
assert (
    depth7_xgb_predictions_constrained >= 0
).all()

# Comparing depth 7 with the retained depth-8 model.
depth7_comparison = pd.DataFrame([
    {
        "model": "XGBoost depth 8",
        "training_minutes": initial_training_seconds / 60,
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "XGBoost depth 7",
        "training_minutes": depth7_training_seconds / 60,
        **calculate_forecast_metrics(
            validation_actual,
            depth7_xgb_predictions_constrained,
        ),
    },
])

display(
    depth7_comparison
    .rename(columns={
        "model": "Model",
        "training_minutes": "Training Minutes",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "Training Minutes": "{:.2f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

### Retaining depth 8 over depth 7

Depth 7 is faster and less biased, but slightly worse on the main accuracy measures. I'll keep depth 8. The differences are small enough that I would not expect this exact ranking to hold for every possible sample.


In [ ]:
# Releasing objects from the rejected depth-7 experiment.
del depth7_xgb_model
del depth7_xgb_history
del depth7_xgb_predictions
del depth7_xgb_predictions_constrained

gc.collect()

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Notebook process RAM after depth-7 cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Creating the stronger child-weight specification.
child50_xgb_params = initial_xgb_params.copy()

child50_xgb_params.update({
    "min_child_weight": 50,
})
# Training with stronger leaf-level regularization.
(
    child50_xgb_model,
    child50_xgb_history,
    child50_training_seconds,
) = train_xgboost_candidate(
    parameters=child50_xgb_params,
    training_matrix=xgb_training_matrix,
    evaluation_matrices=[
        (
            xgb_training_matrix,
            "training",
        ),
        (
            xgb_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert child50_xgb_model.best_iteration >= 0
assert child50_xgb_model.best_score > 0

print()
print(
    "Best boosting round:",
    f"{child50_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{float(child50_xgb_model.best_score):.4f}",
)
print(
    "Training time (minutes):",
    f"{child50_training_seconds / 60:.2f}",
)

In [ ]:
child50_best_iteration_end = (
    child50_xgb_model.best_iteration + 1
)

# Predicting and applying the nonnegative demand constraint.
(
    child50_xgb_predictions,
    child50_xgb_predictions_constrained,
    child50_negative_prediction_count,
    child50_prediction_seconds,
) = predict_xgboost_demand(
    model=child50_xgb_model,
    prediction_matrix=xgb_validation_matrix,
    iteration_end=(
        child50_best_iteration_end
    ),
)


assert len(child50_xgb_predictions_constrained) == len(
    y_validation
)
assert np.isfinite(
    child50_xgb_predictions_constrained
).all()
assert (
    child50_xgb_predictions_constrained >= 0
).all()

# Comparing both child-weight settings.
child50_comparison = pd.DataFrame([
    {
        "model": "XGBoost child weight 20",
        "training_minutes": initial_training_seconds / 60,
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "XGBoost child weight 50",
        "training_minutes": child50_training_seconds / 60,
        **calculate_forecast_metrics(
            validation_actual,
            child50_xgb_predictions_constrained,
        ),
    },
])

display(
    child50_comparison
    .rename(columns={
        "model": "Model",
        "training_minutes": "Training Minutes",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "Training Minutes": "{:.2f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

### Retaining `min_child_weight=20`

Stronger child-node restrictions do not help this fit: MAE, RMSE, and WAPE all move slightly in the wrong direction. I'll retain `min_child_weight=20` and try regularization next.


In [ ]:
# Releasing objects from the rejected child-weight experiment.
del child50_xgb_model
del child50_xgb_history
del child50_xgb_predictions
del child50_xgb_predictions_constrained

gc.collect()

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Notebook process RAM after child-weight cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Creating the stronger L2-regularization specification.
lambda10_xgb_params = initial_xgb_params.copy()

lambda10_xgb_params.update({
    "reg_lambda": 10.0,
})

# Training with stronger L2 leaf-value regularization.
(
    lambda10_xgb_model,
    lambda10_xgb_history,
    lambda10_training_seconds,
) = train_xgboost_candidate(
    parameters=lambda10_xgb_params,
    training_matrix=xgb_training_matrix,
    evaluation_matrices=[
        (
            xgb_training_matrix,
            "training",
        ),
        (
            xgb_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=50,
)

assert lambda10_xgb_model.best_iteration >= 0
assert lambda10_xgb_model.best_score > 0

print()
print(
    "Best boosting round:",
    f"{lambda10_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{float(lambda10_xgb_model.best_score):.4f}",
)
print(
    "Training time (minutes):",
    f"{lambda10_training_seconds / 60:.2f}",
)

In [ ]:
lambda10_best_iteration_end = (
    lambda10_xgb_model.best_iteration + 1
)

# Predicting and applying the nonnegative demand constraint.
(
    lambda10_xgb_predictions,
    lambda10_xgb_predictions_constrained,
    lambda10_negative_prediction_count,
    lambda10_prediction_seconds,
) = predict_xgboost_demand(
    model=lambda10_xgb_model,
    prediction_matrix=xgb_validation_matrix,
    iteration_end=(
        lambda10_best_iteration_end
    ),
)


assert len(lambda10_xgb_predictions_constrained) == len(
    y_validation
)
assert np.isfinite(
    lambda10_xgb_predictions_constrained
).all()
assert (
    lambda10_xgb_predictions_constrained >= 0
).all()

# Comparing both L2-regularization settings.
lambda10_comparison = pd.DataFrame([
    {
        "model": "XGBoost lambda 5",
        "best_round": initial_xgb_model.best_iteration + 1,
        "training_minutes": initial_training_seconds / 60,
        **calculate_forecast_metrics(
            validation_actual,
            initial_xgb_predictions_constrained,
        ),
    },
    {
        "model": "XGBoost lambda 10",
        "best_round": lambda10_xgb_model.best_iteration + 1,
        "training_minutes": lambda10_training_seconds / 60,
        **calculate_forecast_metrics(
            validation_actual,
            lambda10_xgb_predictions_constrained,
        ),
    },
])

display(
    lambda10_comparison
    .rename(columns={
        "model": "Model",
        "best_round": "Best Round",
        "training_minutes": "Training Minutes",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    .style
    .format({
        "Best Round": "{:,.0f}",
        "Training Minutes": "{:.2f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .hide(axis="index")
)

Lambda 10 gives a tiny RMSE gain, noticeably better bias, and fewer trees. That looks promising, but a gain this small could depend on which validation weeks are included. I'll check its behavior over time before keeping it.


In [ ]:
# Extracting validation months from forecast origins.
validation_origin_month = (
    pd.to_datetime(
        validation_evaluation[
            "forecast_origin_utc"
        ],
        utc=True,
    )
    .dt.strftime("%Y-%m")
    .to_numpy()
)

assert len(validation_origin_month) == len(
    validation_actual
)

validation_months = sorted(
    np.unique(validation_origin_month)
)

# Preparing aligned validation data by month.
lambda_month_frame = pd.DataFrame({
    "month": validation_origin_month,
    "total_pickups": validation_actual,
    "lambda_5_prediction": (
        initial_xgb_predictions_constrained
    ),
    "lambda_10_prediction": (
        lambda10_xgb_predictions_constrained
    ),
})

# Comparing both specifications across chronological months.
lambda_month_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=lambda_month_frame,
        group_columns="month",
        prediction_columns={
            "Lambda 5": "lambda_5_prediction",
            "Lambda 10": "lambda_10_prediction",
        },
    )
    .rename(columns={
        "Model": "model",
        "Rows": "rows",
    })
)

# Preserving chronological month order.
lambda_month_comparison["month"] = pd.Categorical(
    lambda_month_comparison["month"],
    categories=validation_months,
    ordered=True,
)

lambda_month_comparison = (
    lambda_month_comparison
    .sort_values(["month", "model"])
    .reset_index(drop=True)
)

# Restoring the original model order within each month.
lambda_model_order = {
    "Lambda 5": 0,
    "Lambda 10": 1,
}

lambda_month_comparison[
    "_model_order"
] = lambda_month_comparison[
    "model"
].map(lambda_model_order)

lambda_month_comparison = (
    lambda_month_comparison
    .sort_values([
        "month",
        "_model_order",
    ])
    .drop(columns="_model_order")
    .reset_index(drop=True)
)

# Checking complete monthly coverage.
assert lambda_month_comparison.shape[0] == (
    len(validation_months) * 2
)

assert (
    lambda_month_comparison
    .groupby(
        "model",
        observed=True,
    )["rows"]
    .sum()
    .eq(len(validation_actual))
    .all()
)

# Releasing the temporary grouped-metric frame.
del lambda_month_frame

display(
    lambda_month_comparison[
        [
            "month",
            "model",
            "rows",
            "mae",
            "rmse",
            "wape_percent",
            "mean_bias",
        ]
    ]
    .rename(columns={
        "month": "Month",
        "model": "Model",
        "rows": "Rows",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    .style
    .format({
        "Rows": "{:,.0f}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
    })
    .hide(axis="index")
)

In [ ]:
# Preparing chronological tuning differences.
lambda_stability_records = []

for month_value in validation_months:
    month_results = (
        lambda_month_comparison.loc[
            lambda_month_comparison["month"]
            == month_value
        ]
        .set_index("model")
    )

    lambda5_results = month_results.loc[
        "Lambda 5"
    ]

    lambda10_results = month_results.loc[
        "Lambda 10"
    ]

    lambda_stability_records.append({
        "Period": month_value,
        "Lambda 5 RMSE": lambda5_results["rmse"],
        "Lambda 10 RMSE": lambda10_results["rmse"],
        "RMSE Change": (
            lambda10_results["rmse"]
            - lambda5_results["rmse"]
        ),
        "MAE Change": (
            lambda10_results["mae"]
            - lambda5_results["mae"]
        ),
        "WAPE Change": (
            lambda10_results["wape_percent"]
            - lambda5_results["wape_percent"]
        ),
        "Lambda 5 Bias": (
            lambda5_results["mean_bias"]
        ),
        "Lambda 10 Bias": (
            lambda10_results["mean_bias"]
        ),
    })

# Reusing the complete validation metrics already calculated.
overall_lambda_results = (
    lambda10_comparison
    .set_index("model")
)

overall_lambda5_metrics = overall_lambda_results.loc[
    "XGBoost lambda 5"
]

overall_lambda10_metrics = overall_lambda_results.loc[
    "XGBoost lambda 10"
]

lambda_stability_records.append({
    "Period": "Overall",
    "Lambda 5 RMSE": overall_lambda5_metrics["rmse"],
    "Lambda 10 RMSE": overall_lambda10_metrics["rmse"],
    "RMSE Change": (
        overall_lambda10_metrics["rmse"]
        - overall_lambda5_metrics["rmse"]
    ),
    "MAE Change": (
        overall_lambda10_metrics["mae"]
        - overall_lambda5_metrics["mae"]
    ),
    "WAPE Change": (
        overall_lambda10_metrics["wape_percent"]
        - overall_lambda5_metrics["wape_percent"]
    ),
    "Lambda 5 Bias": (
        overall_lambda5_metrics["mean_bias"]
    ),
    "Lambda 10 Bias": (
        overall_lambda10_metrics["mean_bias"]
    ),
})

lambda_stability_summary = pd.DataFrame(
    lambda_stability_records
)

# Labeling the direction of the RMSE change.
lambda_stability_summary["Result"] = np.where(
    lambda_stability_summary["RMSE Change"] < 0,
    "Lambda 10 lower",
    "Lambda 5 lower",
)

display(
    lambda_stability_summary.style
    .format({
        "Lambda 5 RMSE": "{:.4f}",
        "Lambda 10 RMSE": "{:.4f}",
        "RMSE Change": "{:+.4f}",
        "MAE Change": "{:+.4f}",
        "WAPE Change": "{:+.4f}",
        "Lambda 5 Bias": "{:+.4f}",
        "Lambda 10 Bias": "{:+.4f}",
    })
    .hide(axis="index")
)

# Visualizing whether the RMSE change is stable.
lambda_change_palette = {
    "Lambda 10 lower": "#4C956C",
    "Lambda 5 lower": "#D97757",
}

fig, ax = plt.subplots(
    figsize=(10, 5.5),
)

sns.barplot(
    data=lambda_stability_summary,
    x="Period",
    y="RMSE Change",
    hue="Result",
    dodge=False,
    palette=lambda_change_palette,
    legend=False,
    ax=ax,
)

ax.axhline(
    0,
    color="#4B5563",
    linestyle="--",
    linewidth=1.15,
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%+.3f",
        padding=4,
        fontsize=10,
    )

ax.set_title(
    "Stronger L2 Regularization",
    loc="left",
)
ax.set_xlabel("Validation period")
ax.set_ylabel(
    "RMSE change: lambda 10 − lambda 5"
)

ax.grid(False)

fig.suptitle(
    "Is the Lambda-10 Improvement Chronologically Stable?",
    fontsize=15,
    fontweight="bold",
    x=0.06,
    ha="left",
    y=1.02,
)

fig.text(
    0.06,
    0.95,
    "Negative values favor lambda 10; positive values favor the retained lambda 5.",
    fontsize=10.5,
    color="#555555",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.92]
)
plt.show()

### Retaining `reg_lambda=10`

I'll retain lambda 10. The RMSE gain is small, but the lower bias and reduced tree count make it a useful trade-off. Validation selects 1,613 rounds, which I'll freeze before the final fit.


In [ ]:
# Identifying event-related features by name.
event_keywords = (
    "game",
    "concert",
    "event",
)

event_feature_candidates = [
    column
    for column in model_input_columns
    if any(
        keyword in column.lower()
        for keyword in event_keywords
    )
]

assert len(event_feature_candidates) > 0

# Auditing event-feature availability and sparsity.
event_feature_audit = pd.DataFrame({
    "Feature": event_feature_candidates,
    "Data Type": [
        str(X_train[column].dtype)
        for column in event_feature_candidates
    ],
    "Training Unique Values": [
        X_train[column].nunique(
            dropna=False
        )
        for column in event_feature_candidates
    ],
    "Validation Unique Values": [
        X_validation[column].nunique(
            dropna=False
        )
        for column in event_feature_candidates
    ],
    "Training Nonzero (%)": [
        (
            X_train[column]
            .fillna(0)
            .ne(0)
            .mean()
            * 100
        )
        for column in event_feature_candidates
    ],
    "Validation Nonzero (%)": [
        (
            X_validation[column]
            .fillna(0)
            .ne(0)
            .mean()
            * 100
        )
        for column in event_feature_candidates
    ],
})

display(
    event_feature_audit.style
    .format({
        "Training Unique Values": "{:,.0f}",
        "Validation Unique Values": "{:,.0f}",
        "Training Nonzero (%)": "{:.3f}",
        "Validation Nonzero (%)": "{:.3f}",
    })
    .hide(axis="index")
)

In [ ]:
# Checking overlap between the narrow and wide concert windows.
concert_overlap_results = []

for period_name, feature_frame in [
    ("Earlier training period", X_train),
    ("Later validation period", X_validation),
]:
    overlap_table = (
        feature_frame
        .groupby(
            ["is_concert_window", "is_wide_concert_window"],
            observed=True,
        )
        .size()
        .rename("rows")
        .reset_index()
    )

    overlap_table["share_percent"] = (
        100 * overlap_table["rows"] / len(feature_frame)
    )

    overlap_table.insert(0, "period", period_name)
    concert_overlap_results.append(overlap_table)

concert_window_overlap = pd.concat(
    concert_overlap_results,
    ignore_index=True,
)

# Validating that every narrow-window row also belongs to the wide window.
invalid_narrow_rows = (
    X_train["is_concert_window"]
    & ~X_train["is_wide_concert_window"]
).sum() + (
    X_validation["is_concert_window"]
    & ~X_validation["is_wide_concert_window"]
).sum()

assert invalid_narrow_rows == 0, (
    "The narrow concert window is not fully contained "
    "within the wide concert window."
)

display(
    concert_window_overlap.style
    .format({
        "rows": "{:,.0f}",
        "share_percent": "{:.4f}",
    })
    .hide(axis="index")
)

### Do I need both concert flags?

Imagine the last train out of an event venue or a crowd leaving later than usual. The flags give the model a rough time window for extra pickups. I compare the same model with different flags so I can see whether the extra hour helps.

The concert flags overlap by design. One covers 10 PM–midnight, and the other extends through the following midnight hour. I want to see whether both are useful, rather than assume the wider flag makes the narrow one redundant.


In [ ]:
# Defining the two overlapping concert features.
concert_feature_names = [
    "is_concert_window",
    "is_wide_concert_window",
]

# Checking that training and validation use the same feature structure.
assert X_train.columns.equals(X_validation.columns)
assert set(concert_feature_names).issubset(X_train.columns)

# Defining controlled concert-ablation feature sets.
concert_ablation_features = {
    "No concert features": [
        feature for feature in X_train.columns
        if feature not in concert_feature_names
    ],
    "Narrow window only": [
        feature for feature in X_train.columns
        if feature != "is_wide_concert_window"
    ],
    "Wide window only": [
        feature for feature in X_train.columns
        if feature != "is_concert_window"
    ],
    "Both concert windows": X_train.columns.tolist(),
}

# Validating that the game feature remains in every experiment.
assert all(
    "is_post_game_demand" in feature_list
    for feature_list in concert_ablation_features.values()
)

concert_ablation_design = pd.DataFrame({
    "experiment": concert_ablation_features.keys(),
    "feature_count": [
        len(feature_list)
        for feature_list in concert_ablation_features.values()
    ],
    "concert_features_retained": [
        ", ".join(
            feature for feature in concert_feature_names
            if feature in feature_list
        ) or "None"
        for feature_list in concert_ablation_features.values()
    ],
})

display(
    concert_ablation_design.style
    .hide(axis="index")
)

In [ ]:
# Selecting the strict no-concert control features.
no_concert_feature_names = concert_ablation_features[
    "No concert features"
]
# Building paired no-concert matrices with training-derived cuts.
(
    no_concert_training_matrix,
    no_concert_validation_matrix,
    no_concert_matrix_seconds,
) = build_quantile_matrix_pair(
    training_features=(
        X_train[
            no_concert_feature_names
        ]
    ),
    training_target=y_train,
    evaluation_features=(
        X_validation[
            no_concert_feature_names
        ]
    ),
    evaluation_target=y_validation,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)


# Validating the resulting matrix structure.
assert no_concert_training_matrix.num_row() == len(X_train)
assert no_concert_validation_matrix.num_row() == len(X_validation)
assert no_concert_training_matrix.num_col() == 28
assert no_concert_validation_matrix.num_col() == 28

no_concert_matrix_summary = pd.DataFrame({
    "Measure": [
        "Training rows",
        "Validation rows",
        "Features",
        "Build time (seconds)",
    ],
    "Value": [
        f"{no_concert_training_matrix.num_row():,}",
        f"{no_concert_validation_matrix.num_row():,}",
        no_concert_training_matrix.num_col(),
        f"{no_concert_matrix_seconds:.2f}",
    ],
})

display(
    no_concert_matrix_summary.style
    .hide(axis="index")
)

In [ ]:
# Recreating the retained XGBoost configuration for the ablation.
concert_ablation_xgb_params = initial_xgb_params.copy()
concert_ablation_xgb_params.update({
    "reg_lambda": 10.0,
})

assert concert_ablation_xgb_params["reg_lambda"] == 10.0
assert concert_ablation_xgb_params["objective"] == "reg:squarederror"
assert concert_ablation_xgb_params["device"] == "cuda"

# Training the no-concert control.
(
    no_concert_xgb_model,
    no_concert_xgb_history,
    no_concert_training_seconds,
) = train_xgboost_candidate(
    parameters=concert_ablation_xgb_params,
    training_matrix=no_concert_training_matrix,
    evaluation_matrices=[
        (
            no_concert_training_matrix,
            "training",
        ),
        (
            no_concert_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=100,
)

# Converting the reusable timer to the displayed unit.
no_concert_training_minutes = (
    no_concert_training_seconds
    / 60
)

print()
print(
    "Best boosting round:",
    f"{no_concert_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{no_concert_xgb_model.best_score:.4f}",
)
print(
    "Training time (minutes):",
    f"{no_concert_training_minutes:.2f}",
)

In [ ]:
# Generating constrained predictions for the later period.
(
    no_concert_xgb_predictions,
    no_concert_xgb_predictions_constrained,
    no_concert_negative_predictions,
    no_concert_prediction_seconds,
) = predict_xgboost_demand(
    model=no_concert_xgb_model,
    prediction_matrix=(
        no_concert_validation_matrix
    ),
    iteration_end=(
        no_concert_xgb_model.best_iteration
        + 1
    ),
)

assert len(no_concert_xgb_predictions) == len(y_validation)
assert np.isfinite(no_concert_xgb_predictions).all()



no_concert_negative_predictions = int(
    np.count_nonzero(no_concert_xgb_predictions < 0)
)

# Calculating complete validation metrics.
no_concert_xgb_metrics = calculate_forecast_metrics(
    actual=y_validation,
    prediction=no_concert_xgb_predictions_constrained,
)

no_concert_metric_table = pd.DataFrame([{
    "Model": "XGBoost without concert features",
    "MAE": no_concert_xgb_metrics["mae"],
    "RMSE": no_concert_xgb_metrics["rmse"],
    "WAPE (%)": no_concert_xgb_metrics["wape_percent"],
    "Mean Bias": no_concert_xgb_metrics["mean_bias"],
    "R²": no_concert_xgb_metrics["r_squared"],
    "Negative Predictions": no_concert_negative_predictions,
    "Prediction Seconds": no_concert_prediction_seconds,
}])

display(
    no_concert_metric_table.style
    .format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
        "Negative Predictions": "{:,.0f}",
        "Prediction Seconds": "{:.2f}",
    })
    .hide(axis="index")
)

In [ ]:
# Defining validation segments from event information only.
concert_validation_masks = {
    "All validation rows": np.ones(
        len(X_validation),
        dtype=bool,
    ),
    "Any concert-window row": (
        X_validation["is_wide_concert_window"]
        .to_numpy()
    ),
    "Core 10 PM–midnight window": (
        X_validation["is_concert_window"]
        .to_numpy()
    ),
    "Midnight extension only": (
        X_validation["is_wide_concert_window"]
        & ~X_validation["is_concert_window"]
    ).to_numpy(),
}

# Validating the retained prediction arrays.
assert len(lambda10_xgb_predictions_constrained) == len(
    y_validation
)
assert len(no_concert_xgb_predictions_constrained) == len(
    y_validation
)

# Preparing the shared validation values.
concert_validation_base = pd.DataFrame({
    "total_pickups": np.asarray(y_validation),
    "both_windows_prediction": (
        lambda10_xgb_predictions_constrained
    ),
    "no_concert_prediction": (
        no_concert_xgb_predictions_constrained
    ),
})

# Expanding overlapping rows into their applicable segments.
concert_segment_frames = []

for segment_name, segment_mask in (
    concert_validation_masks.items()
):
    assert segment_mask.sum() > 0

    segment_frame = concert_validation_base.loc[
        segment_mask
    ].copy()

    segment_frame.insert(
        0,
        "Segment",
        segment_name,
    )

    concert_segment_frames.append(segment_frame)

concert_validation_frame = pd.concat(
    concert_segment_frames,
    ignore_index=True,
)

# Calculating both models within every validation segment.
concert_validation_comparison = (
    calculate_metrics_by_group(
        evaluation_frame=concert_validation_frame,
        group_columns="Segment",
        prediction_columns={
            "Both concert windows": (
                "both_windows_prediction"
            ),
            "No concert features": (
                "no_concert_prediction"
            ),
        },
    )
)

# Calculating mean demand once per segment.
concert_segment_mean_demand = (
    concert_validation_frame
    .groupby(
        "Segment",
        sort=False,
    )["total_pickups"]
    .mean()
)

concert_validation_comparison[
    "Mean Actual Demand"
] = concert_validation_comparison[
    "Segment"
].map(concert_segment_mean_demand)

# Preserving the original table columns and labels.
concert_validation_comparison = (
    concert_validation_comparison
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    [
        [
            "Segment",
            "Model",
            "Rows",
            "Mean Actual Demand",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

# Checking complete segment and model coverage.
assert concert_validation_comparison.shape[0] == 8

assert (
    concert_validation_comparison[
        [
            "Mean Actual Demand",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
    .notna()
    .all()
    .all()
)

# Releasing temporary grouped-metric objects.
del concert_validation_base
del concert_segment_frames
del concert_validation_frame
del concert_segment_mean_demand
del segment_frame

display(
    concert_validation_comparison.style
    .format({
        "Rows": "{:,.0f}",
        "Mean Actual Demand": "{:.2f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .hide(axis="index")
)

### What changes when I remove both concert flags?

Without concert flags, the model misses event-period pickups on the low side by about 32 per row. Adding both flags removes much of that bias, but the error improvements are uneven. I'll separate the core window from its midnight extension before deciding what to keep.


In [ ]:
# Releasing heavy objects from the completed no-concert experiment.
del no_concert_xgb_model
del no_concert_xgb_history
del no_concert_training_matrix
del no_concert_validation_matrix
del no_concert_xgb_predictions

gc.collect()

# Checking memory while preserving the constrained predictions.
assert len(no_concert_xgb_predictions_constrained) == len(y_validation)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "No-concert predictions preserved:",
    len(no_concert_xgb_predictions_constrained)
    == len(y_validation),
)
print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Selecting the wide-window-only feature set.
wide_only_feature_names = concert_ablation_features[
    "Wide window only"
]

assert "is_wide_concert_window" in wide_only_feature_names
assert "is_concert_window" not in wide_only_feature_names
assert len(wide_only_feature_names) == 29
# Building paired wide-window matrices with training-derived cuts.
(
    wide_only_training_matrix,
    wide_only_validation_matrix,
    wide_only_matrix_seconds,
) = build_quantile_matrix_pair(
    training_features=(
        X_train[
            wide_only_feature_names
        ]
    ),
    training_target=y_train,
    evaluation_features=(
        X_validation[
            wide_only_feature_names
        ]
    ),
    evaluation_target=y_validation,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)

# Validating the resulting matrix structure.
assert wide_only_training_matrix.num_row() == len(X_train)
assert wide_only_validation_matrix.num_row() == len(X_validation)
assert wide_only_training_matrix.num_col() == 29
assert wide_only_validation_matrix.num_col() == 29

wide_only_matrix_summary = pd.DataFrame({
    "Measure": [
        "Training rows",
        "Validation rows",
        "Features",
        "Build time (seconds)",
    ],
    "Value": [
        f"{wide_only_training_matrix.num_row():,}",
        f"{wide_only_validation_matrix.num_row():,}",
        wide_only_training_matrix.num_col(),
        f"{wide_only_matrix_seconds:.2f}",
    ],
})

display(
    wide_only_matrix_summary.style
    .hide(axis="index")
)

In [ ]:
# Training the wide-window-only concert model.
(
    wide_only_xgb_model,
    wide_only_xgb_history,
    wide_only_training_seconds,
) = train_xgboost_candidate(
    parameters=concert_ablation_xgb_params,
    training_matrix=wide_only_training_matrix,
    evaluation_matrices=[
        (
            wide_only_training_matrix,
            "training",
        ),
        (
            wide_only_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=100,
)

# Converting the reusable timer to minutes.
wide_only_training_minutes = (
    wide_only_training_seconds
    / 60
)

print()
print(
    "Best boosting round:",
    f"{wide_only_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{wide_only_xgb_model.best_score:.4f}",
)
print(
    "Training time (minutes):",
    f"{wide_only_training_minutes:.2f}",
)

In [ ]:
# Generating constrained wide-window-only predictions.
(
    wide_only_xgb_predictions,
    wide_only_xgb_predictions_constrained,
    wide_only_negative_prediction_count,
    wide_only_prediction_seconds,
) = predict_xgboost_demand(
    model=wide_only_xgb_model,
    prediction_matrix=(
        wide_only_validation_matrix
    ),
    iteration_end=(
        wide_only_xgb_model.best_iteration
        + 1
    ),
)

assert len(wide_only_xgb_predictions) == len(
    y_validation
)
assert np.isfinite(wide_only_xgb_predictions).all()

# Preparing the new model's overlapping validation segments.
wide_only_validation_base = pd.DataFrame({
    "total_pickups": np.asarray(y_validation),
    "wide_only_prediction": (
        wide_only_xgb_predictions_constrained
    ),
})

wide_only_segment_frames = []

for segment_name, segment_mask in (
    concert_validation_masks.items()
):
    assert segment_mask.sum() > 0

    segment_frame = wide_only_validation_base.loc[
        segment_mask
    ].copy()

    segment_frame.insert(
        0,
        "Segment",
        segment_name,
    )

    wide_only_segment_frames.append(segment_frame)

wide_only_segment_frame = pd.concat(
    wide_only_segment_frames,
    ignore_index=True,
)

# Calculating only the newly trained model.
wide_only_segment_metrics = (
    calculate_metrics_by_group(
        evaluation_frame=wide_only_segment_frame,
        group_columns="Segment",
        prediction_columns={
            "Wide window only": (
                "wide_only_prediction"
            ),
        },
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    [
        [
            "Segment",
            "Model",
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

# Reusing the two model results calculated previously.
previous_concert_results = (
    concert_validation_comparison[
        [
            "Segment",
            "Model",
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
    .copy()
)

wide_only_validation_comparison = pd.concat(
    [
        wide_only_segment_metrics,
        previous_concert_results,
    ],
    ignore_index=True,
)

# Restoring the original segment and model order.
concert_segment_order = {
    segment_name: position
    for position, segment_name in enumerate(
        concert_validation_masks
    )
}

concert_model_order = {
    "Wide window only": 0,
    "Both concert windows": 1,
    "No concert features": 2,
}

wide_only_validation_comparison[
    "_segment_order"
] = wide_only_validation_comparison[
    "Segment"
].map(concert_segment_order)

wide_only_validation_comparison[
    "_model_order"
] = wide_only_validation_comparison[
    "Model"
].map(concert_model_order)

wide_only_validation_comparison = (
    wide_only_validation_comparison
    .sort_values([
        "_segment_order",
        "_model_order",
    ])
    .drop(columns=[
        "_segment_order",
        "_model_order",
    ])
    .reset_index(drop=True)
)

# Checking complete segment and model coverage.
assert wide_only_validation_comparison.shape[0] == 12

assert (
    wide_only_validation_comparison
    .groupby("Segment")["Model"]
    .nunique()
    .eq(3)
    .all()
)

# Releasing temporary grouped-metric objects.
del wide_only_validation_base
del wide_only_segment_frames
del wide_only_segment_frame
del wide_only_segment_metrics
del previous_concert_results
del segment_frame

display(
    wide_only_validation_comparison.style
    .format({
        "Rows": "{:,.0f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .hide(axis="index")
)

### Interpreting the Wide-Window-Only Experiment

Wide-only has the lowest pooled RMSE here, but its advantage is only about 0.008. It does not give the same advantage on the event segments. That is too little evidence to choose it from the headline score alone.


In [ ]:
# Releasing heavy objects from the rejected wide-only experiment.
del wide_only_xgb_model
del wide_only_xgb_history
del wide_only_training_matrix
del wide_only_validation_matrix
del wide_only_xgb_predictions

gc.collect()

# Checking that the constrained predictions remain available.
assert len(wide_only_xgb_predictions_constrained) == len(y_validation)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Wide-only predictions preserved:",
    len(wide_only_xgb_predictions_constrained)
    == len(y_validation),
)
print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

There is one remaining alternative: keep the core 10 PM–midnight flag and remove the extension. I'll build those matrices so all four concert representations can be compared.


In [ ]:
# Selecting the narrow-window-only feature set.
narrow_only_feature_names = concert_ablation_features[
    "Narrow window only"
]

assert "is_concert_window" in narrow_only_feature_names
assert "is_wide_concert_window" not in narrow_only_feature_names
assert len(narrow_only_feature_names) == 29

# Building paired narrow-window matrices with training-derived cuts.
(
    narrow_only_training_matrix,
    narrow_only_validation_matrix,
    narrow_only_matrix_seconds,
) = build_quantile_matrix_pair(
    training_features=(
        X_train[
            narrow_only_feature_names
        ]
    ),
    training_target=y_train,
    evaluation_features=(
        X_validation[
            narrow_only_feature_names
        ]
    ),
    evaluation_target=y_validation,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)


# Validating the resulting matrix structure.
assert narrow_only_training_matrix.num_row() == len(X_train)
assert narrow_only_validation_matrix.num_row() == len(X_validation)
assert narrow_only_training_matrix.num_col() == 29
assert narrow_only_validation_matrix.num_col() == 29

narrow_only_matrix_summary = pd.DataFrame({
    "Measure": [
        "Training rows",
        "Validation rows",
        "Features",
        "Build time (seconds)",
    ],
    "Value": [
        f"{narrow_only_training_matrix.num_row():,}",
        f"{narrow_only_validation_matrix.num_row():,}",
        narrow_only_training_matrix.num_col(),
        f"{narrow_only_matrix_seconds:.2f}",
    ],
})

display(
    narrow_only_matrix_summary.style
    .hide(axis="index")
)

I'll fit the narrow-only version with the same retained settings. After that, I can compare the complete picture rather than decide from one promising segment.


In [ ]:
# Training the narrow-window-only concert model.
(
    narrow_only_xgb_model,
    narrow_only_xgb_history,
    narrow_only_training_seconds,
) = train_xgboost_candidate(
    parameters=concert_ablation_xgb_params,
    training_matrix=narrow_only_training_matrix,
    evaluation_matrices=[
        (
            narrow_only_training_matrix,
            "training",
        ),
        (
            narrow_only_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=100,
)

# Converting the reusable timer to minutes.
narrow_only_training_minutes = (
    narrow_only_training_seconds
    / 60
)

print()
print(
    "Best boosting round:",
    f"{narrow_only_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{narrow_only_xgb_model.best_score:.4f}",
)
print(
    "Training time (minutes):",
    f"{narrow_only_training_minutes:.2f}",
)

In [ ]:
# Generating constrained narrow-window-only predictions.
(
    narrow_only_xgb_predictions,
    narrow_only_xgb_predictions_constrained,
    narrow_only_negative_prediction_count,
    narrow_only_prediction_seconds,
) = predict_xgboost_demand(
    model=narrow_only_xgb_model,
    prediction_matrix=(
        narrow_only_validation_matrix
    ),
    iteration_end=(
        narrow_only_xgb_model.best_iteration
        + 1
    ),
)

assert len(narrow_only_xgb_predictions) == len(
    y_validation
)
assert np.isfinite(narrow_only_xgb_predictions).all()

# Preparing the new model's overlapping validation segments.
narrow_only_validation_base = pd.DataFrame({
    "total_pickups": np.asarray(y_validation),
    "narrow_only_prediction": (
        narrow_only_xgb_predictions_constrained
    ),
})

narrow_only_segment_frames = []

for segment_name, segment_mask in (
    concert_validation_masks.items()
):
    assert segment_mask.sum() > 0

    segment_frame = narrow_only_validation_base.loc[
        segment_mask
    ].copy()

    segment_frame.insert(
        0,
        "Segment",
        segment_name,
    )

    narrow_only_segment_frames.append(segment_frame)

narrow_only_segment_frame = pd.concat(
    narrow_only_segment_frames,
    ignore_index=True,
)

# Calculating only the newly trained narrow-window model.
narrow_only_segment_metrics = (
    calculate_metrics_by_group(
        evaluation_frame=narrow_only_segment_frame,
        group_columns="Segment",
        prediction_columns={
            "Narrow window only": (
                "narrow_only_prediction"
            ),
        },
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    [
        [
            "Segment",
            "Model",
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

# Reusing the three previously calculated model results.
previous_concert_ablation_results = (
    wide_only_validation_comparison.copy()
)

concert_ablation_results = pd.concat(
    [
        previous_concert_ablation_results,
        narrow_only_segment_metrics,
    ],
    ignore_index=True,
)

# Restoring the original segment and model order.
concert_segment_order = {
    segment_name: position
    for position, segment_name in enumerate(
        concert_validation_masks
    )
}

concert_ablation_model_order = {
    "Both concert windows": 0,
    "Narrow window only": 1,
    "Wide window only": 2,
    "No concert features": 3,
}

concert_ablation_results[
    "_segment_order"
] = concert_ablation_results[
    "Segment"
].map(concert_segment_order)

concert_ablation_results[
    "_model_order"
] = concert_ablation_results[
    "Model"
].map(concert_ablation_model_order)

concert_ablation_results = (
    concert_ablation_results
    .sort_values([
        "_segment_order",
        "_model_order",
    ])
    .drop(columns=[
        "_segment_order",
        "_model_order",
    ])
    .reset_index(drop=True)
)

# Checking complete segment and model coverage.
assert concert_ablation_results.shape[0] == 16

assert (
    concert_ablation_results
    .groupby("Segment")["Model"]
    .nunique()
    .eq(4)
    .all()
)

# Releasing temporary grouped-metric objects.
del narrow_only_validation_base
del narrow_only_segment_frames
del narrow_only_segment_frame
del narrow_only_segment_metrics
del previous_concert_ablation_results
del segment_frame

display(
    concert_ablation_results.style
    .format({
        "Rows": "{:,.0f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .hide(axis="index")
)

In [ ]:
from IPython.display import Markdown as IPythonMarkdown

# Defining concise explanations for each validation segment.
concert_segment_descriptions = {
    "All validation rows": (
        "Overall model-selection performance across the complete "
        "later validation period."
    ),
    "Any concert-window row": (
        "Performance across all 1,098 rows covered by either "
        "concert window."
    ),
    "Core 10 PM–midnight window": (
        "Performance during the two-hour core concert-demand window."
    ),
    "Midnight extension only": (
        "Performance during the additional midnight hour included "
        "only by the wide window."
    ),
}

model_display_order = [
    "Both concert windows",
    "Narrow window only",
    "Wide window only",
    "No concert features",
]

# Highlighting the mean bias closest to zero.
def highlight_closest_bias(values):
    styles = [""] * len(values)
    best_position = np.abs(values.to_numpy()).argmin()
    styles[best_position] = (
        "background-color: #6DD3E1; "
        "font-weight: bold;"
    )
    return styles


# Displaying one focused table for each validation question.
for segment_name, segment_description in (
    concert_segment_descriptions.items()
):
    segment_table = (
        concert_ablation_results.loc[
            concert_ablation_results["Segment"]
            == segment_name,
            [
                "Model",
                "Rows",
                "MAE",
                "RMSE",
                "WAPE (%)",
                "Mean Bias",
            ],
        ]
        .copy()
    )

    segment_table["Model"] = pd.Categorical(
        segment_table["Model"],
        categories=model_display_order,
        ordered=True,
    )

    segment_table = (
        segment_table
        .sort_values("Model")
        .reset_index(drop=True)
    )

    display(
        IPythonMarkdown(
            f"### {segment_name}\n\n"
            f"{segment_description}"
        )
    )

    display(
        segment_table.style
        .format({
            "Rows": "{:,.0f}",
            "MAE": "{:.3f}",
            "RMSE": "{:.3f}",
            "WAPE (%)": "{:.3f}",
            "Mean Bias": "{:+.3f}",
        })
        .highlight_min(
            subset=["MAE", "RMSE", "WAPE (%)"],
            color="#6DD3E1",
        )
        .apply(
            highlight_closest_bias,
            subset=["Mean Bias"],
            
        )
        .set_properties(
            subset=["Model"],
            **{"text-align": "left"},
        )
        .hide(axis="index")
    )

    # Preparing shorter labels for the chart.
concert_chart_data = concert_ablation_results.copy()

concert_chart_data["Segment"] = concert_chart_data[
    "Segment"
].replace({
    "All validation rows": "All validation",
    "Any concert-window row": "Any concert window",
    "Core 10 PM–midnight window": "Core window",
    "Midnight extension only": "Midnight extension",
})

concert_segment_order = [
    "All validation",
    "Any concert window",
    "Core window",
    "Midnight extension",
]

concert_model_order = [
    "Both concert windows",
    "Narrow window only",
    "Wide window only",
    "No concert features",
]

# Emphasizing the leading configuration while muting alternatives.
concert_model_palette = {
    "Both concert windows": "#2563EB",
    "Narrow window only": "#9BBF96",
    "Wide window only": "#E3A07E",
    "No concert features": "#9CA3AF",
}

# Creating vertically stacked comparison charts.
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 13),
    dpi=460,
)

sns.barplot(
    data=concert_chart_data,
    x="Segment",
    y="RMSE",
    hue="Model",
    order=concert_segment_order,
    hue_order=concert_model_order,
    palette=concert_model_palette,
    errorbar=None,
    ax=axes[0],
)

axes[0].set_title(
    "Large-Error Performance by Concert Segment",
    loc="left",
)
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE")
# Moving the shared legend above the plotting area.
legend_handles, legend_labels = (
    axes[0].get_legend_handles_labels()
)

axes[0].legend_.remove()

fig.legend(
    legend_handles,
    legend_labels,
    title="Feature configuration",
    loc="upper center",
    bbox_to_anchor=(0.5, 0.905),
    ncols=4,
    frameon=False,
    columnspacing=1.8,
    handletextpad=0.6,
)


sns.barplot(
    data=concert_chart_data,
    x="Segment",
    y="Mean Bias",
    hue="Model",
    order=concert_segment_order,
    hue_order=concert_model_order,
    palette=concert_model_palette,
    errorbar=None,
    ax=axes[1],
)

axes[1].axhline(
    0,
    color="#4B5563",
    linewidth=1.0,
    linestyle="--",
)
axes[1].set_title(
    "Concert Features Substantially Change Forecast Bias",
    loc="left",
)
axes[1].set_xlabel("Validation demand segment")
axes[1].set_ylabel("Mean bias")
axes[1].legend_.remove()



# Keeping only horizontal grid lines.
for axis in axes:
    axis.grid(axis="y", visible=True)
    axis.grid(axis="x", visible=False)
    axis.margins(y=0.12)
    sns.despine(
        ax=axis,
        top=True,
        right=True,
        left=True,
    )

# Outlining the lowest-RMSE model within each segment.
rmse_winners = [
    (2, 0),  # Wide only: all validation
    (0, 1),  # Both windows: any concert window
    (3, 2),  # No concerts: core window
    (0, 3),  # Both windows: midnight extension
]

# Adding RMSE labels and emphasizing segment winners.
for model_position, container in enumerate(
    axes[0].containers
):
    rmse_labels = axes[0].bar_label(
        container,
        fmt="%.1f",
        padding=4,
        fontsize=8,
    )

    for segment_position, label in enumerate(
        rmse_labels
    ):
        if (
            model_position,
            segment_position,
        ) in rmse_winners:
            label.set_fontweight("bold")
            label.set_fontsize(10)
            label.set_color("#111827")
            label.set_bbox({
                "boxstyle": "round,pad=0.20",
                "facecolor": "#EFF6FF",
                "edgecolor": "#2563EB",
                "linewidth": 0.8,
            })


# Outlining the model with bias closest to zero.
bias_winners = [
    (0, 0),  # Both windows: all validation
    (1, 1),  # Narrow only: any concert window
    (0, 2),  # Both windows: core window
    (0, 3),  # Both windows: midnight extension
]

# Adding bias labels and emphasizing values closest to zero.
for model_position, container in enumerate(
    axes[1].containers
):
    bias_labels = axes[1].bar_label(
        container,
        fmt="%+.1f",
        padding=4,
        fontsize=8,
    )

    for segment_position, label in enumerate(
        bias_labels
    ):
        if (
            model_position,
            segment_position,
        ) in bias_winners:
            label.set_fontweight("bold")
            label.set_fontsize(10)
            label.set_color("#111827")
            label.set_bbox({
                "boxstyle": "round,pad=0.20",
                "facecolor": "#EFF6FF",
                "edgecolor": "#2563EB",
                "linewidth": 0.8,
            })

axes[0].text(
    0,
    1.,
    "Outlined bars show the lowest RMSE within each segment.",
    transform=axes[0].transAxes,
    fontsize=9.5,
    color="#4B5563",
)

axes[1].text(
    0,
    1.,
    "Outlined bars show the mean bias closest to zero.",
    transform=axes[1].transAxes,
    fontsize=9.5,
    color="#4B5563",
)

fig.suptitle(
    "Do Concert Features Improve Forecasts Where They Should?",
    x=0.08,
    y=0.985,
    ha="left",
    fontsize=17,
    fontweight="bold",
)

fig.text(
    0.08,
    0.952,
    (
        "Models use the earlier training period; all comparisons use "
        "the same later validation rows. The test period remains untouched."
    ),
    ha="left",
    fontsize=10.5,
    color="#4B5563",
)

# Reserving space above the charts for the shared legend.
plt.tight_layout(
    rect=[0, 0, 1, 0.86],
    h_pad=3.0,
)
plt.show()

### Which concert windows should stay?

I'll keep both concert flags. Together they give a better balance of event errors and bias than either simpler version. Their timing is approximate: setlist.fm supplies dates and venues, but not consistent ending times. These results support the flags as predictors, not as verified departure records. Earlier window exploration also used dates overlapping later evaluation, which is why new-period confirmation still matters.


## What changes when I remove sports?

Now that the concert definition is fixed, I'll remove only `is_post_game_demand`. This asks what the approximate post-game window adds beyond the lags, weather, calendar, and concert information already in the model.

| Experiment | Sports feature | Concert features | Status |
|---|---:|---:|---|
| All event features | Included | Both windows included | Already trained |
| Sports only | Included | Excluded | Already trained as the no-concert model |
| Concerts only | Excluded | Both windows included | To be trained |
| No event features | Excluded | Excluded | To be trained |


In [ ]:
# Releasing heavy objects from the rejected narrow-only experiment.
del narrow_only_xgb_model
del narrow_only_xgb_history
del narrow_only_training_matrix
del narrow_only_validation_matrix
del narrow_only_xgb_predictions

gc.collect()

# Checking that all concert-ablation predictions remain available.
assert len(narrow_only_xgb_predictions_constrained) == len(y_validation)
assert len(wide_only_xgb_predictions_constrained) == len(y_validation)
assert len(no_concert_xgb_predictions_constrained) == len(y_validation)
assert len(lambda10_xgb_predictions_constrained) == len(y_validation)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Concert-ablation predictions preserved:",
    True,
)
print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Defining the complete set of event features.
sports_feature_name = "is_post_game_demand"

all_event_feature_names = [
    sports_feature_name,
    "is_concert_window",
    "is_wide_concert_window",
]

assert set(all_event_feature_names).issubset(
    X_train.columns
)

# Defining the controlled two-by-two event feature sets.
event_ablation_features = {
    "All event features": X_train.columns.tolist(),
    "Sports only": [
        feature for feature in X_train.columns
        if feature not in concert_feature_names
    ],
    "Concerts only": [
        feature for feature in X_train.columns
        if feature != sports_feature_name
    ],
    "No event features": [
        feature for feature in X_train.columns
        if feature not in all_event_feature_names
    ],
}

# Validating the intended feature counts.
expected_event_feature_counts = {
    "All event features": 30,
    "Sports only": 28,
    "Concerts only": 29,
    "No event features": 27,
}

assert {
    experiment: len(features)
    for experiment, features in event_ablation_features.items()
} == expected_event_feature_counts

event_ablation_design = pd.DataFrame({
    "Experiment": event_ablation_features.keys(),
    "Feature Count": [
        len(features)
        for features in event_ablation_features.values()
    ],
    "Sports Retained": [
        sports_feature_name in features
        for features in event_ablation_features.values()
    ],
    "Concert Features Retained": [
        sum(
            feature in features
            for feature in concert_feature_names
        )
        for features in event_ablation_features.values()
    ],
    "Training Status": [
        "Already trained",
        "Already trained",
        "To be trained",
        "To be trained",
    ],
})

display(
    event_ablation_design.style
    .hide(axis="index")
)

In [ ]:
# Checking overlap between estimated sports and concert windows.
event_overlap_results = []

for period_name, feature_frame in [
    ("Earlier training period", X_train),
    ("Later validation period", X_validation),
]:
    event_overlap_frame = pd.DataFrame({
        "sports_window": feature_frame[
            "is_post_game_demand"
        ].to_numpy(),
        "any_concert_window": feature_frame[
            "is_wide_concert_window"
        ].to_numpy(),
    })

    period_overlap = (
        event_overlap_frame
        .groupby(
            ["sports_window", "any_concert_window"],
            observed=True,
        )
        .size()
        .rename("rows")
        .reset_index()
    )

    period_overlap["share_percent"] = (
        100 * period_overlap["rows"]
        / len(feature_frame)
    )

    period_overlap.insert(
        0,
        "period",
        period_name,
    )

    event_overlap_results.append(period_overlap)

event_window_overlap = pd.concat(
    event_overlap_results,
    ignore_index=True,
)

display(
    event_window_overlap.style
    .format({
        "rows": "{:,.0f}",
        "share_percent": "{:.4f}",
    })
    .hide(axis="index")
)

### Checking Whether Sports and Concert Windows Overlap

Sports and concert windows rarely overlap in these data. That makes their separate segments easier to read, although the number of distinct events and target hours is still much smaller than the repeated forecast-row count.


In [ ]:
# Selecting the concerts-only feature set.
concerts_only_feature_names = event_ablation_features[
    "Concerts only"
]

assert "is_post_game_demand" not in concerts_only_feature_names
assert "is_concert_window" in concerts_only_feature_names
assert "is_wide_concert_window" in concerts_only_feature_names
assert len(concerts_only_feature_names) == 29

# Building paired concerts-only matrices with training-derived cuts.
(
    concerts_only_training_matrix,
    concerts_only_validation_matrix,
    concerts_only_matrix_seconds,
) = build_quantile_matrix_pair(
    training_features=(
        X_train[
            concerts_only_feature_names
        ]
    ),
    training_target=y_train,
    evaluation_features=(
        X_validation[
            concerts_only_feature_names
        ]
    ),
    evaluation_target=y_validation,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)

# Validating the matrix structure.
assert concerts_only_training_matrix.num_row() == len(X_train)
assert concerts_only_validation_matrix.num_row() == len(X_validation)
assert concerts_only_training_matrix.num_col() == 29
assert concerts_only_validation_matrix.num_col() == 29

concerts_only_matrix_summary = pd.DataFrame({
    "Measure": [
        "Training rows",
        "Validation rows",
        "Features",
        "Build time (seconds)",
    ],
    "Value": [
        f"{concerts_only_training_matrix.num_row():,}",
        f"{concerts_only_validation_matrix.num_row():,}",
        concerts_only_training_matrix.num_col(),
        f"{concerts_only_matrix_seconds:.2f}",
    ],
})

display(
    concerts_only_matrix_summary.style
    .hide(axis="index")
)

In [ ]:
# Training the concerts-only event model.
(
    concerts_only_xgb_model,
    concerts_only_xgb_history,
    concerts_only_training_seconds,
) = train_xgboost_candidate(
    parameters=concert_ablation_xgb_params,
    training_matrix=concerts_only_training_matrix,
    evaluation_matrices=[
        (
            concerts_only_training_matrix,
            "training",
        ),
        (
            concerts_only_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=100,
)

# Converting the reusable timer to minutes.
concerts_only_training_minutes = (
    concerts_only_training_seconds
    / 60
)

print()
print(
    "Best boosting round:",
    f"{concerts_only_xgb_model.best_iteration + 1:,}",
)
print(
    "Best validation RMSE:",
    f"{concerts_only_xgb_model.best_score:.4f}",
)
print(
    "Training time (minutes):",
    f"{concerts_only_training_minutes:.2f}",
)

In [ ]:
# Generating constrained concerts-only predictions.
(
    concerts_only_xgb_predictions,
    concerts_only_xgb_predictions_constrained,
    concerts_only_negative_prediction_count,
    concerts_only_prediction_seconds,
) = predict_xgboost_demand(
    model=concerts_only_xgb_model,
    prediction_matrix=(
        concerts_only_validation_matrix
    ),
    iteration_end=(
        concerts_only_xgb_model.best_iteration
        + 1
    ),
)

assert len(concerts_only_xgb_predictions) == len(
    y_validation
)
assert np.isfinite(concerts_only_xgb_predictions).all()

# Defining sports-focused validation segments.
sports_validation_mask = (
    X_validation["is_post_game_demand"]
    .to_numpy()
)

concert_validation_mask = (
    X_validation["is_wide_concert_window"]
    .to_numpy()
)

sports_validation_masks = {
    "All validation rows": np.ones(
        len(X_validation),
        dtype=bool,
    ),
    "Any sports-window row": sports_validation_mask,
    "Isolated sports-window row": (
        sports_validation_mask
        & ~concert_validation_mask
    ),
    "Non-sports rows": ~sports_validation_mask,
}

assert sports_validation_masks[
    "Any sports-window row"
].sum() == 712

assert sports_validation_masks[
    "Isolated sports-window row"
].sum() == 700

# Preparing the shared validation values.
sports_validation_base = pd.DataFrame({
    "total_pickups": np.asarray(y_validation),
    "sports_included_prediction": (
        lambda10_xgb_predictions_constrained
    ),
    "sports_excluded_prediction": (
        concerts_only_xgb_predictions_constrained
    ),
})

# Expanding overlapping rows into their applicable segments.
sports_segment_frames = []

for segment_name, segment_mask in (
    sports_validation_masks.items()
):
    assert segment_mask.sum() > 0

    segment_frame = sports_validation_base.loc[
        segment_mask
    ].copy()

    segment_frame.insert(
        0,
        "Segment",
        segment_name,
    )

    sports_segment_frames.append(segment_frame)

sports_validation_frame = pd.concat(
    sports_segment_frames,
    ignore_index=True,
)

# Comparing models that differ only by the sports feature.
sports_incremental_results = (
    calculate_metrics_by_group(
        evaluation_frame=sports_validation_frame,
        group_columns="Segment",
        prediction_columns={
            "Sports included": (
                "sports_included_prediction"
            ),
            "Sports excluded": (
                "sports_excluded_prediction"
            ),
        },
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    [
        [
            "Segment",
            "Model",
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

# Checking complete segment and model coverage.
assert sports_incremental_results.shape[0] == 8

assert (
    sports_incremental_results
    .groupby("Segment")["Model"]
    .nunique()
    .eq(2)
    .all()
)

# Releasing temporary grouped-metric objects.
del sports_validation_base
del sports_segment_frames
del sports_validation_frame
del segment_frame

display(
    sports_incremental_results.style
    .format({
        "Rows": "{:,.0f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .hide(axis="index")
)

In [ ]:
# Isolating sports rows without concert overlap.
isolated_sports_mask = sports_validation_masks[
    "Isolated sports-window row"
]

isolated_sports_evaluation = pd.DataFrame({
    "target_hour_utc": validation_evaluation.loc[
        isolated_sports_mask,
        "target_hour_utc",
    ].to_numpy(),
    "location_id": X_validation.loc[
        isolated_sports_mask,
        "location_id",
    ].astype("int64").to_numpy(),
    "forecast_horizon": X_validation.loc[
        isolated_sports_mask,
        "forecast_horizon",
    ].astype("int64").to_numpy(),
    "actual": y_validation[isolated_sports_mask],
    "sports_included": (
        lambda10_xgb_predictions_constrained[
            isolated_sports_mask
        ]
    ),
    "sports_excluded": (
        concerts_only_xgb_predictions_constrained[
            isolated_sports_mask
        ]
    ),
})

# Checking that repeated forecasts share the same realized target.
actual_values_per_target = (
    isolated_sports_evaluation
    .groupby(
        ["target_hour_utc", "location_id"],
        observed=True,
    )["actual"]
    .nunique()
)

assert actual_values_per_target.max() == 1

# Aggregating repeated horizons to distinct target zone-hours.
isolated_sports_targets = (
    isolated_sports_evaluation
    .groupby(
        ["target_hour_utc", "location_id"],
        observed=True,
    )
    .agg(
        forecast_rows=("actual", "size"),
        actual=("actual", "first"),
        sports_included=("sports_included", "mean"),
        sports_excluded=("sports_excluded", "mean"),
    )
    .reset_index()
)

sports_effective_sample_summary = pd.DataFrame({
    "Measure": [
        "Forecast rows",
        "Distinct target zone-hours",
        "Distinct target hours",
        "Distinct venue zones",
        "Minimum forecasts per target",
        "Median forecasts per target",
        "Maximum forecasts per target",
    ],
    "Value": [
        len(isolated_sports_evaluation),
        len(isolated_sports_targets),
        isolated_sports_targets["target_hour_utc"].nunique(),
        isolated_sports_targets["location_id"].nunique(),
        isolated_sports_targets["forecast_rows"].min(),
        isolated_sports_targets["forecast_rows"].median(),
        isolated_sports_targets["forecast_rows"].max(),
    ],
})

display(
    sports_effective_sample_summary.style
    .format({"Value": "{:,.0f}"})
    .hide(axis="index")
)

# Comparing models after weighting each target zone-hour once.
sports_unique_target_results = (
    compare_forecast_models(
        actual=isolated_sports_targets["actual"],
        model_predictions={
            "Sports included": (
                isolated_sports_targets[
                    "sports_included"
                ]
            ),
            "Sports excluded": (
                isolated_sports_targets[
                    "sports_excluded"
                ]
            ),
        },
        model_column="Model",
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
)

# Preserving the existing target-count column.
sports_unique_target_results.insert(
    1,
    "Target Zone-Hours",
    len(isolated_sports_targets),
)

# Preserving the original displayed columns.
sports_unique_target_results = (
    sports_unique_target_results[
        [
            "Model",
            "Target Zone-Hours",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

assert len(sports_unique_target_results) == 2

display(
    sports_unique_target_results.style
    .format({
        "Target Zone-Hours": "{:,.0f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .hide(axis="index")
)

In [ ]:
# Calculating both models within each sports venue zone.
sports_zone_results = (
    calculate_metrics_by_group(
        evaluation_frame=isolated_sports_targets,
        group_columns="location_id",
        prediction_columns={
            "Sports included": "sports_included",
            "Sports excluded": "sports_excluded",
        },
        actual_column="actual",
    )
    .rename(columns={
        "Rows": "Target Zone-Hours",
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
)

# Calculating mean actual demand once per venue zone.
sports_zone_mean_demand = (
    isolated_sports_targets
    .groupby(
        "location_id",
        observed=True,
    )["actual"]
    .mean()
)

sports_zone_results[
    "Mean Actual Demand"
] = sports_zone_results[
    "location_id"
].map(sports_zone_mean_demand)

# Preserving the original location and model order.
sports_zone_results = (
    sports_zone_results
    .sort_values(
        ["location_id", "Model"]
    )
    .reset_index(drop=True)
)

# Mapping sports venue zones to their home teams.
sports_venue_labels = {
    93: "Citi Field — New York Mets",
    181: "Barclays Center — Brooklyn Nets",
    186: (
        "Madison Square Garden — "
        "Knicks and Rangers"
    ),
    247: "Yankee Stadium — New York Yankees",
}

# Adding readable venue labels.
sports_zone_results["Venue"] = (
    sports_zone_results["location_id"]
    .map(sports_venue_labels)
)

assert sports_zone_results["Venue"].notna().all()

assert (
    sports_zone_results
    .groupby(
        "location_id",
        observed=True,
    )["Model"]
    .nunique()
    .eq(2)
    .all()
)

# Releasing the temporary grouped summary.
del sports_zone_mean_demand

sports_zone_results_display = sports_zone_results[
    [
        "location_id",
        "Venue",
        "Model",
        "Target Zone-Hours",
        "Mean Actual Demand",
        "MAE",
        "RMSE",
        "WAPE (%)",
        "Mean Bias",
    ]
]

display(
    sports_zone_results_display.style
    .format({
        "location_id": "{:.0f}",
        "Target Zone-Hours": "{:,.0f}",
        "Mean Actual Demand": "{:.2f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .set_properties(
        subset=["Venue", "Model"],
        **{"text-align": "left"},
    )
    .hide(axis="index")
)

In [ ]:
# Preparing a clearly ordered venue-level table.
sports_zone_presentation = (
    sports_zone_results_display
    .copy()
)

sports_model_order = [
    "Sports included",
    "Sports excluded",
]

sports_zone_presentation["Model"] = pd.Categorical(
    sports_zone_presentation["Model"],
    categories=sports_model_order,
    ordered=True,
)

sports_zone_presentation = (
    sports_zone_presentation
    .sort_values(["location_id", "Model"])
    .reset_index(drop=True)
)

# Shading rows according to the sports-feature configuration.
def shade_sports_model_rows(row):
    if row["Model"] == "Sports included":
        row_style = (
            "background-color: #2563EB; "
            "color: white; "
            "font-weight: bold; "
            "border-bottom: 1px solid #D1D5DB;"
        )
    else:
        row_style = (
            "background-color: #505762; "
            "color: white; "
            "border-bottom: 1px solid #D1D5DB;"
        )

    return [row_style] * len(row)


display(
    sports_zone_presentation.style
    .format({
        "location_id": "{:.0f}",
        "Target Zone-Hours": "{:,.0f}",
        "Mean Actual Demand": "{:.2f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .apply(
        shade_sports_model_rows,
        axis=1,
    )
    .set_properties(
        subset=["Venue", "Model"],
        **{"text-align": "left"},
    )
    .hide(axis="index")
)


# Preparing shorter venue labels with effective sample sizes.
sports_venue_chart_labels = {
    93: "Citi Field\n(n=6)",
    181: "Barclays Center\n(n=42)",
    186: "Madison Square Garden\n(n=71)",
}

sports_zone_chart_data = (
    sports_zone_presentation
    .copy()
)

sports_zone_chart_data["Venue Label"] = (
    sports_zone_chart_data["location_id"]
    .map(sports_venue_chart_labels)
)

sports_model_palette = {
    "Sports included": "#2563EB",
    "Sports excluded": "#505762",
}

# Creating vertically stacked venue comparisons.
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(11, 10),
    dpi=560,
)

sns.barplot(
    data=sports_zone_chart_data,
    x="Venue Label",
    y="RMSE",
    hue="Model",
    hue_order=sports_model_order,
    palette=sports_model_palette,
    errorbar=None,
    ax=axes[0],
)

axes[0].set_title(
    "Sports Information Reduces Large Errors at Every Venue",
    loc="left",
)
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE")

# Emphasizing the sports-included RMSE values.
for model_position, container in enumerate(
    axes[0].containers
):
    value_labels = axes[0].bar_label(
        container,
        fmt="%.1f",
        padding=4,
        fontsize=12.5,
    )

    if model_position == 0:
        for bar, label in zip(container, value_labels):
            bar.set_edgecolor("#1E3A8A")
            bar.set_linewidth(2.0)
            label.set_fontweight("bold")
            label.set_bbox({
                "boxstyle": "round,pad=0.18",
                "facecolor": "#EFF6FF",
                "edgecolor": "#2563EB",
                "linewidth": 0.8,
            })

sns.barplot(
    data=sports_zone_chart_data,
    x="Venue Label",
    y="Mean Bias",
    hue="Model",
    hue_order=sports_model_order,
    palette=sports_model_palette,
    errorbar=None,
    ax=axes[1],
)

axes[1].axhline(
    0,
    color="#4B5563",
    linewidth=1.0,
    linestyle="--",
)
axes[1].set_title(
    "Sports Information Removes Severe Venue Underprediction",
    loc="left",
)
axes[1].set_xlabel("Sports venue zone")
axes[1].set_ylabel("Mean bias")

# Emphasizing the sports-included bias values.
for model_position, container in enumerate(
    axes[1].containers
):
    value_labels = axes[1].bar_label(
        container,
        fmt="%+.1f",
        padding=4,
        fontsize=12.5,
    )

    if model_position == 0:
        for bar, label in zip(container, value_labels):
            bar.set_edgecolor("#1E3A8A")
            bar.set_linewidth(2.0)
            label.set_fontweight("bold")
            label.set_bbox({
                "boxstyle": "round,pad=0.18",
                "facecolor": "#EFF6FF",
                "edgecolor": "#2563EB",
                "linewidth": 0.8,
            })

# Moving the shared legend outside the plotting areas.
legend_handles, legend_labels = (
    axes[0].get_legend_handles_labels()
)

axes[0].legend_.remove()
axes[1].legend_.remove()

fig.legend(
    legend_handles,
    legend_labels,
    title="Feature configuration",
    loc="upper center",
    bbox_to_anchor=(0.5, 0.91),
    ncols=2,
    frameon=False,
)

# Keeping only horizontal grid lines.
for axis in axes:
    axis.grid(axis="y", visible=True)
    axis.grid(axis="x", visible=False)
    axis.margins(y=0.15)
    sns.despine(
        ax=axis,
        top=True,
        right=True,
        left=True,
    )

fig.suptitle(
    "Does the Estimated Post-Game Window Improve Venue Forecasts?",
    x=0.08,
    y=0.985,
    ha="left",
    fontsize=17,
    fontweight="bold",
)

fig.text(
    0.08,
    0.952,
    (
        "Metrics give each distinct target zone-hour equal weight "
        "in the later validation period."
    ),
    ha="left",
    fontsize=10.5,
    color="#4B5563",
)

plt.tight_layout(
    rect=[0, 0, 1, 0.86],
    h_pad=3.0,
)
plt.show()

### Interpreting the Sports-Event Results by Venue

The sports flag helps at every venue zone represented in validation. Citi Field stands out: without the flag, the model badly underpredicts activity after games. This is a strong reason to keep the feature, while remembering that the window still comes from assumed game durations.


In [ ]:
# Defining all engineered professional sports venue zones.
sports_venue_audit_labels = {
    93: "Citi Field — Mets",
    181: "Barclays Center — Nets",
    186: "Madison Square Garden — Knicks/Rangers",
    247: "Yankee Stadium — Yankees",
}

sports_venue_ids = list(
    sports_venue_audit_labels
)

# Counting flagged forecast rows in each permitted period.
training_sports_counts = (
    X_train.loc[
        X_train["is_post_game_demand"],
        "location_id",
    ]
    .astype("int64")
    .value_counts()
    .reindex(sports_venue_ids, fill_value=0)
)

validation_sports_counts = (
    X_validation.loc[
        X_validation["is_post_game_demand"],
        "location_id",
    ]
    .astype("int64")
    .value_counts()
    .reindex(sports_venue_ids, fill_value=0)
)

sports_venue_coverage_audit = pd.DataFrame({
    "location_id": sports_venue_ids,
    "Venue": [
        sports_venue_audit_labels[location_id]
        for location_id in sports_venue_ids
    ],
    "Earlier Training Rows": (
        training_sports_counts.to_numpy()
    ),
    "Later Validation Rows": (
        validation_sports_counts.to_numpy()
    ),
})

# Checking that all sports flags belong to mapped venue zones.
assert set(
    X_train.loc[
        X_train["is_post_game_demand"],
        "location_id",
    ].astype("int64").unique()
).issubset(sports_venue_ids)

assert set(
    X_validation.loc[
        X_validation["is_post_game_demand"],
        "location_id",
    ].astype("int64").unique()
).issubset(sports_venue_ids)

display(
    sports_venue_coverage_audit.style
    .format({
        "location_id": "{:.0f}",
        "Earlier Training Rows": "{:,.0f}",
        "Later Validation Rows": "{:,.0f}",
    })
    .set_properties(
        subset=["Venue"],
        **{"text-align": "left"},
    )
    .hide(axis="index")
)

### Where is Yankee Stadium in this comparison?

Yankee Stadium is absent from the validation event comparison because January–March has no flagged games there, not because its mapping was lost. Training does contain its event hours. A later baseball-season evaluation will give a more useful test for that venue.


In [ ]:
# Releasing heavy objects from the completed sports-feature control.
del concerts_only_xgb_model
del concerts_only_xgb_history
del concerts_only_training_matrix
del concerts_only_validation_matrix
del concerts_only_xgb_predictions

gc.collect()

# Checking that the paired sports predictions remain available.
assert len(
    concerts_only_xgb_predictions_constrained
) == len(y_validation)

assert len(
    lambda10_xgb_predictions_constrained
) == len(y_validation)

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Sports comparison predictions preserved:",
    True,
)
print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Selecting the feature set without sports or concert indicators.
no_event_feature_names = event_ablation_features[
    "No event features"
]

event_feature_names = [
    "is_post_game_demand",
    "is_concert_window",
    "is_wide_concert_window",
]

# Confirming that all event indicators are excluded.
assert all(
    feature_name not in no_event_feature_names
    for feature_name in event_feature_names
)
assert len(no_event_feature_names) == 27

# Building paired no-event matrices with training-derived cuts.
(
    no_event_training_matrix,
    no_event_validation_matrix,
    no_event_matrix_seconds,
) = build_quantile_matrix_pair(
    training_features=(
        X_train[
            no_event_feature_names
        ]
    ),
    training_target=y_train,
    evaluation_features=(
        X_validation[
            no_event_feature_names
        ]
    ),
    evaluation_target=y_validation,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)

# Validating the resulting matrix structure.
assert no_event_training_matrix.num_row() == len(X_train)
assert no_event_validation_matrix.num_row() == len(X_validation)
assert no_event_training_matrix.num_col() == 27
assert no_event_validation_matrix.num_col() == 27

no_event_matrix_summary = pd.DataFrame({
    "Measure": [
        "Training rows",
        "Validation rows",
        "Features",
        "Build time (seconds)",
    ],
    "Value": [
        f"{no_event_training_matrix.num_row():,}",
        f"{no_event_validation_matrix.num_row():,}",
        no_event_training_matrix.num_col(),
        f"{no_event_matrix_seconds:.2f}",
    ],
})

display(
    no_event_matrix_summary.style
    .hide(axis="index")
)


In [ ]:
# Copying the retained lambda-10 parameters.
no_event_xgb_params = (
    concert_ablation_xgb_params.copy()
)

# Training without sports or concert features.
(
    no_event_xgb_model,
    no_event_xgb_history,
    no_event_training_seconds,
) = train_xgboost_candidate(
    parameters=no_event_xgb_params,
    training_matrix=no_event_training_matrix,
    evaluation_matrices=[
        (
            no_event_training_matrix,
            "training",
        ),
        (
            no_event_validation_matrix,
            "validation",
        ),
    ],
    num_boost_round=maximum_boosting_rounds,
    early_stopping_rounds=(
        early_stopping_rounds
    ),
    verbose_eval=100,
)

# Converting the reusable timer to minutes.
no_event_training_minutes = (
    no_event_training_seconds
    / 60
)
# Extracting the best validation result.
no_event_best_round = (
    no_event_xgb_model.best_iteration + 1
)

no_event_best_rmse = (
    no_event_xgb_model.best_score
)

print()
print(
    "Best boosting round:",
    f"{no_event_best_round:,}",
)
print(
    "Best validation RMSE:",
    f"{float(no_event_best_rmse):.4f}",
)
print(
    "Training time (minutes):",
    f"{no_event_training_minutes:.2f}",
)

In [ ]:
# Generating constrained predictions without event features.
(
    no_event_xgb_predictions,
    no_event_xgb_predictions_constrained,
    no_event_negative_predictions,
    no_event_prediction_seconds,
) = predict_xgboost_demand(
    model=no_event_xgb_model,
    prediction_matrix=(
        no_event_validation_matrix
    ),
    iteration_end=(
        no_event_xgb_model.best_iteration
        + 1
    ),
)

# Checking the prediction structure before evaluation.
assert len(
    no_event_xgb_predictions_constrained
) == len(y_validation)

assert np.isfinite(
    no_event_xgb_predictions_constrained
).all()

assert (
    no_event_xgb_predictions_constrained >= 0
).all()

# Calculating validation metrics on the constrained forecasts.
no_event_metrics = calculate_forecast_metrics(
    actual=y_validation,
    prediction=no_event_xgb_predictions_constrained,
)

no_event_results = pd.DataFrame({
    "Model": [
        "XGBoost without event features",
    ],
    "MAE": [
        no_event_metrics["mae"],
    ],
    "RMSE": [
        no_event_metrics["rmse"],
    ],
    "WAPE (%)": [
        no_event_metrics["wape_percent"],
    ],
    "Mean Bias": [
        no_event_metrics["mean_bias"],
    ],
    "R²": [
        no_event_metrics["r_squared"],
    ],
    "Negative Predictions": [
        no_event_negative_predictions,
    ],
    "Prediction Seconds": [
        no_event_prediction_seconds,
    ],
})

display(
    no_event_results.style
    .hide(axis="index")
    .format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
        "Negative Predictions": "{:,}",
        "Prediction Seconds": "{:.2f}",
    })
)

In [ ]:
# Defining segments for the controlled sports comparison.
sports_ablation_masks = {
    "All validation rows": np.ones(
        len(y_validation),
        dtype=bool,
    ),
    "Any sports-window row": (
        sports_validation_mask
    ),
    "Isolated sports-window row": (
        sports_validation_mask
        & ~concert_validation_mask
    ),
    "Non-sports rows": (
        ~sports_validation_mask
    ),
}

# Checking the most important comparison sizes.
assert (
    sports_ablation_masks[
        "Any sports-window row"
    ].sum()
    == 712
)

assert (
    sports_ablation_masks[
        "Isolated sports-window row"
    ].sum()
    == 700
)

# Preparing the shared validation values.
sports_ablation_base = pd.DataFrame({
    "total_pickups": np.asarray(y_validation),
    "sports_included_prediction": (
        no_concert_xgb_predictions_constrained
    ),
    "sports_excluded_prediction": (
        no_event_xgb_predictions_constrained
    ),
})

# Expanding overlapping rows into their applicable segments.
sports_ablation_segment_frames = []

for segment_name, segment_mask in (
    sports_ablation_masks.items()
):
    assert segment_mask.sum() > 0

    segment_frame = sports_ablation_base.loc[
        segment_mask
    ].copy()

    segment_frame.insert(
        0,
        "Segment",
        segment_name,
    )

    sports_ablation_segment_frames.append(
        segment_frame
    )

sports_ablation_frame = pd.concat(
    sports_ablation_segment_frames,
    ignore_index=True,
)

# Comparing both models within each validation segment.
sports_ablation_results = (
    calculate_metrics_by_group(
        evaluation_frame=sports_ablation_frame,
        group_columns="Segment",
        prediction_columns={
            "Sports included": (
                "sports_included_prediction"
            ),
            "Sports excluded": (
                "sports_excluded_prediction"
            ),
        },
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    [
        [
            "Segment",
            "Model",
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

# Checking complete segment and model coverage.
assert sports_ablation_results.shape[0] == 8

assert (
    sports_ablation_results
    .groupby("Segment")["Model"]
    .nunique()
    .eq(2)
    .all()
)

# Releasing temporary grouped-metric objects.
del sports_ablation_base
del sports_ablation_segment_frames
del sports_ablation_frame
del segment_frame

display(
    sports_ablation_results.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
    .set_properties(
        subset=["Segment"],
        **{"font-weight": "bold"},
    )
    .set_table_styles([
        {
            "selector": "tbody tr:nth-child(4n+1), "
                        "tbody tr:nth-child(4n+2)",
            "props": [
                ("background-color", "#a0c6f8"),
            ],
        },
    ])
)

In [ ]:
# Defining the completed event-feature experiment.
event_factorial_predictions = {
    "All event features": (
        lambda10_xgb_predictions_constrained
    ),
    "Concert features only": (
        concerts_only_xgb_predictions_constrained
    ),
    "Sports feature only": (
        no_concert_xgb_predictions_constrained
    ),
    "No event features": (
        no_event_xgb_predictions_constrained
    ),
}

event_factorial_structure = {
    "All event features": {
        "Sports": "Included",
        "Concert Windows": "Included",
        "Features": 30,
    },
    "Concert features only": {
        "Sports": "Excluded",
        "Concert Windows": "Included",
        "Features": 29,
    },
    "Sports feature only": {
        "Sports": "Included",
        "Concert Windows": "Excluded",
        "Features": 28,
    },
    "No event features": {
        "Sports": "Excluded",
        "Concert Windows": "Excluded",
        "Features": 27,
    },
}

# Calculating overall metrics for all four configurations.
event_factorial_results = (
    compare_forecast_models(
        actual=y_validation,
        model_predictions=(
            event_factorial_predictions
        ),
        model_column="Model",
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
)

# Adding the controlled feature structure.
event_factorial_results[
    "Sports"
] = event_factorial_results[
    "Model"
].map({
    model_name: settings["Sports"]
    for model_name, settings in (
        event_factorial_structure.items()
    )
})

event_factorial_results[
    "Concert Windows"
] = event_factorial_results[
    "Model"
].map({
    model_name: settings["Concert Windows"]
    for model_name, settings in (
        event_factorial_structure.items()
    )
})

event_factorial_results[
    "Features"
] = event_factorial_results[
    "Model"
].map({
    model_name: settings["Features"]
    for model_name, settings in (
        event_factorial_structure.items()
    )
})

# Preserving the original columns and RMSE ranking.
event_factorial_results = (
    event_factorial_results[
        [
            "Model",
            "Sports",
            "Concert Windows",
            "Features",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
            "R²",
        ]
    ]
    .sort_values("RMSE")
    .reset_index(drop=True)
)

assert len(event_factorial_results) == 4
assert event_factorial_results["Model"].is_unique

display(
    event_factorial_results.style
    .hide(axis="index")
    .format({
        "Features": "{:,}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .highlight_min(
        subset=[
            "MAE",
            "RMSE",
            "WAPE (%)",
        ],
        color="#67ddce",
    )
    .highlight_max(
        subset=["R²"],
        color="#67ddce",
    )
)

### Which event features will I keep?

I'll retain sports and both concert flags. Their pooled contribution is small because event hours are rare, but the sports-only target segment shows why that small average can matter: the flag greatly reduces severe underprediction near venues. I still need more seasons and events to judge how stable those gains are.


In [ ]:
# Releasing heavy objects from the completed no-event experiment.
del no_event_xgb_model
del no_event_xgb_history
del no_event_training_matrix
del no_event_validation_matrix
del no_event_xgb_predictions

gc.collect()

# Checking that all four ablation prediction arrays remain available.
event_prediction_arrays = {
    "All event features": (
        lambda10_xgb_predictions_constrained
    ),
    "Concert features only": (
        concerts_only_xgb_predictions_constrained
    ),
    "Sports feature only": (
        no_concert_xgb_predictions_constrained
    ),
    "No event features": (
        no_event_xgb_predictions_constrained
    ),
}

for model_name, predictions in (
    event_prediction_arrays.items()
):
    assert len(predictions) == len(y_validation)
    assert np.isfinite(predictions).all()

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Event-ablation predictions preserved:",
    True,
)
print(
    "Notebook process RAM after cleanup (GB):",
    f"{process_ram_gb:.2f}",
)

## How does the chosen model do on later months?

This is where I stop changing the model and ask how it does on later months. I want the same comparison I started with: how much better is it than repeating last week, and where would I still be wrong?

The model choices are now fixed: 30 features, lambda 10, and 1,613 rounds. I'll refit on training plus validation, keep the purged test boundary, and score the later period without using it for early stopping or another round of tuning. Weaknesses found here become follow-up work. This period is held out from the final fitting decisions, with the earlier event-exploration qualification noted above.


In [ ]:
# Selecting the frozen modeling splits from the manifest.
final_training_origins = split_manifest[
    split_manifest["split"].eq("training")
].copy()

final_validation_origins = split_manifest[
    split_manifest["split"].eq("validation")
].copy()

final_test_origins = split_manifest[
    split_manifest["split"].eq("test")
].copy()

final_purged_origins = split_manifest[
    split_manifest["split"].eq("purged_boundary")
].copy()

# Calculating the chronological gap before the test targets.
validation_last_target = (
    final_validation_origins["target_end_utc"].max()
)

test_first_target = (
    final_test_origins["target_start_utc"].min()
)

validation_test_gap_hours = (
    test_first_target - validation_last_target
).total_seconds() / 3600

# Checking the frozen split structure.
assert len(final_training_origins) == 12_208
assert len(final_validation_origins) == 2_153
assert len(final_test_origins) == 2_178
assert len(final_purged_origins) == 12

assert int(
    final_test_origins["rows"].sum()
) == 3_436_884

assert validation_last_target < test_first_target
assert validation_test_gap_hours > 0

test_split_audit = pd.DataFrame({
    "Measure": [
        "Test forecast origins",
        "Expected rows",
        "First test origin",
        "Last test origin",
        "First test target",
        "Last test target",
        "Validation last target",
        "Validation-to-test gap (hours)",
        "Purged boundary origins",
    ],
    "Value": [
        f"{len(final_test_origins):,}",
        f"{int(final_test_origins['rows'].sum()):,}",
        final_test_origins[
            "forecast_origin_utc"
        ].min(),
        final_test_origins[
            "forecast_origin_utc"
        ].max(),
        final_test_origins[
            "target_start_utc"
        ].min(),
        final_test_origins[
            "target_end_utc"
        ].max(),
        validation_last_target,
        f"{validation_test_gap_hours:.1f}",
        f"{len(final_purged_origins):,}",
    ],
})

display(
    test_split_audit.style
    .hide(axis="index")
)

In [ ]:
# Reconstructing the production partition inventory.
final_partition_paths = sorted(
    PRODUCTION_DATA_ROOT.glob(
        "year=*/month=*/production_features_*.parquet"
    )
)

assert len(final_partition_paths) == 23

# Reading the production schema without loading dataset rows.
production_schema_names = (
    pq.ParquetFile(final_partition_paths[0])
    .schema_arrow
    .names
)

# Reusing the approved model and evaluation loading contract.
test_loading_columns = model_loading_columns.copy()

assert len(test_loading_columns) == 37
assert test_loading_columns == model_loading_columns

# Checking that every requested column exists.
missing_test_columns = sorted(
    set(test_loading_columns)
    - set(production_schema_names)
)

assert not missing_test_columns, (
    f"Missing test columns: {missing_test_columns}"
)

# Constructing the Arrow dataset from feature partitions only.
production_feature_dataset = ds.dataset(
    [
        str(partition_path)
        for partition_path in final_partition_paths
    ],
    format="parquet",
)

test_first_origin = (
    final_test_origins["forecast_origin_utc"].min()
)

test_last_origin = (
    final_test_origins["forecast_origin_utc"].max()
)

# Loading only rows within the frozen test-origin range.
test_arrow_table = production_feature_dataset.to_table(
    columns=test_loading_columns,
    filter=(
        (
            ds.field("forecast_origin_utc")
            >= test_first_origin.to_pydatetime()
        )
        & (
            ds.field("forecast_origin_utc")
            <= test_last_origin.to_pydatetime()
        )
    ),
)

# Auditing the loaded test structure.
loaded_test_origins = pc.count_distinct(
    test_arrow_table["forecast_origin_utc"]
).as_py()

loaded_test_targets = pc.count_distinct(
    test_arrow_table["target_hour_utc"]
).as_py()

test_arrow_size_gb = (
    test_arrow_table.nbytes / 1024**3
)

assert test_arrow_table.num_rows == 3_436_884
assert loaded_test_origins == 2_178
assert test_arrow_table.num_columns == len(
    test_loading_columns
)

test_loading_summary = pd.DataFrame({
    "Measure": [
        "Loaded test rows",
        "Loaded columns",
        "Distinct forecast origins",
        "Distinct target hours",
        "Arrow table size (GB)",
    ],
    "Value": [
        f"{test_arrow_table.num_rows:,}",
        f"{test_arrow_table.num_columns:,}",
        f"{loaded_test_origins:,}",
        f"{loaded_test_targets:,}",
        f"{test_arrow_size_gb:.2f}",
    ],
})

display(
    test_loading_summary.style
    .hide(axis="index")
)

In [ ]:
test_conversion_start = time.perf_counter()

# Converting the selected test columns to pandas.
test_frame = test_arrow_table.to_pandas(
    split_blocks=True,
)

# Creating the frozen test feature matrix.
X_test = test_frame[
    primary_features
].copy()

# Creating the final test target.
y_test = test_frame[
    "total_pickups"
].to_numpy(
    dtype=np.float32,
    copy=True,
)

# Preserving columns needed for residual analysis.
test_evaluation_columns = list(dict.fromkeys([
    "forecast_origin_utc",
    "target_hour_utc",
    "target_hour_local",
    "location_id",
    "forecast_horizon",
    "hour_of_day",
    "day_of_week",
    "month",
    "is_post_game_demand",
    "is_concert_window",
    "is_wide_concert_window",
    "is_ambiguous_dst_target",
    "weather_forecast_has_missing",
    "weather_forecast_complete",
    "total_pickups",
]))

test_evaluation = test_frame[
    test_evaluation_columns
].copy()

unseen_category_rows = {}

# Aligning test categories with the training categories.
for feature_name in categorical_features:
    assert isinstance(
        X_train[feature_name].dtype,
        pd.CategoricalDtype,
    )

    training_categories = (
        X_train[feature_name]
        .cat.categories
    )

    test_values = X_test[
        feature_name
    ]

    unseen_mask = (
        test_values.notna()
        & ~test_values.isin(training_categories)
    )

    unseen_category_rows[feature_name] = int(
        unseen_mask.sum()
    )

    X_test[feature_name] = pd.Categorical(
        test_values,
        categories=training_categories,
        ordered=(
            X_train[feature_name]
            .cat.ordered
        ),
    )

# Checking the frozen test structure.
assert len(X_test) == 3_436_884
assert len(y_test) == 3_436_884
assert X_test.shape[1] == 30
assert list(X_test.columns) == primary_features
assert np.isfinite(y_test).all()

assert all(
    count == 0
    for count in unseen_category_rows.values()
)

test_conversion_seconds = (
    time.perf_counter() - test_conversion_start
)

test_feature_audit = pd.DataFrame({
    "Measure": [
        "Test rows",
        "Primary features",
        "Missing feature values",
        "Unseen categorical values",
        "Minimum target",
        "Maximum target",
        "Conversion time (seconds)",
    ],
    "Value": [
        f"{len(X_test):,}",
        f"{X_test.shape[1]:,}",
        f"{int(X_test.isna().sum().sum()):,}",
        f"{sum(unseen_category_rows.values()):,}",
        f"{float(y_test.min()):,.0f}",
        f"{float(y_test.max()):,.0f}",
        f"{test_conversion_seconds:.2f}",
    ],
})

display(
    test_feature_audit.style
    .hide(axis="index")
)

display(
    pd.DataFrame({
        "Feature": list(
            unseen_category_rows.keys()
        ),
        "Unseen Test Rows": list(
            unseen_category_rows.values()
        ),
    }).style
    .hide(axis="index")
    .format({
        "Unseen Test Rows": "{:,}",
    })
)

In [ ]:
# Summarizing missing values by test feature.
test_missing_by_feature = (
    X_test.isna()
    .sum()
    .rename("Missing Values")
    .to_frame()
    .query("`Missing Values` > 0")
    .reset_index()
    .rename(columns={
        "index": "Feature",
    })
)

test_missing_by_feature[
    "Missing Rows (%)"
] = (
    test_missing_by_feature["Missing Values"]
    / len(X_test)
    * 100
)

# Identifying rows with at least one missing model feature.
test_any_feature_missing = (
    X_test.isna().any(axis=1)
)

# Comparing model-feature missingness with weather flags.
test_missing_flag_comparison = (
    pd.DataFrame({
        "Any Model Feature Missing": (
            test_any_feature_missing
        ),
        "Weather Missing Flag": (
            test_evaluation[
                "weather_forecast_has_missing"
            ].astype(bool)
        ),
        "Weather Complete Flag": (
            test_evaluation[
                "weather_forecast_complete"
            ].astype(bool)
        ),
    })
    .value_counts()
    .rename("Rows")
    .reset_index()
    .sort_values(
        [
            "Any Model Feature Missing",
            "Weather Missing Flag",
        ]
    )
)

# Checking that the earlier total is reproduced.
assert int(
    test_missing_by_feature[
        "Missing Values"
    ].sum()
) == 75_744

print(
    "Rows with any missing model feature:",
    f"{int(test_any_feature_missing.sum()):,}",
)

display(
    test_missing_by_feature.style
    .hide(axis="index")
    .format({
        "Missing Values": "{:,}",
        "Missing Rows (%)": "{:.4f}",
    })
)

display(
    test_missing_flag_comparison.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
    })
)

In [ ]:


# Checking categorical consistency before combining periods.
for feature_name in categorical_features:
    assert list(
        X_train[feature_name].cat.categories
    ) == list(
        X_validation[feature_name].cat.categories
    )

    assert list(
        X_train[feature_name].cat.categories
    ) == list(
        X_test[feature_name].cat.categories
    )

# Combining training and validation after all choices are frozen.
final_development_features = pd.concat(
    [
        X_train[primary_features],
        X_validation[primary_features],
    ],
    axis=0,
    ignore_index=True,
    copy=False,
)

final_development_target = np.concatenate(
    [
        y_train,
        y_validation,
    ]
).astype(
    np.float32,
    copy=False,
)

# Checking the combined development structure.
assert len(final_development_features) == 22_661_658
assert len(final_development_target) == 22_661_658
assert final_development_features.shape[1] == 30
assert list(
    final_development_features.columns
) == primary_features

# Building paired final matrices with development-derived cuts.
(
    final_development_matrix,
    final_test_matrix,
    final_matrix_seconds,
) = build_quantile_matrix_pair(
    training_features=(
        final_development_features
    ),
    training_target=(
        final_development_target
    ),
    evaluation_features=(
        X_test[primary_features]
    ),
    evaluation_target=y_test,
    max_bin=xgb_max_bin,
    enable_categorical=True,
)

# Validating the final matrices.
assert final_development_matrix.num_row() == 22_661_658
assert final_test_matrix.num_row() == 3_436_884
assert final_development_matrix.num_col() == 30
assert final_test_matrix.num_col() == 30

final_matrix_summary = pd.DataFrame({
    "Measure": [
        "Development rows",
        "Test rows",
        "Features",
        "Build time (seconds)",
    ],
    "Value": [
        f"{final_development_matrix.num_row():,}",
        f"{final_test_matrix.num_row():,}",
        final_development_matrix.num_col(),
        f"{final_matrix_seconds:.2f}",
    ],
})

display(
    final_matrix_summary.style
    .hide(axis="index")
)

In [ ]:
# Releasing temporary development and test data objects.
del final_development_features
del final_development_target
del X_test
del test_frame
del test_arrow_table
del production_feature_dataset

# Releasing original feature frames after final matrix creation.
del X_train
del X_validation

gc.collect()

# Confirming that final modeling and evaluation objects remain.
assert final_development_matrix.num_row() == 22_661_658
assert final_test_matrix.num_row() == 3_436_884
assert len(y_test) == 3_436_884
assert len(test_evaluation) == 3_436_884

process_ram_gb = (
    psutil.Process()
    .memory_info()
    .rss
    / 1024**3
)

print(
    "Final matrices preserved:",
    True,
)
print(
    "Test evaluation data preserved:",
    True,
)
print(
    "Notebook process RAM before final training (GB):",
    f"{process_ram_gb:.2f}",
)

In [ ]:
# Copying the frozen lambda-10 model parameters.
final_xgb_params = (
    concert_ablation_xgb_params.copy()
)

final_boosting_rounds = 1_613
# Training the frozen model on combined development data.
(
    final_xgb_model,
    final_xgb_history,
    final_training_seconds,
) = train_xgboost_candidate(
    parameters=final_xgb_params,
    training_matrix=final_development_matrix,
    evaluation_matrices=[
        (
            final_development_matrix,
            "development",
        ),
    ],
    num_boost_round=final_boosting_rounds,
    early_stopping_rounds=None,
    verbose_eval=100,
)

# Converting the reusable timer to minutes.
final_training_minutes = (
    final_training_seconds
    / 60
)

final_development_rmse = float(
    final_xgb_history[
        "development"
    ]["rmse"][-1]
)

# Checking that the requested number of rounds was completed.
assert final_xgb_model.num_boosted_rounds() == 1_613
assert len(
    final_xgb_history["development"]["rmse"]
) == 1_613

print()
print(
    "Completed boosting rounds:",
    f"{final_xgb_model.num_boosted_rounds():,}",
)
print(
    "Final development RMSE:",
    f"{final_development_rmse:.4f}",
)
print(
    "Final training time (minutes):",
    f"{final_training_minutes:.2f}",
)

In [ ]:
# Generating forecasts from every tree in the frozen model.
(
    final_test_predictions,
    final_test_predictions_constrained,
    final_test_negative_predictions,
    final_test_prediction_seconds,
) = predict_xgboost_demand(
    model=final_xgb_model,
    prediction_matrix=final_test_matrix,
    iteration_end=None,
)

# Validating the final prediction array.
assert len(
    final_test_predictions_constrained
) == len(y_test)

assert np.isfinite(
    final_test_predictions_constrained
).all()

assert (
    final_test_predictions_constrained >= 0
).all()

# Calculating untouched test metrics.
final_test_metrics = calculate_forecast_metrics(
    actual=y_test,
    prediction=final_test_predictions_constrained,
)

final_test_results = pd.DataFrame({
    "Model": [
        "Final XGBoost",
    ],
    "Rows": [
        len(y_test),
    ],
    "MAE": [
        final_test_metrics["mae"],
    ],
    "RMSE": [
        final_test_metrics["rmse"],
    ],
    "WAPE (%)": [
        final_test_metrics["wape_percent"],
    ],
    "Mean Bias": [
        final_test_metrics["mean_bias"],
    ],
    "R²": [
        final_test_metrics["r_squared"],
    ],
    "Negative Raw Predictions": [
        final_test_negative_predictions,
    ],
    "Prediction Seconds": [
        final_test_prediction_seconds,
    ],
})

display(
    final_test_results.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
        "Negative Raw Predictions": "{:,}",
        "Prediction Seconds": "{:.2f}",
    })
)

In [ ]:
# Confirming the weekly lag remains in the frozen feature set.
assert "pickup_lag_168" in primary_features

# Reconstructing a lightweight dataset for the test baseline.
baseline_feature_dataset = ds.dataset(
    [
        str(partition_path)
        for partition_path in final_partition_paths
    ],
    format="parquet",
)

test_baseline_columns = [
    "forecast_origin_utc",
    "target_hour_utc",
    "location_id",
    "pickup_lag_168",
]

test_baseline_arrow = baseline_feature_dataset.to_table(
    columns=test_baseline_columns,
    filter=(
        (
            ds.field("forecast_origin_utc")
            >= test_first_origin.to_pydatetime()
        )
        & (
            ds.field("forecast_origin_utc")
            <= test_last_origin.to_pydatetime()
        )
    ),
)

test_baseline_frame = (
    test_baseline_arrow.to_pandas(
        split_blocks=True,
    )
)

# Verifying exact row alignment before comparing forecasts.
alignment_columns = [
    "forecast_origin_utc",
    "target_hour_utc",
    "location_id",
]

assert test_baseline_frame[
    alignment_columns
].reset_index(
    drop=True
).equals(
    test_evaluation[
        alignment_columns
    ].reset_index(drop=True)
)

# Using demand from the same zone and hour one week earlier.
test_weekly_naive_predictions = (
    test_baseline_frame[
        "pickup_lag_168"
    ].to_numpy(
        dtype=np.float32,
        copy=True,
    )
)

assert len(
    test_weekly_naive_predictions
) == len(y_test)

assert np.isfinite(
    test_weekly_naive_predictions
).all()

assert (
    test_weekly_naive_predictions >= 0
).all()

# Calculating baseline metrics on identical test rows.
test_weekly_naive_metrics = (
    calculate_forecast_metrics(
        actual=y_test,
        prediction=test_weekly_naive_predictions,
    )
)

test_model_comparison = pd.DataFrame([
    {
        "Model": "Final XGBoost",
        "MAE": final_test_metrics["mae"],
        "RMSE": final_test_metrics["rmse"],
        "WAPE (%)": (
            final_test_metrics["wape_percent"]
        ),
        "Mean Bias": (
            final_test_metrics["mean_bias"]
        ),
        "R²": final_test_metrics["r_squared"],
    },
    {
        "Model": "Weekly seasonal naive",
        "MAE": test_weekly_naive_metrics["mae"],
        "RMSE": test_weekly_naive_metrics["rmse"],
        "WAPE (%)": (
            test_weekly_naive_metrics[
                "wape_percent"
            ]
        ),
        "Mean Bias": (
            test_weekly_naive_metrics["mean_bias"]
        ),
        "R²": (
            test_weekly_naive_metrics["r_squared"]
        ),
    },
])

display(
    test_model_comparison.style
    .hide(axis="index")
    .format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .highlight_min(
        subset=[
            "MAE",
            "RMSE",
            "WAPE (%)",
        ],
        color="#d9ead3",
    )
    .highlight_max(
        subset=["R²"],
        color="#d9ead3",
    )
)

In [ ]:
# Attaching final predictions to the test evaluation frame.
test_evaluation[
    "xgboost_prediction"
] = final_test_predictions_constrained

test_evaluation[
    "weekly_naive_prediction"
] = test_weekly_naive_predictions

# Calculating residuals as actual minus predicted.
test_evaluation[
    "xgboost_residual"
] = (
    y_test
    - final_test_predictions_constrained
)

test_evaluation[
    "weekly_naive_residual"
] = (
    y_test
    - test_weekly_naive_predictions
)

# Calculating absolute errors for later segment analysis.
test_evaluation[
    "xgboost_absolute_error"
] = np.abs(
    test_evaluation["xgboost_residual"]
)

test_evaluation[
    "weekly_naive_absolute_error"
] = np.abs(
    test_evaluation[
        "weekly_naive_residual"
    ]
)

# Confirming the residual sign convention.
assert np.isclose(
    test_evaluation[
        "xgboost_residual"
    ].mean(),
    -final_test_metrics["mean_bias"],
    atol=1e-4,
)

assert np.isclose(
    test_evaluation[
        "weekly_naive_residual"
    ].mean(),
    -test_weekly_naive_metrics["mean_bias"],
    atol=1e-4,
)

residual_summary_rows = []

# Summarizing the complete residual distributions.
for model_name, residual_column in {
    "Final XGBoost": "xgboost_residual",
    "Weekly seasonal naive": (
        "weekly_naive_residual"
    ),
}.items():
    residual_values = test_evaluation[
        residual_column
    ].to_numpy()

    residual_summary_rows.append({
        "Model": model_name,
        "Mean Residual": (
            residual_values.mean()
        ),
        "Median Residual": (
            np.median(residual_values)
        ),
        "Residual SD": (
            residual_values.std()
        ),
        "1st Percentile": (
            np.percentile(
                residual_values,
                1,
            )
        ),
        "99th Percentile": (
            np.percentile(
                residual_values,
                99,
            )
        ),
        "Minimum": residual_values.min(),
        "Maximum": residual_values.max(),
    })

test_residual_summary = pd.DataFrame(
    residual_summary_rows
)

display(
    test_residual_summary.style
    .hide(axis="index")
    .format({
        "Mean Residual": "{:+.3f}",
        "Median Residual": "{:+.3f}",
        "Residual SD": "{:.3f}",
        "1st Percentile": "{:.1f}",
        "99th Percentile": "{:.1f}",
        "Minimum": "{:.1f}",
        "Maximum": "{:.1f}",
    })
)

In [ ]:
# Calculating both model results with the reusable helper.
test_horizon_long_results = (
    calculate_metrics_by_group(
        evaluation_frame=test_evaluation,
        group_columns="forecast_horizon",
        prediction_columns={
            "Final XGBoost": "xgboost_prediction",
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
        },
    )
)

# Preparing the XGBoost horizon metrics.
test_horizon_xgboost = (
    test_horizon_long_results.loc[
        test_horizon_long_results["Model"]
        == "Final XGBoost",
        [
            "forecast_horizon",
            "Rows",
            "mae",
            "rmse",
            "wape_percent",
            "mean_bias",
        ],
    ]
    .rename(columns={
        "forecast_horizon": "Horizon",
        "mae": "XGBoost MAE",
        "rmse": "XGBoost RMSE",
        "wape_percent": "XGBoost WAPE (%)",
        "mean_bias": "XGBoost Mean Bias",
    })
)

# Preparing the baseline horizon metric needed for comparison.
test_horizon_baseline = (
    test_horizon_long_results.loc[
        test_horizon_long_results["Model"]
        == "Weekly Seasonal Naive",
        [
            "forecast_horizon",
            "rmse",
        ],
    ]
    .rename(columns={
        "forecast_horizon": "Horizon",
        "rmse": "Naive RMSE",
    })
)

# Joining the two model summaries by forecast horizon.
test_horizon_results = (
    test_horizon_xgboost
    .merge(
        test_horizon_baseline,
        on="Horizon",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("Horizon")
    .reset_index(drop=True)
)

# Calculating the relative RMSE improvement.
test_horizon_results[
    "RMSE Improvement (%)"
] = (
    (
        test_horizon_results["Naive RMSE"]
        - test_horizon_results["XGBoost RMSE"]
    )
    / test_horizon_results["Naive RMSE"]
    * 100
)

# Preserving the original table-column order.
test_horizon_results = test_horizon_results[
    [
        "Horizon",
        "Rows",
        "XGBoost MAE",
        "XGBoost RMSE",
        "XGBoost WAPE (%)",
        "XGBoost Mean Bias",
        "Naive RMSE",
        "RMSE Improvement (%)",
    ]
]

# Confirming complete and equal horizon coverage.
assert test_horizon_results["Horizon"].tolist() == [
    1, 2, 3, 4, 5, 6
]

assert test_horizon_results[
    "Rows"
].nunique() == 1

assert test_horizon_results[
    "Rows"
].sum() == len(test_evaluation)

display(
    test_horizon_results.style
    .hide(axis="index")
    .format({
        "Horizon": "H{}",
        "Rows": "{:,}",
        "XGBoost MAE": "{:.3f}",
        "XGBoost RMSE": "{:.3f}",
        "XGBoost WAPE (%)": "{:.3f}",
        "XGBoost Mean Bias": "{:+.3f}",
        "Naive RMSE": "{:.3f}",
        "RMSE Improvement (%)": "{:.1f}",
    })

    # Highlighting lower XGBoost errors more strongly.
    .background_gradient(
        subset=["XGBoost RMSE"],
        cmap="Blues_r",
    )

    # Emphasizing larger baseline errors in red.
    .background_gradient(
        subset=["Naive RMSE"],
        cmap="Reds",
    )

    # Highlighting larger XGBoost improvements in green.
    .background_gradient(
        subset=["RMSE Improvement (%)"],
        cmap="Greens",
    )

    # Making the winning results more prominent.
    .set_properties(
        subset=[
            "XGBoost RMSE",
            "RMSE Improvement (%)",
        ],
        **{
            "font-weight": "bold",
        },
    )
)

In [ ]:
# Aggregating repeated forecasts for each target zone-hour.
test_target_forecast_audit = (
    test_evaluation
    .groupby(
        [
            "target_hour_utc",
            "location_id",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        Forecasts=(
            "forecast_horizon",
            "size",
        ),
        Distinct_Actual_Values=(
            "total_pickups",
            "nunique",
        ),
        Prediction_Minimum=(
            "xgboost_prediction",
            "min",
        ),
        Prediction_Maximum=(
            "xgboost_prediction",
            "max",
        ),
    )
    .reset_index()
)

# Calculating how much forecasts change across origins.
test_target_forecast_audit[
    "Prediction Range"
] = (
    test_target_forecast_audit[
        "Prediction_Maximum"
    ]
    - test_target_forecast_audit[
        "Prediction_Minimum"
    ]
)

repeated_target_mask = (
    test_target_forecast_audit[
        "Forecasts"
    ] > 1
)

repeated_prediction_ranges = (
    test_target_forecast_audit.loc[
        repeated_target_mask,
        "Prediction Range",
    ].to_numpy()
)

# Checking that repeated rows share one actual target value.
assert (
    test_target_forecast_audit[
        "Distinct_Actual_Values"
    ] == 1
).all()

assert test_target_forecast_audit[
    "Forecasts"
].sum() == len(test_evaluation)

horizon_sensitivity_summary = pd.DataFrame({
    "Measure": [
        "Distinct target zone-hours",
        "Targets with multiple forecasts",
        "Minimum forecasts per target",
        "Median forecasts per target",
        "Maximum forecasts per target",
        "Exactly identical repeated forecasts (%)",
        "Median prediction range",
        "90th-percentile prediction range",
        "99th-percentile prediction range",
        "Maximum prediction range",
    ],
    "Value": [
        f"{len(test_target_forecast_audit):,}",
        f"{int(repeated_target_mask.sum()):,}",
        f"{int(test_target_forecast_audit['Forecasts'].min()):,}",
        f"{test_target_forecast_audit['Forecasts'].median():.0f}",
        f"{int(test_target_forecast_audit['Forecasts'].max()):,}",
        (
            f"{(
                np.mean(
                    repeated_prediction_ranges == 0
                ) * 100
            ):.3f}"
        ),
        f"{np.median(repeated_prediction_ranges):.3f}",
        (
            f"{np.percentile(
                repeated_prediction_ranges,
                90,
            ):.3f}"
        ),
        (
            f"{np.percentile(
                repeated_prediction_ranges,
                99,
            ):.3f}"
        ),
        f"{repeated_prediction_ranges.max():.3f}",
    ],
})

display(
    horizon_sensitivity_summary.style
    .hide(axis="index")
)

The predictions do change across origins for most repeated targets, but the typical revision is small. That fits the short horizon and relatively stable lag inputs. I'll also score unique targets to see whether the main finding depends heavily on counting several forecasts of the same outcome.


In [ ]:
# Collapsing repeated forecast issuances to one target zone-hour.
test_unique_target_results = (
    test_evaluation
    .groupby(
        [
            "target_hour_utc",
            "location_id",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        Actual_Demand=(
            "total_pickups",
            "first",
        ),
        XGBoost_Prediction=(
            "xgboost_prediction",
            "mean",
        ),
        Naive_Prediction=(
            "weekly_naive_prediction",
            "mean",
        ),
        Forecast_Issuances=(
            "forecast_horizon",
            "size",
        ),
    )
    .reset_index()
)

# Checking the unique-target structure.
assert len(
    test_unique_target_results
) == 574_129

assert (
    test_unique_target_results[
        "Forecast_Issuances"
    ].between(1, 6)
).all()

unique_target_actual = (
    test_unique_target_results[
        "Actual_Demand"
    ].to_numpy()
)

unique_target_xgboost = (
    test_unique_target_results[
        "XGBoost_Prediction"
    ].to_numpy()
)

unique_target_naive = (
    test_unique_target_results[
        "Naive_Prediction"
    ].to_numpy()
)

# Calculating metrics with one vote per target zone-hour.
unique_target_xgboost_metrics = (
    calculate_forecast_metrics(
        actual=unique_target_actual,
        prediction=unique_target_xgboost,
    )
)

unique_target_naive_metrics = (
    calculate_forecast_metrics(
        actual=unique_target_actual,
        prediction=unique_target_naive,
    )
)

unique_target_comparison = pd.DataFrame([
    {
        "Model": "Final XGBoost",
        "Target Zone-Hours": len(
            test_unique_target_results
        ),
        "MAE": (
            unique_target_xgboost_metrics["mae"]
        ),
        "RMSE": (
            unique_target_xgboost_metrics["rmse"]
        ),
        "WAPE (%)": (
            unique_target_xgboost_metrics[
                "wape_percent"
            ]
        ),
        "Mean Bias": (
            unique_target_xgboost_metrics[
                "mean_bias"
            ]
        ),
        "R²": (
            unique_target_xgboost_metrics[
                "r_squared"
            ]
        ),
    },
    {
        "Model": "Weekly seasonal naive",
        "Target Zone-Hours": len(
            test_unique_target_results
        ),
        "MAE": (
            unique_target_naive_metrics["mae"]
        ),
        "RMSE": (
            unique_target_naive_metrics["rmse"]
        ),
        "WAPE (%)": (
            unique_target_naive_metrics[
                "wape_percent"
            ]
        ),
        "Mean Bias": (
            unique_target_naive_metrics[
                "mean_bias"
            ]
        ),
        "R²": (
            unique_target_naive_metrics[
                "r_squared"
            ]
        ),
    },
])

display(
    unique_target_comparison.style
    .hide(axis="index")
    .format({
        "Target Zone-Hours": "{:,}",
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "WAPE (%)": "{:.4f}",
        "Mean Bias": "{:+.4f}",
        "R²": "{:.5f}",
    })
    .highlight_min(
        subset=[
            "MAE",
            "RMSE",
            "WAPE (%)",
        ],
        color="#68DDCC",
    )
    .highlight_max(
        subset=["R²"],
        color="#68DDCC",
    )
)

The unique-target scores are close to the issuance-level scores, and the advantage over the weekly baseline remains. That is reassuring for this particular aggregation check. The averaged prediction combines different origins, though, so it is a diagnostic rather than a forecast available at one issuance time.


In [ ]:
# Calculating both model results with the reusable helper.
test_month_long_results = (
    calculate_metrics_by_group(
        evaluation_frame=test_evaluation,
        group_columns="month",
        prediction_columns={
            "Final XGBoost": (
                "xgboost_prediction"
            ),
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
        },
    )
)

# Separating model results for the original reporting layout.
xgboost_month_results = (
    test_month_long_results
    .loc[
        test_month_long_results["Model"]
        == "Final XGBoost"
    ]
    .copy()
)

naive_month_results = (
    test_month_long_results
    .loc[
        test_month_long_results["Model"]
        == "Weekly Seasonal Naive"
    ]
    .copy()
)

# Calculating mean demand once per month.
monthly_actual_demand = (
    test_evaluation
    .groupby(
        "month",
        observed=True,
        sort=True,
    )["total_pickups"]
    .mean()
    .rename("Mean Actual Demand")
    .reset_index()
)

# Reconstructing the established monthly reporting table.
test_month_results = (
    xgboost_month_results[
        [
            "month",
            "Rows",
            "mae",
            "rmse",
            "wape_percent",
            "mean_bias",
        ]
    ]
    .rename(columns={
        "mae": "XGBoost MAE",
        "rmse": "XGBoost RMSE",
        "wape_percent": "XGBoost WAPE (%)",
        "mean_bias": "XGBoost Mean Bias",
    })
    .merge(
        naive_month_results[
            [
                "month",
                "rmse",
            ]
        ].rename(columns={
            "rmse": "Naive RMSE",
        }),
        on="month",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        monthly_actual_demand,
        on="month",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("month")
    .reset_index(drop=True)
)

# Calculating improvement against the fixed baseline.
test_month_results[
    "RMSE Improvement (%)"
] = (
    (
        test_month_results["Naive RMSE"]
        - test_month_results["XGBoost RMSE"]
    )
    / test_month_results["Naive RMSE"]
    * 100
)

# Creating readable year-month labels.
test_month_results["Month"] = (
    test_month_results["month"]
    .astype(int)
    .map(
        lambda month_value: pd.Timestamp(
            year=2026,
            month=month_value,
            day=1,
        ).strftime("%Y-%m")
    )
)

# Restoring the exact established column order.
test_month_results = test_month_results[
    [
        "Month",
        "Rows",
        "Mean Actual Demand",
        "XGBoost MAE",
        "XGBoost RMSE",
        "XGBoost WAPE (%)",
        "XGBoost Mean Bias",
        "Naive RMSE",
        "RMSE Improvement (%)",
    ]
]

# Verifying complete, mutually exclusive test coverage.
assert (
    xgboost_month_results["Rows"].sum()
    == len(test_evaluation)
)

assert (
    naive_month_results["Rows"].sum()
    == len(test_evaluation)
)

assert (
    test_month_results["Rows"].sum()
    == len(test_evaluation)
)

assert test_month_results["Month"].is_unique
assert np.isfinite(
    test_month_results.select_dtypes(
        include=np.number
    )
).all().all()

display(
    test_month_results.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "Mean Actual Demand": "{:.2f}",
        "XGBoost MAE": "{:.3f}",
        "XGBoost RMSE": "{:.3f}",
        "XGBoost WAPE (%)": "{:.3f}",
        "XGBoost Mean Bias": "{:+.3f}",
        "Naive RMSE": "{:.3f}",
        "RMSE Improvement (%)": "{:.1f}",
    })

    # Highlighting lower XGBoost errors more strongly.
    .background_gradient(
        subset=["XGBoost RMSE"],
        cmap="Blues_r",
    )

    # Emphasizing the larger baseline errors in red.
    .background_gradient(
        subset=["Naive RMSE"],
        cmap="Reds",
    )

    # Highlighting larger XGBoost improvements in green.
    .background_gradient(
        subset=["RMSE Improvement (%)"],
        cmap="Greens",
    )

    # Making the winning model and improvement more prominent.
    .set_properties(
        subset=[
            "XGBoost RMSE",
            "RMSE Improvement (%)",
        ],
        **{
            "font-weight": "bold",
        },
    )
)

The improvement holds in all three test months. June is harder than April, with higher RMSE and WAPE, so the pooled result should not be read as constant performance over time. This is a pattern the eventual monitoring will need to track.


In [ ]:
# Calculating both model results with the reusable helper.
test_hour_long_results = (
    calculate_metrics_by_group(
        evaluation_frame=test_evaluation,
        group_columns="hour_of_day",
        prediction_columns={
            "Final XGBoost": (
                "xgboost_prediction"
            ),
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
        },
    )
)

# Separating model results for the established reporting layout.
xgboost_hour_results = (
    test_hour_long_results
    .loc[
        test_hour_long_results["Model"]
        == "Final XGBoost"
    ]
    .copy()
)

naive_hour_results = (
    test_hour_long_results
    .loc[
        test_hour_long_results["Model"]
        == "Weekly Seasonal Naive"
    ]
    .copy()
)

# Calculating mean observed demand by local target hour.
hourly_actual_demand = (
    test_evaluation
    .groupby(
        "hour_of_day",
        observed=True,
        sort=True,
    )["total_pickups"]
    .mean()
    .rename("Mean Actual Demand")
    .reset_index()
)

# Reconstructing the established hourly reporting table.
test_hour_results = (
    xgboost_hour_results[
        [
            "hour_of_day",
            "Rows",
            "mae",
            "rmse",
            "wape_percent",
            "mean_bias",
        ]
    ]
    .rename(columns={
        "hour_of_day": "Hour",
        "mae": "XGBoost MAE",
        "rmse": "XGBoost RMSE",
        "wape_percent": "XGBoost WAPE (%)",
        "mean_bias": "XGBoost Mean Bias",
    })
    .merge(
        naive_hour_results[
            [
                "hour_of_day",
                "rmse",
            ]
        ].rename(columns={
            "hour_of_day": "Hour",
            "rmse": "Naive RMSE",
        }),
        on="Hour",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        hourly_actual_demand.rename(
            columns={
                "hour_of_day": "Hour",
            }
        ),
        on="Hour",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("Hour")
    .reset_index(drop=True)
)

# Standardizing the hour values for reporting and charting.
test_hour_results["Hour"] = (
    test_hour_results["Hour"]
    .astype(int)
)

# Calculating improvement against the fixed baseline.
test_hour_results[
    "RMSE Improvement (%)"
] = (
    (
        test_hour_results["Naive RMSE"]
        - test_hour_results["XGBoost RMSE"]
    )
    / test_hour_results["Naive RMSE"]
    * 100
)

# Restoring the exact established column order.
test_hour_results = test_hour_results[
    [
        "Hour",
        "Rows",
        "Mean Actual Demand",
        "XGBoost MAE",
        "XGBoost RMSE",
        "XGBoost WAPE (%)",
        "XGBoost Mean Bias",
        "Naive RMSE",
        "RMSE Improvement (%)",
    ]
]

# Checking complete and mutually exclusive test coverage.
assert len(test_hour_results) == 24
assert set(test_hour_results["Hour"]) == set(
    range(24)
)

assert (
    xgboost_hour_results["Rows"].sum()
    == len(test_evaluation)
)

assert (
    naive_hour_results["Rows"].sum()
    == len(test_evaluation)
)

assert (
    test_hour_results["Rows"].sum()
    == len(test_evaluation)
)

assert np.isfinite(
    test_hour_results.select_dtypes(
        include=np.number
    )
).all().all()

display(
    test_hour_results.style
    .hide(axis="index")
    .format({
        "Hour": "{:02d}:00",
        "Rows": "{:,}",
        "Mean Actual Demand": "{:.2f}",
        "XGBoost MAE": "{:.3f}",
        "XGBoost RMSE": "{:.3f}",
        "XGBoost WAPE (%)": "{:.3f}",
        "XGBoost Mean Bias": "{:+.3f}",
        "Naive RMSE": "{:.3f}",
        "RMSE Improvement (%)": "{:.1f}",
    })
    .background_gradient(
        subset=["XGBoost RMSE"],
        cmap="YlOrRd",
    )
)

# Preparing readable hourly labels.
hourly_chart_data = (
    test_hour_results.copy()
)

hourly_chart_data[
    "Hour Label"
] = hourly_chart_data[
    "Hour"
].map(
    lambda value: f"{value:02d}:00"
)

# Defining restrained model colors.
xgboost_color = "#3F6FB5"
baseline_color = "#9AA0A6"
bias_color = "#D18463"

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 10),
    sharex=True,
    gridspec_kw={
        "height_ratios": [1.35, 1],
        "hspace": 0.18,
    },
)

# Comparing hourly RMSE for both models.
sns.lineplot(
    data=hourly_chart_data,
    x="Hour",
    y="XGBoost RMSE",
    marker="o",
    linewidth=2.4,
    markersize=6,
    color=xgboost_color,
    label="Final XGBoost",
    ax=axes[0],
)

sns.lineplot(
    data=hourly_chart_data,
    x="Hour",
    y="Naive RMSE",
    marker="o",
    linewidth=2.2,
    markersize=5,
    color=baseline_color,
    label="Weekly seasonal naive",
    ax=axes[0],
)

axes[0].set_title(
    "XGBoost Reduces Error at Every Hour",
    loc="left",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

axes[0].set_ylabel(
    "RMSE",
    fontsize=12,
)

axes[0].set_xlabel("")

axes[0].legend(
    title="Model",
    frameon=False,
    ncol=2,
    loc="upper center",
)

axes[0].grid(
    axis="y",
    color="#D9DEE7",
    linewidth=0.8,
    alpha=0.8,
)

axes[0].grid(
    axis="x",
    visible=False,
)

# Displaying hourly XGBoost bias separately.
sns.barplot(
    data=hourly_chart_data,
    x="Hour",
    y="XGBoost Mean Bias",
    color=bias_color,
    edgecolor="white",
    linewidth=0.6,
    ax=axes[1],
)

axes[1].axhline(
    0,
    color="#3F4650",
    linestyle="--",
    linewidth=1.2,
)

axes[1].set_title(
    "Underprediction Is Strongest During Morning and Evening Peaks",
    loc="left",
    fontsize=16,
    fontweight="bold",
    pad=14,
)

axes[1].set_ylabel(
    "Mean bias\n(prediction − actual)",
    fontsize=12,
)

axes[1].set_xlabel(
    "Local target hour",
    fontsize=12,
    labelpad=10,
)

axes[1].set_ylim(
    -4.2,
    0.5,
)

axes[1].grid(
    axis="y",
    color="#D9DEE7",
    linewidth=0.8,
    alpha=0.8,
)

axes[1].grid(
    axis="x",
    visible=False,
)

# Adding bias values without crowding the RMSE panel.
for bar, bias_value in zip(
    axes[1].patches,
    hourly_chart_data[
        "XGBoost Mean Bias"
    ],
):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bias_value - 0.10,
        f"{bias_value:.1f}",
        ha="center",
        va="top",
        fontsize=11,
        color="#343A40",
    )

# Applying consistent hourly tick labels.
axes[1].set_xticks(
    range(24)
)

axes[1].set_xticklabels(
    hourly_chart_data[
        "Hour Label"
    ],
    rotation=45,
    ha="right",
)

# Adding the main analytical question and context.
fig.suptitle(
    "When Does the Final Model Make Its Largest Test Errors?",
    x=0.07,
    y=0.995,
    ha="left",
    fontsize=21,
    fontweight="bold",
)

fig.text(
    0.07,
    0.948,
    (
        "Untouched April–June 2026 test period; "
        "all 263 zones and forecast horizons 1–6."
    ),
    ha="left",
    fontsize=13.5,
    color="#596273",
)

sns.despine(
    fig=fig,
    left=False,
    bottom=False,
)

plt.subplots_adjust(
    top=0.88,
    left=0.08,
    right=0.98,
    bottom=0.10,
)

plt.show()

### Better at every hour, but still too low

The model beats the weekly baseline at every hour, but it still underpredicts throughout the day. The largest pickup-unit errors appear around midnight and late evening. I'll keep that distinction clear: improving on the baseline does not mean the busy hours are solved.


In [ ]:
# Preparing hourly residuals by weekday and weekend.
hour_day_residual_chart_data = (
    test_evaluation
    .assign(
        Day_Type=np.where(
            test_evaluation[
                "day_of_week"
            ].astype(int) <= 4,
            "Weekday",
            "Weekend",
        )
    )
    .groupby(
        [
            "Day_Type",
            "hour_of_day",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        Mean_Residual=(
            "xgboost_residual",
            "mean",
        )
    )
)

hour_day_residual_chart_data[
    "hour_of_day"
] = (
    hour_day_residual_chart_data[
        "hour_of_day"
    ].astype(int)
)

# Using the same vertical scale for a fair comparison.
residual_limit = (
    hour_day_residual_chart_data[
        "Mean_Residual"
    ].abs().max()
    + 0.8
)

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    gridspec_kw={
        "hspace": 0.24,
    },
)

chart_settings = [
    (
        "Weekday",
        "#3F6FB5",
        "Weekday Residuals Show Morning and Evening Demand Peaks",
    ),
    (
        "Weekend",
        "#D18463",
        "Weekend Residuals Follow a Different Late-Day Pattern",
    ),
]

# Creating separate weekday and weekend charts.
for ax, (
    day_type,
    chart_color,
    chart_title,
) in zip(
    axes,
    chart_settings,
):
    day_chart_data = (
        hour_day_residual_chart_data[
            hour_day_residual_chart_data[
                "Day_Type"
            ].eq(day_type)
        ]
        .sort_values("hour_of_day")
    )

    sns.barplot(
        data=day_chart_data,
        x="hour_of_day",
        y="Mean_Residual",
        color=chart_color,
        edgecolor="white",
        linewidth=0.6,
        ax=ax,
    )

    ax.axhline(
        0,
        color="#3F4650",
        linestyle="--",
        linewidth=1.2,
    )

    ax.set_title(
        chart_title,
        loc="left",
        fontsize=16,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Mean residual\n(actual − prediction)",
        fontsize=11.5,
    )

    ax.set_xlabel("")

    ax.set_ylim(
        -1,
        residual_limit,
    )

    ax.grid(
        axis="y",
        # color="#D9DEE7",
        # linewidth=0.8,
        # alpha=0.8,
        visible=False
    )

    ax.grid(
        axis="x",
        visible=False,
    )

    # Adding residual values to each hourly bar.
    for bar, residual_value in zip(
        ax.patches,
        day_chart_data[
            "Mean_Residual"
        ],
    ):
        vertical_offset = (
            0.12
            if residual_value >= 0
            else -0.12
        )

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            residual_value
            + vertical_offset,
            f"{residual_value:+.1f}",
            ha="center",
            va=(
                "bottom"
                if residual_value >= 0
                else "top"
            ),
            fontsize=10,
            color="#343A40",
        )

# Formatting the shared hourly axis.
axes[1].set_xlabel(
    "Local target hour",
    fontsize=12,
    labelpad=10,
)

axes[1].set_xticks(
    range(24)
)

axes[1].set_xticklabels(
    [
        f"{hour:02d}:00"
        for hour in range(24)
    ],
    rotation=45,
    ha="right",
)

fig.suptitle(
    "How Does Forecast Bias Change Between Weekdays and Weekends?",
    x=0.07,
    y=0.995,
    ha="left",
    fontsize=21,
    fontweight="bold",
)

fig.text(
    0.07,
    0.958,
    (
        "Positive residuals indicate underprediction; "
        "negative residuals indicate overprediction."
    ),
    ha="left",
    fontsize=11.5,
    color="#596273",
)

sns.despine(
    fig=fig,
    left=False,
    bottom=False,
)

plt.subplots_adjust(
    top=0.88,
    left=0.08,
    right=0.98,
    bottom=0.10,
)

plt.show()

### Weekday and weekend residual interpretation

Weekend underprediction is stronger across several afternoon and evening hours. Weekdays show a clearer commuting pattern. A single overall bias would hide those differences, which is why the hourly split is useful.


### Why are the busiest hours still too low?

The pattern makes me suspect that unusual high activity is being pulled toward more typical levels. That is one possible explanation, not something the chart proves. I'll compare residuals by actual and predicted pickups to understand it better.


In [ ]:
# Defining interpretable actual-demand ranges.
demand_bin_edges = [
    -0.5,
    0.5,
    25.5,
    50.5,
    100.5,
    200.5,
    400.5,
    np.inf,
]

demand_bin_labels = [
    "0",
    "1–25",
    "26–50",
    "51–100",
    "101–200",
    "201–400",
    "401+",
]

# Assigning test rows to realized-demand ranges.
test_evaluation[
    "Actual Demand Range"
] = pd.cut(
    test_evaluation[
        "total_pickups"
    ],
    bins=demand_bin_edges,
    labels=demand_bin_labels,
    ordered=True,
)

# Summarizing residuals across demand levels.
demand_residual_chart_data = (
    test_evaluation
    .groupby(
        "Actual Demand Range",
        observed=True,
        as_index=False,
    )
    .agg(
        Rows=(
            "xgboost_residual",
            "size",
        ),
        Mean_Actual_Demand=(
            "total_pickups",
            "mean",
        ),
        Mean_Residual=(
            "xgboost_residual",
            "mean",
        ),
    )
)

fig, ax = plt.subplots(
    figsize=(13.5, 6.5)
)

# Charting whether underprediction increases with demand.
sns.barplot(
    data=demand_residual_chart_data,
    x="Actual Demand Range",
    y="Mean_Residual",
    hue="Actual Demand Range",
    palette="Blues",
    legend=False,
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

ax.axhline(
    0,
    color="#3F4650",
    linestyle="--",
    linewidth=1.2,
)

ax.set_title(
    "Does Underprediction Increase When Actual Demand Is Higher?",
    loc="left",
    fontsize=19,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.025,
    (
        "Positive residuals indicate actual demand exceeded "
        "the forecast; bins use realized test demand."
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=11.5,
    color="#596273",
)

ax.set_xlabel(
    "Actual hourly pickups per zone",
    fontsize=12,
    labelpad=10,
)

ax.set_ylabel(
    "Mean residual\n(actual − prediction)",
    fontsize=12,
)

ax.grid(
    axis="y",
    # color="#D9DEE7",
    # linewidth=0.8,
    # alpha=0.8,
    visible=False
)

ax.grid(
    axis="x",
    visible=False,
)

# Adding residual values and sample sizes.
for bar, row in zip(
    ax.patches,
    demand_residual_chart_data.itertuples(
        index=False
    ),
):
    residual_value = row.Mean_Residual

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        residual_value
        + (
            1.2
            if residual_value >= 0
            else -1.2
        ),
        f"{residual_value:+.1f}",
        ha="center",
        va=(
            "bottom"
            if residual_value >= 0
            else "top"
        ),
        fontsize=10,
        fontweight="bold",
        color="#26313D",
    )

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        0.02,
        f"n={row.Rows:,.0f}",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=8,
        color="#FFFFFF",
    )

sns.despine(
    ax=ax,
    left=False,
    bottom=False,
)

plt.tight_layout()
plt.show()

### How does error change as actual pickups rise?

Errors become more negative in the groups with high actual pickups. That is worth examining, but these groups were chosen after seeing the outcomes. I'll also group by predicted counts, which gives a more direct view of what a user would know before the hour occurs.


In [ ]:
# Defining forecast-demand ranges.
calibration_bin_edges = [
    -0.001,
    0.5,
    25.5,
    50.5,
    100.5,
    200.5,
    400.5,
    np.inf,
]

calibration_bin_labels = [
    "0",
    "1–25",
    "26–50",
    "51–100",
    "101–200",
    "201–400",
    "401+",
]

# Grouping rows by the model's predicted demand.
calibration_prediction_ranges = pd.cut(
    test_evaluation[
        "xgboost_prediction"
    ],
    bins=calibration_bin_edges,
    labels=calibration_bin_labels,
    ordered=True,
    include_lowest=True,
)

# Calculating calibration from the complete test data.
test_calibration_data = (
    test_evaluation
    .assign(
        Predicted_Demand_Range=(
            calibration_prediction_ranges
        )
    )
    .groupby(
        "Predicted_Demand_Range",
        observed=True,
        as_index=False,
    )
    .agg(
        Rows=(
            "xgboost_prediction",
            "size",
        ),
        Mean_Predicted_Demand=(
            "xgboost_prediction",
            "mean",
        ),
        Mean_Actual_Demand=(
            "total_pickups",
            "mean",
        ),
    )
)

test_calibration_data[
    "Calibration_Gap"
] = (
    test_calibration_data[
        "Mean_Actual_Demand"
    ]
    - test_calibration_data[
        "Mean_Predicted_Demand"
    ]
)

# Adding sample sizes directly to the category labels.
test_calibration_data[
    "Forecast_Range_Label"
] = [
    (
        f"{demand_range}\n"
        f"n={row_count:,}"
    )
    for demand_range, row_count in zip(
        test_calibration_data[
            "Predicted_Demand_Range"
        ].astype(str),
        test_calibration_data[
            "Rows"
        ],
    )
]

fig, ax = plt.subplots(
    figsize=(13.5, 6.5)
)

# Displaying calibration gaps across forecast levels.
sns.barplot(
    data=test_calibration_data,
    x="Forecast_Range_Label",
    y="Calibration_Gap",
    hue="Forecast_Range_Label",
    palette="Blues",
    legend=False,
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

ax.axhline(
    0,
    color="#3F4650",
    linestyle="--",
    linewidth=1.2,
)

ax.set_title(
    "How Does Calibration Change Across Forecast Demand Levels?",
    loc="left",
    fontsize=19,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.025,
    (
        "Calibration gap = mean actual demand − mean predicted demand; "
        "positive values indicate underprediction."
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=13.5,
    color="#596273",
)

ax.set_xlabel(
    "Predicted hourly pickups per zone",
    fontsize=12,
    labelpad=12,
)

ax.set_ylabel(
    "Calibration gap\n(actual − prediction)",
    fontsize=12,
)

ax.set_ylim(
    -1,
    (
        test_calibration_data[
            "Calibration_Gap"
        ].max()
        + 2
    ),
)

ax.grid(
    axis="y",
    # color="#D9DEE7",
    # linewidth=0.8,
    # alpha=0.8,
    visible=False
)

ax.grid(
    axis="x",
    visible=False,
)

# Adding calibration values above the bars.
for bar, gap_value in zip(
    ax.patches,
    test_calibration_data[
        "Calibration_Gap"
    ],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        gap_value + 0.20,
        f"{gap_value:+.1f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="#26313D",
    )

sns.despine(
    ax=ax,
    left=False,
    bottom=False,
)

plt.tight_layout()
plt.show()

### Do forecast counts line up with observed counts?

The predicted-count groups also show conservative forecasts at the high end. That gives me a reason to investigate calibration, rather than rely only on the actual-count plot. Any correction should be fitted on development or new calibration data, not on these same test outcomes.


In [ ]:
# Creating a reproducible sample for the residual cloud.
residual_plot_sample = (
    test_evaluation[
        [
            "xgboost_prediction",
            "xgboost_residual",
        ]
    ]
    .sample(
        n=100_000,
        random_state=42,
    )
)

# Calculating a stable residual trend from the full test set.
test_evaluation[
    "Prediction Quantile"
] = pd.qcut(
    test_evaluation[
        "xgboost_prediction"
    ],
    q=30,
    duplicates="drop",
)

residual_trend_data = (
    test_evaluation
    .groupby(
        "Prediction Quantile",
        observed=True,
        as_index=False,
    )
    .agg(
        Mean_Prediction=(
            "xgboost_prediction",
            "mean",
        ),
        Mean_Residual=(
            "xgboost_residual",
            "mean",
        ),
    )
)

# Limiting only the displayed range to prevent extremes
# from compressing the central residual pattern.
prediction_display_maximum = np.percentile(
    test_evaluation[
        "xgboost_prediction"
    ],
    99.5,
)

residual_display_limits = np.percentile(
    test_evaluation[
        "xgboost_residual"
    ],
    [
        1,
        99,
    ],
)

fig, ax = plt.subplots(
    figsize=(13.5, 8)
)

# Displaying the sampled residual cloud.
sns.scatterplot(
    data=residual_plot_sample,
    x="xgboost_prediction",
    y="xgboost_residual",
    color="#3A78AE",
    alpha=0.25,
    s=10,
    linewidth=0,
    label="Sampled test residuals",
    ax=ax,
)

# Displaying the full-data binned residual trend.
sns.lineplot(
    data=residual_trend_data,
    x="Mean_Prediction",
    y="Mean_Residual",
    color="#D05A4E",
    marker="o",
    markersize=5,
    linewidth=2.6,
    label="Mean residual trend",
    ax=ax,
)

ax.axhline(
    0,
    color="#26313D",
    linestyle="--",
    linewidth=1.3,
)

ax.set_title(
    "Do Residuals Change as Predicted Demand Increases?",
    loc="left",
    fontsize=19,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.025,
    (
        "The cloud uses a reproducible 100,000-row sample; "
        "the red trend uses all test rows."
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=13.5,
    color="#7A89A6",
)

ax.set_xlabel(
    "Predicted hourly pickups per zone",
    fontsize=12,
    labelpad=10,
)

ax.set_ylabel(
    "Residual\n(actual − prediction)",
    fontsize=12,
)

ax.set_xlim(
    0,
    prediction_display_maximum,
)

ax.set_ylim(
    residual_display_limits[0],
    residual_display_limits[1],
)

ax.grid(
    # color="#D9DEE7",
    # linewidth=0.8,
    # alpha=0.7,
    visible=False
)

ax.legend(
    frameon=False,
    loc='upper right',
)

sns.despine(
    ax=ax,
    left=False,
    bottom=False,
)

plt.tight_layout()
plt.show()

### Where are the largest misses by forecast size?

The residual cloud spreads out as predicted pickups increase. A forecast of 500 pickups has more room for a large miss than a forecast of 20. This suggests that useful uncertainty intervals should vary with the forecast level rather than add the same fixed band everywhere.


In [ ]:
# Checking that weather-availability flags form a valid partition.
weather_complete_mask = (
    test_evaluation[
        "weather_forecast_complete"
    ].astype(bool)
)

weather_missing_mask = (
    test_evaluation[
        "weather_forecast_has_missing"
    ].astype(bool)
)

assert not (
    weather_complete_mask
    & weather_missing_mask
).any()

assert (
    weather_complete_mask
    | weather_missing_mask
).all()

assert int(
    weather_missing_mask.sum()
) == 28_404

# Preparing the production-availability evaluation frame.
weather_metric_frame = (
    test_evaluation[
        [
            "total_pickups",
            "xgboost_prediction",
            "weekly_naive_prediction",
        ]
    ]
    .copy()
)

weather_metric_frame[
    "Weather Forecast Availability"
] = np.where(
    weather_complete_mask,
    "Complete archived weather",
    "Incomplete archived weather",
)

# Preserving the established group order.
weather_availability_order = [
    "Complete archived weather",
    "Incomplete archived weather",
]

weather_metric_frame[
    "Weather Forecast Availability"
] = pd.Categorical(
    weather_metric_frame[
        "Weather Forecast Availability"
    ],
    categories=weather_availability_order,
    ordered=True,
)

# Calculating both model results with the reusable helper.
weather_availability_results = (
    calculate_metrics_by_group(
        evaluation_frame=weather_metric_frame,
        group_columns=(
            "Weather Forecast Availability"
        ),
        prediction_columns={
            "Final XGBoost": (
                "xgboost_prediction"
            ),
            "Weekly seasonal naive": (
                "weekly_naive_prediction"
            ),
        },
    )
)

# Calculating mean observed demand once per availability group.
weather_mean_actual = (
    weather_metric_frame
    .groupby(
        "Weather Forecast Availability",
        observed=True,
        sort=True,
    )["total_pickups"]
    .mean()
    .rename("Mean Actual Demand")
    .reset_index()
)

# Restoring the established reporting schema.
weather_availability_results = (
    weather_availability_results
    .merge(
        weather_mean_actual,
        on="Weather Forecast Availability",
        how="left",
        validate="many_to_one",
    )
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
    })
    .sort_values(
        [
            "Weather Forecast Availability",
            "Model",
        ],
        key=lambda values: (
            values.map({
                "Final XGBoost": 0,
                "Weekly seasonal naive": 1,
            })
            if values.name == "Model"
            else values
        ),
    )
    .reset_index(drop=True)
)

weather_availability_results = (
    weather_availability_results[
        [
            "Weather Forecast Availability",
            "Model",
            "Rows",
            "Mean Actual Demand",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
)

# Confirming complete coverage for each model.
weather_row_totals = (
    weather_availability_results
    .groupby(
        "Model",
        observed=True,
    )["Rows"]
    .sum()
)

assert len(
    weather_availability_results
) == 4

assert (
    weather_row_totals
    == len(test_evaluation)
).all()

assert np.isfinite(
    weather_availability_results[
        [
            "Mean Actual Demand",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
        ]
    ]
).all().all()

display(
    weather_availability_results.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "Mean Actual Demand": "{:.2f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
    })
)

In [ ]:
incomplete_weather_rows = (
    test_evaluation.loc[
        test_evaluation[
            "weather_forecast_has_missing"
        ].astype(bool)
    ].copy()
)

# Summarizing incomplete weather by forecast origin.
incomplete_weather_origin_summary = (
    incomplete_weather_rows
    .groupby(
        "forecast_origin_utc",
        observed=True,
        as_index=False,
    )
    .agg(
        Rows=(
            "location_id",
            "size",
        ),
        Distinct_Zones=(
            "location_id",
            "nunique",
        ),
        Distinct_Horizons=(
            "forecast_horizon",
            "nunique",
        ),
        First_Target=(
            "target_hour_utc",
            "min",
        ),
        Last_Target=(
            "target_hour_utc",
            "max",
        ),
    )
)

# Auditing how concentrated the missing forecasts are.
incomplete_weather_audit = pd.DataFrame({
    "Measure": [
        "Incomplete forecast rows",
        "Affected forecast origins",
        "First affected origin",
        "Last affected origin",
        "Minimum rows per origin",
        "Maximum rows per origin",
        "Expected full-origin rows",
    ],
    "Value": [
        f"{len(incomplete_weather_rows):,}",
        (
            f"{incomplete_weather_origin_summary[
                'forecast_origin_utc'
            ].nunique():,}"
        ),
        (
            incomplete_weather_origin_summary[
                "forecast_origin_utc"
            ].min()
        ),
        (
            incomplete_weather_origin_summary[
                "forecast_origin_utc"
            ].max()
        ),
        (
            f"{incomplete_weather_origin_summary[
                'Rows'
            ].min():,}"
        ),
        (
            f"{incomplete_weather_origin_summary[
                'Rows'
            ].max():,}"
        ),
        "1,578",
    ],
})

assert incomplete_weather_origin_summary[
    "Rows"
].sum() == 28_404

display(
    incomplete_weather_audit.style
    .hide(axis="index")
)

display(
    incomplete_weather_origin_summary.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "Distinct_Zones": "{:,}",
        "Distinct_Horizons": "{:,}",
    })
)

### Archived-weather availability during the test period

The weather gaps affect many rows but only 18 origins on two June dates. That is a much smaller number of independent data problems than the row count suggests. I'll report the subgroup, but I would want more outage periods before judging resilience from it.


Next I'll look at the geography. A good pooled forecast can still miss badly in a few locations, and absolute errors may tell a different story from percentages.


In [ ]:
# Defining the taxi-zone description path.
TAXI_ZONE_LOOKUP_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "taxi_zone_lookup.csv"
)

assert TAXI_ZONE_LOOKUP_PATH.exists()

# Loading the taxi-zone descriptions.
taxi_zone_lookup = pd.read_csv(
    TAXI_ZONE_LOOKUP_PATH
)

print(
    "Lookup rows:",
    f"{len(taxi_zone_lookup):,}",
)
print(
    "Lookup columns:",
    list(taxi_zone_lookup.columns),
)

display(
    taxi_zone_lookup.head()
)

In [ ]:
# Validating the taxi-zone lookup.
assert taxi_zone_lookup[
    "LocationID"
].is_unique

modeled_location_ids = set(
    test_evaluation[
        "location_id"
    ].astype(int).unique()
)

lookup_location_ids = set(
    taxi_zone_lookup[
        "LocationID"
    ].astype(int)
)

missing_zone_descriptions = sorted(
    modeled_location_ids
    - lookup_location_ids
)

assert not missing_zone_descriptions
assert len(modeled_location_ids) == 263

# Preparing squared errors for zone-level RMSE.
test_evaluation[
    "xgboost_squared_error"
] = (
    test_evaluation[
        "xgboost_residual"
    ] ** 2
)

test_evaluation[
    "weekly_naive_squared_error"
] = (
    test_evaluation[
        "weekly_naive_residual"
    ] ** 2
)

# Aggregating error components by taxi zone.
zone_residual_results = (
    test_evaluation
    .groupby(
        "location_id",
        observed=True,
        as_index=False,
    )
    .agg(
        Rows=(
            "total_pickups",
            "size",
        ),
        Mean_Actual_Demand=(
            "total_pickups",
            "mean",
        ),
        Actual_Demand_Total=(
            "total_pickups",
            "sum",
        ),
        XGBoost_Absolute_Error_Total=(
            "xgboost_absolute_error",
            "sum",
        ),
        XGBoost_MAE=(
            "xgboost_absolute_error",
            "mean",
        ),
        XGBoost_MSE=(
            "xgboost_squared_error",
            "mean",
        ),
        XGBoost_Mean_Residual=(
            "xgboost_residual",
            "mean",
        ),
        Naive_MSE=(
            "weekly_naive_squared_error",
            "mean",
        ),
    )
)

# Converting aggregated components into final metrics.
zone_residual_results[
    "XGBoost_RMSE"
] = np.sqrt(
    zone_residual_results[
        "XGBoost_MSE"
    ]
)

zone_residual_results[
    "XGBoost_WAPE_Percent"
] = (
    zone_residual_results[
        "XGBoost_Absolute_Error_Total"
    ]
    / zone_residual_results[
        "Actual_Demand_Total"
    ]
    * 100
)

zone_residual_results[
    "XGBoost_Mean_Bias"
] = (
    -zone_residual_results[
        "XGBoost_Mean_Residual"
    ]
)

zone_residual_results[
    "Naive_RMSE"
] = np.sqrt(
    zone_residual_results[
        "Naive_MSE"
    ]
)

zone_residual_results[
    "RMSE_Improvement_Percent"
] = (
    (
        zone_residual_results[
            "Naive_RMSE"
        ]
        - zone_residual_results[
            "XGBoost_RMSE"
        ]
    )
    / zone_residual_results[
        "Naive_RMSE"
    ]
    * 100
)

# Adding readable zone and borough descriptions.
zone_residual_results = (
    zone_residual_results
    .merge(
        taxi_zone_lookup[
            [
                "LocationID",
                "Borough",
                "Zone",
                "service_zone",
            ]
        ],
        left_on="location_id",
        right_on="LocationID",
        how="left",
        validate="one_to_one",
    )
)

zone_residual_results[
    "Zone Label"
] = (
    zone_residual_results["Zone"]
    + " — "
    + zone_residual_results["Borough"]
)

assert zone_residual_results[
    "Zone"
].notna().all()

assert len(
    zone_residual_results
) == 263

# Displaying zones with the largest absolute forecast errors.
worst_zone_rmse_results = (
    zone_residual_results
    .nlargest(
        15,
        "XGBoost_RMSE",
    )
    [
        [
            "location_id",
            "Zone",
            "Borough",
            "Mean_Actual_Demand",
            "XGBoost_MAE",
            "XGBoost_RMSE",
            "XGBoost_WAPE_Percent",
            "XGBoost_Mean_Bias",
            "Naive_RMSE",
            "RMSE_Improvement_Percent",
        ]
    ]
)

display(
    worst_zone_rmse_results.style
    .hide(axis="index")
    .format({
        "location_id": "{:.0f}",
        "Mean_Actual_Demand": "{:.2f}",
        "XGBoost_MAE": "{:.3f}",
        "XGBoost_RMSE": "{:.3f}",
        "XGBoost_WAPE_Percent": "{:.3f}",
        "XGBoost_Mean_Bias": "{:+.3f}",
        "Naive_RMSE": "{:.3f}",
        "RMSE_Improvement_Percent": "{:.1f}",
    })

    # Highlighting lower XGBoost errors more strongly.
    .background_gradient(
        subset=["XGBoost_RMSE"],
        cmap="Blues_r",
    )

    # Emphasizing larger baseline errors in red.
    .background_gradient(
        subset=["Naive_RMSE"],
        cmap="Reds",
    )

    # Highlighting larger XGBoost improvements in green.
    .background_gradient(
        subset=["RMSE_Improvement_Percent"],
        cmap="Greens",
    )

    # Making the winning results more prominent.
    .set_properties(
        subset=[
            "XGBoost_RMSE",
            "RMSE_Improvement_Percent",
        ],
        **{
            "font-weight": "bold",
        },
    )
)

# Preparing the 15 zones with the largest XGBoost RMSE.
zone_rmse_chart_data = (
    worst_zone_rmse_results
    .copy()
    .sort_values(
        "XGBoost_RMSE",
        ascending=True,
    )
)

# Confirming that chart metrics are complete.
chart_metric_columns = [
    "XGBoost_RMSE",
    "Naive_RMSE",
    "RMSE_Improvement_Percent",
]

assert zone_rmse_chart_data[
    chart_metric_columns
].notna().all().all(), (
    "One or more zone metrics are missing. "
    "Inspect zone_rmse_chart_data before plotting."
)

zone_labels = (
    zone_rmse_chart_data["Zone"]
    + " — "
    + zone_rmse_chart_data["Borough"]
)

y_positions = np.arange(
    len(zone_rmse_chart_data)
)

bar_height = 0.36

fig, ax = plt.subplots(
    figsize=(15, 10)
)

ax.barh(
    y_positions - bar_height / 2,
    zone_rmse_chart_data["XGBoost_RMSE"],
    height=bar_height,
    color="#3F73B9",
    label="Final XGBoost",
)

ax.barh(
    y_positions + bar_height / 2,
    zone_rmse_chart_data["Naive_RMSE"],
    height=bar_height,
    color="#A3A9AF",
    label="Weekly seasonal naive",
)

# Labeling each RMSE value.
for position, xgb_rmse, naive_rmse in zip(
    y_positions,
    zone_rmse_chart_data["XGBoost_RMSE"],
    zone_rmse_chart_data["Naive_RMSE"],
):
    ax.text(
        xgb_rmse + 2,
        position - bar_height / 2,
        f"{xgb_rmse:.1f}",
        va="center",
        fontsize=11,
        color="#24476F",
        fontweight="bold",
    )

    ax.text(
        naive_rmse + 2,
        position + bar_height / 2,
        f"{naive_rmse:.1f}",
        va="center",
        fontsize=11,
        color="#555B61",
    )

ax.set_yticks(
    y_positions
)

ax.set_yticklabels(
    zone_labels
)

ax.set_xlabel(
    "RMSE"
)

ax.set_ylabel(
    ""
)

ax.set_title(
    "Where Does the Final Model Make Its Largest Geographic Errors?",
    loc="left",
    fontsize=20,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.02,
    (
        "Zones are ranked by final-model RMSE. "
        "Absolute error is naturally larger in high-demand zones."
    ),
    transform=ax.transAxes,
    fontsize=13.5,
    color="#596579",
)

ax.legend(
    title="Model",
    frameon=False,
    loc="lower right",
)

ax.grid(
   False
)

ax.grid(
    axis="y",
    visible=False,
)

sns.despine(
    ax=ax,
    left=True,
    bottom=False,
)

plt.tight_layout()
plt.show()

### Which zones have the largest errors in pickups?

LaGuardia and JFK have the largest absolute errors. Scheduled flight arrivals might help explain some of that variation, but this plot cannot establish the cause. I'll treat airport information as a future feature experiment, with its own timing and validation checks.


In [ ]:
# Defining a minimum demand level for meaningful WAPE comparison.
minimum_mean_zone_demand = 10

# Selecting zones with sufficient average demand.
zone_wape_chart_data = (
    zone_residual_results.loc[
        zone_residual_results[
            "Mean_Actual_Demand"
        ] >= minimum_mean_zone_demand
    ]
    .nlargest(
        15,
        "XGBoost_WAPE_Percent",
    )
    .sort_values(
        "XGBoost_WAPE_Percent",
        ascending=True,
    )
    .copy()
)

# Checking the chart inputs.
assert len(zone_wape_chart_data) == 15

assert zone_wape_chart_data[
    [
        "XGBoost_WAPE_Percent",
        "Mean_Actual_Demand",
    ]
].notna().all().all()

assert (
    zone_wape_chart_data[
        "Mean_Actual_Demand"
    ] >= minimum_mean_zone_demand
).all()

zone_wape_chart_data["Zone_Label"] = (
    zone_wape_chart_data["Zone"]
    + " — "
    + zone_wape_chart_data["Borough"]
)

# Creating the relative-error chart.
fig, ax = plt.subplots(
    figsize=(15, 10)
)

bars = ax.barh(
    zone_wape_chart_data["Zone_Label"],
    zone_wape_chart_data[
        "XGBoost_WAPE_Percent"
    ],
    color="#4C78A8",
    height=0.64,
)

# Adding WAPE and demand-scale labels.
for bar, wape, mean_demand in zip(
    bars,
    zone_wape_chart_data[
        "XGBoost_WAPE_Percent"
    ],
    zone_wape_chart_data[
        "Mean_Actual_Demand"
    ],
):
    ax.text(
        wape + 0.3,
        bar.get_y() + bar.get_height() / 2,
        (
            f"{wape:.1f}%  "
            f"(mean demand {mean_demand:.1f})"
        ),
        va="center",
        fontsize=12,
        color="#334155",
    )

ax.set_title(
    "Which Zones Have the Largest Errors Relative to Demand?",
    loc="left",
    fontsize=20,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.02,
    (
        "Zones are ranked by final-model WAPE; "
        "only zones averaging at least 10 pickups per hour are included."
    ),
    transform=ax.transAxes,
    fontsize=12,
    color="#596579",
)

ax.set_xlabel(
    "WAPE (%)"
)

ax.set_ylabel(
    ""
)

ax.grid(False)

sns.despine(
    ax=ax,
    left=True,
    bottom=False,
)

# Providing enough room for the annotations.
ax.set_xlim(
    0,
    zone_wape_chart_data[
        "XGBoost_WAPE_Percent"
    ].max() * 1.45,
)

plt.tight_layout()
plt.show()

### What changes when I account for zone volume?

The WAPE ranking tells a different story from RMSE. Busy airports can have large misses in pickup units without the largest relative errors, while a quiet zone can look poor in percentage terms. I'll keep volume beside both metrics when deciding which locations need attention.


In [ ]:
# Selecting zones with sufficient demand for bias analysis.
eligible_zone_bias_results = (
    zone_residual_results.loc[
        zone_residual_results[
            "Mean_Actual_Demand"
        ] >= minimum_mean_zone_demand
    ]
    .copy()
)

# Selecting the strongest underprediction and overprediction.
most_underpredicted_zones = (
    eligible_zone_bias_results
    .nsmallest(
        10,
        "XGBoost_Mean_Bias",
    )
)

most_overpredicted_zones = (
    eligible_zone_bias_results
    .nlargest(
        10,
        "XGBoost_Mean_Bias",
    )
)

zone_bias_chart_data = (
    pd.concat(
        [
            most_underpredicted_zones,
            most_overpredicted_zones,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset="location_id"
    )
    .sort_values(
        "XGBoost_Mean_Bias",
        ascending=True,
    )
    .copy()
)

# Checking the selected bias values.
assert zone_bias_chart_data[
    "XGBoost_Mean_Bias"
].notna().all()

assert zone_bias_chart_data[
    "location_id"
].is_unique

zone_bias_chart_data["Zone_Label"] = (
    zone_bias_chart_data["Zone"]
    + " — "
    + zone_bias_chart_data["Borough"]
)

bias_colors = np.where(
    zone_bias_chart_data[
        "XGBoost_Mean_Bias"
    ] < 0,
    "#C9856B",
    "#4C78A8",
)

# Creating the diverging bias chart.
fig, ax = plt.subplots(
    figsize=(15, 11)
)

bars = ax.barh(
    zone_bias_chart_data["Zone_Label"],
    zone_bias_chart_data[
        "XGBoost_Mean_Bias"
    ],
    color=bias_colors,
    height=0.65,
)

# Adding signed bias labels.
for bar, bias_value in zip(
    bars,
    zone_bias_chart_data[
        "XGBoost_Mean_Bias"
    ],
):
    label_offset = (
        0.25
        if bias_value >= 0
        else -0.25
    )

    ax.text(
        bias_value + label_offset,
        bar.get_y() + bar.get_height() / 2,
        f"{bias_value:+.1f}",
        va="center",
        ha=(
            "left"
            if bias_value >= 0
            else "right"
        ),
        fontsize=10,
        fontweight="bold",
        color="#334155",
    )

ax.axvline(
    0,
    color="#364152",
    linewidth=1.3,
    linestyle="--",
)

ax.set_title(
    "Which Zones Are Systematically Underpredicted or Overpredicted?",
    loc="left",
    fontsize=20,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.02,
    (
        "Negative bias indicates underprediction; "
        "positive bias indicates overprediction. "
        "Zones average at least 10 pickups per hour."
    ),
    transform=ax.transAxes,
    fontsize=12,
    color="#596579",
)

ax.set_xlabel(
    "Mean bias (prediction − actual)"
)

ax.set_ylabel(
    ""
)

ax.grid(False)

sns.despine(
    ax=ax,
    left=True,
    bottom=False,
)

plt.tight_layout()
plt.show()

### Which zones do I consistently miss high or low?

The negative bias is concentrated rather than evenly spread across the city. Several Queens and Brooklyn zones contribute large shortfalls. That helps identify where to investigate first, but it is not a reason to fit zone corrections directly from the test results.


In [ ]:
# Creating the location-to-borough mapping.
location_to_borough = (
    taxi_zone_lookup
    .set_index("LocationID")[
        "Borough"
    ]
    .to_dict()
)

# Adding borough descriptions to the test evaluation rows.
test_evaluation["borough"] = (
    test_evaluation["location_id"]
    .astype("int64")
    .map(location_to_borough)
)

# Checking complete lookup coverage.
assert test_evaluation[
    "borough"
].notna().all()

assert test_evaluation[
    "borough"
].nunique() >= 5

# Aggregating model performance across every row in each borough.
borough_residual_results = (
    test_evaluation
    .groupby(
        "borough",
        observed=True,
    )
    .agg(
        Rows=(
            "total_pickups",
            "size",
        ),
        Mean_Actual_Demand=(
            "total_pickups",
            "mean",
        ),
        Total_Actual_Demand=(
            "total_pickups",
            "sum",
        ),
        XGBoost_Absolute_Error_Total=(
            "xgboost_absolute_error",
            "sum",
        ),
        XGBoost_Squared_Error_Mean=(
            "xgboost_squared_error",
            "mean",
        ),
        XGBoost_Mean_Residual=(
            "xgboost_residual",
            "mean",
        ),
        Naive_Absolute_Error_Total=(
            "weekly_naive_absolute_error",
            "sum",
        ),
        Naive_Squared_Error_Mean=(
            "weekly_naive_squared_error",
            "mean",
        ),
        Naive_Mean_Residual=(
            "weekly_naive_residual",
            "mean",
        ),
    )
    .reset_index()
)

# Calculating interpretable borough-level metrics.
borough_residual_results[
    "XGBoost_RMSE"
] = np.sqrt(
    borough_residual_results[
        "XGBoost_Squared_Error_Mean"
    ]
)

borough_residual_results[
    "XGBoost_WAPE_Percent"
] = (
    100
    * borough_residual_results[
        "XGBoost_Absolute_Error_Total"
    ]
    / borough_residual_results[
        "Total_Actual_Demand"
    ]
)

borough_residual_results[
    "XGBoost_Mean_Bias"
] = -borough_residual_results[
    "XGBoost_Mean_Residual"
]

borough_residual_results[
    "Naive_RMSE"
] = np.sqrt(
    borough_residual_results[
        "Naive_Squared_Error_Mean"
    ]
)

borough_residual_results[
    "RMSE_Improvement_Percent"
] = (
    100
    * (
        borough_residual_results[
            "Naive_RMSE"
        ]
        - borough_residual_results[
            "XGBoost_RMSE"
        ]
    )
    / borough_residual_results[
        "Naive_RMSE"
    ]
)

# Checking that all test rows remain represented.
assert borough_residual_results[
    "Rows"
].sum() == len(test_evaluation)

assert np.isfinite(
    borough_residual_results[
        [
            "XGBoost_RMSE",
            "XGBoost_WAPE_Percent",
            "XGBoost_Mean_Bias",
            "Naive_RMSE",
            "RMSE_Improvement_Percent",
        ]
    ].to_numpy()
).all()

borough_display_columns = [
    "borough",
    "Rows",
    "Mean_Actual_Demand",
    "XGBoost_RMSE",
    "XGBoost_WAPE_Percent",
    "XGBoost_Mean_Bias",
    "Naive_RMSE",
    "RMSE_Improvement_Percent",
]

display(
    borough_residual_results[
        borough_display_columns
    ]
    .sort_values(
        "XGBoost_Mean_Bias"
    )
    .style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "Mean_Actual_Demand": "{:.2f}",
        "XGBoost_RMSE": "{:.3f}",
        "XGBoost_WAPE_Percent": "{:.3f}",
        "XGBoost_Mean_Bias": "{:+.3f}",
        "Naive_RMSE": "{:.3f}",
        "RMSE_Improvement_Percent": "{:.1f}",
    })
)

In [ ]:
# Isolating the EWR taxi zone for a structural audit.
ewr_test_rows = test_evaluation.loc[
    test_evaluation["borough"] == "EWR"
].copy()

assert len(ewr_test_rows) == 13_068
assert ewr_test_rows[
    "location_id"
].nunique() == 1

# Summarizing the unusual EWR demand and predictions.
ewr_audit_summary = pd.DataFrame({
    "Measure": [
        "Test rows",
        "Distinct forecast origins",
        "Distinct target hours",
        "Actual demand total",
        "Actual demand mean",
        "Actual demand maximum",
        "Zero-demand rows",
        "Zero-demand rows (%)",
        "XGBoost prediction mean",
        "XGBoost prediction maximum",
        "Weekly naive prediction mean",
        "Weekly naive prediction maximum",
    ],
    "Value": [
        len(ewr_test_rows),
        ewr_test_rows[
            "forecast_origin_utc"
        ].nunique(),
        ewr_test_rows[
            "target_hour_utc"
        ].nunique(),
        ewr_test_rows[
            "total_pickups"
        ].sum(),
        ewr_test_rows[
            "total_pickups"
        ].mean(),
        ewr_test_rows[
            "total_pickups"
        ].max(),
        int(
            ewr_test_rows[
                "total_pickups"
            ].eq(0).sum()
        ),
        (
            100
            * ewr_test_rows[
                "total_pickups"
            ].eq(0).mean()
        ),
        ewr_test_rows[
            "xgboost_prediction"
        ].mean(),
        ewr_test_rows[
            "xgboost_prediction"
        ].max(),
        ewr_test_rows[
            "weekly_naive_prediction"
        ].mean(),
        ewr_test_rows[
            "weekly_naive_prediction"
        ].max(),
    ],
})

display(
    ewr_audit_summary.style
    .hide(axis="index")
    .format({
        "Value": lambda value: (
            f"{value:,.6f}"
            if isinstance(value, (float, np.floating))
            else f"{value:,}"
        ),
    })
)

In [ ]:
# Defining the operational NYC-only evaluation scope.
nyc_test_mask = (
    test_evaluation["borough"] != "EWR"
)

assert nyc_test_mask.sum() == (
    len(test_evaluation) - 13_068
)

assert test_evaluation.loc[
    nyc_test_mask,
    "location_id",
].nunique() == 262

# Calculating NYC-only model metrics.
nyc_only_xgboost_metrics = (
    calculate_forecast_metrics(
        test_evaluation.loc[
            nyc_test_mask,
            "total_pickups",
        ].to_numpy(),
        test_evaluation.loc[
            nyc_test_mask,
            "xgboost_prediction",
        ].to_numpy(),
    )
)

nyc_only_naive_metrics = (
    calculate_forecast_metrics(
        test_evaluation.loc[
            nyc_test_mask,
            "total_pickups",
        ].to_numpy(),
        test_evaluation.loc[
            nyc_test_mask,
            "weekly_naive_prediction",
        ].to_numpy(),
    )
)

# Comparing the original and operational evaluation scopes.
evaluation_scope_comparison = pd.DataFrame([
    {
        "Evaluation Scope": "All 263 TLC zones",
        "Model": "Final XGBoost",
        "Rows": len(test_evaluation),
        **final_test_metrics,
    },
    {
        "Evaluation Scope": "NYC only — 262 zones",
        "Model": "Final XGBoost",
        "Rows": int(nyc_test_mask.sum()),
        **nyc_only_xgboost_metrics,
    },
    {
        "Evaluation Scope": "All 263 TLC zones",
        "Model": "Weekly seasonal naive",
        "Rows": len(test_evaluation),
        **test_weekly_naive_metrics,
    },
    {
        "Evaluation Scope": "NYC only — 262 zones",
        "Model": "Weekly seasonal naive",
        "Rows": int(nyc_test_mask.sum()),
        **nyc_only_naive_metrics,
    },
])

display(
    evaluation_scope_comparison.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "mae": "{:.4f}",
        "rmse": "{:.4f}",
        "wape_percent": "{:.4f}",
        "mean_bias": "{:+.4f}",
        "r_squared": "{:.5f}",
    })
)

### Which zones belong in the NYC replay?

Newark is in the historical TLC lookup, but outside the planned NYC-origin forecast scope. I'll preserve the 263-zone evaluation and report the NYC-only summary separately. Excluding it from a table does not change the trained model; a final 262-zone version needs a new fit with the frozen configuration and that scope applied from the start.


In [ ]:
# Converting negative model bias into positive underprediction magnitude.
nyc_borough_underprediction_data = (
    borough_residual_results.loc[
        borough_residual_results[
            "borough"
        ] != "EWR"
    ]
    .copy()
)

nyc_borough_underprediction_data[
    "Mean_Underprediction"
] = (
    -nyc_borough_underprediction_data[
        "XGBoost_Mean_Bias"
    ]
)

# Checking that every NYC borough is underpredicted.
assert (
    nyc_borough_underprediction_data[
        "Mean_Underprediction"
    ] > 0
).all()

# Ordering the bars so the largest underprediction appears first.
nyc_borough_underprediction_data = (
    nyc_borough_underprediction_data
    .sort_values(
        "Mean_Underprediction",
        ascending=True,
    )
)

# Creating the positive-scale underprediction chart.
fig, ax = plt.subplots(
    figsize=(12, 7)
)

bars = ax.barh(
    nyc_borough_underprediction_data[
        "borough"
    ],
    nyc_borough_underprediction_data[
        "Mean_Underprediction"
    ],
    color="#C9856B",
    height=0.62,
)

# Adding underprediction values.
for bar, value in zip(
    bars,
    nyc_borough_underprediction_data[
        "Mean_Underprediction"
    ],
):
    ax.text(
        value + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        va="center",
        ha="left",
        fontsize=11,
        fontweight="bold",
        color="#334155",
    )

ax.set_title(
    "Where Is the Model's Average Underprediction Strongest?",
    loc="left",
    fontsize=20,
    fontweight="bold",
    pad=34,
)

ax.text(
    0,
    1.02,
    (
        "Values show mean actual demand minus mean predicted demand "
        "across all test forecasts."
    ),
    transform=ax.transAxes,
    fontsize=12,
    color="#596579",
)

ax.set_xlabel(
    "Mean underprediction (pickups per forecast)"
)

ax.set_ylabel(
    ""
)

ax.set_xlim(
    0,
    nyc_borough_underprediction_data[
        "Mean_Underprediction"
    ].max() * 1.15,
)

ax.grid(False)

sns.despine(
    ax=ax,
    left=True,
    bottom=False,
)

plt.tight_layout()
plt.show()

In [ ]:
# Identifying day-type features used by the final model.
day_type_candidates = [
    "is_weekend",
    "is_weekday_rush_hour",
    "is_weekend_night",
    "is_holiday",
    "is_unofficial_holiday",
]

day_type_feature_audit = pd.DataFrame({
    "Feature": day_type_candidates,
    "Used by Final Model": [
        feature_name in primary_features
        for feature_name in day_type_candidates
    ],
    "Available in Test Evaluation": [
        feature_name in test_evaluation.columns
        for feature_name in day_type_candidates
    ],
})

display(
    day_type_feature_audit.style
    .hide(axis="index")
)

print(
    "Available test evaluation columns:",
    sorted(test_evaluation.columns),
)

In [ ]:
# Defining the original day-type columns needed for evaluation.
test_day_type_columns = [
    "forecast_origin_utc",
    "target_hour_utc",
    "location_id",
    "forecast_horizon",
    "is_weekend",
    "is_weekday_rush_hour",
    "is_weekend_night",
    "is_holiday",
    "is_unofficial_holiday",
]

# Reconstructing the production feature dataset.
day_type_feature_dataset = ds.dataset(
    [
        str(path)
        for path in final_partition_paths
    ],
    format="parquet",
)

# Recovering the frozen test-origin boundaries.
day_type_first_origin = (
    test_evaluation[
        "forecast_origin_utc"
    ].min()
)

day_type_last_origin = (
    test_evaluation[
        "forecast_origin_utc"
    ].max()
)

# Loading only the required test columns.
test_day_type_table = (
    day_type_feature_dataset.to_table(
        columns=test_day_type_columns,
        filter=(
            (
                ds.field("forecast_origin_utc")
                >= day_type_first_origin.to_pydatetime()
            )
            & (
                ds.field("forecast_origin_utc")
                <= day_type_last_origin.to_pydatetime()
            )
        ),
    )
)

test_day_type_frame = (
    test_day_type_table.to_pandas(
        split_blocks=True
    )
)

# Checking the recovered test structure.
assert len(test_day_type_frame) == len(
    test_evaluation
)

test_alignment_keys = [
    "forecast_origin_utc",
    "target_hour_utc",
    "location_id",
    "forecast_horizon",
]

# Verifying exact row alignment before attaching flags.
for key_name in test_alignment_keys:
    left_values = (
        test_evaluation[key_name]
        .reset_index(drop=True)
    )

    right_values = (
        test_day_type_frame[key_name]
        .reset_index(drop=True)
    )

    assert left_values.equals(
        right_values
    ), (
        f"Test row alignment failed for "
        f"{key_name}."
    )

# Attaching the original production day-type flags.
for feature_name in day_type_candidates:
    test_evaluation[feature_name] = (
        test_day_type_frame[
            feature_name
        ]
        .astype(bool)
        .to_numpy()
    )

# Confirming that all flags were recovered.
assert set(
    day_type_candidates
).issubset(
    test_evaluation.columns
)

assert not test_evaluation[
    day_type_candidates
].isna().any().any()

day_type_recovery_summary = pd.DataFrame({
    "Feature": day_type_candidates,
    "True Rows": [
        int(
            test_evaluation[
                feature_name
            ].sum()
        )
        for feature_name in day_type_candidates
    ],
    "True Rows (%)": [
        (
            100
            * test_evaluation[
                feature_name
            ].mean()
        )
        for feature_name in day_type_candidates
    ],
})

display(
    day_type_recovery_summary.style
    .hide(axis="index")
    .format({
        "True Rows": "{:,}",
        "True Rows (%)": "{:.3f}",
    })
)

# Releasing temporary reload objects.
del test_day_type_frame
del test_day_type_table
del day_type_feature_dataset

gc.collect()

### Final performance by operational day type

Calendar flags overlap: a holiday can also be a weekend night. I'll use mutually exclusive day types here so the comparisons have a clear meaning and the same rows do not appear in several seemingly separate groups.


In [ ]:
# Defining mutually exclusive operational day types.
day_type_conditions = [
    test_evaluation["is_holiday"],
    (
        ~test_evaluation["is_holiday"]
        & test_evaluation["is_weekend_night"]
    ),
    (
        ~test_evaluation["is_holiday"]
        & test_evaluation["is_weekend"]
        & ~test_evaluation["is_weekend_night"]
    ),
    (
        ~test_evaluation["is_holiday"]
        & ~test_evaluation["is_weekend"]
        & test_evaluation["is_weekday_rush_hour"]
    ),
]

day_type_labels = [
    "Official holiday",
    "Weekend late night",
    "Weekend daytime/evening",
    "Weekday rush hours",
]

day_type_order = [
    "Regular weekday hours",
    "Weekday rush hours",
    "Weekend daytime/evening",
    "Weekend late night",
    "Official holiday",
]

test_evaluation["operational_day_type"] = np.select(
    day_type_conditions,
    day_type_labels,
    default="Regular weekday hours",
)

# Checking that every test row belongs to exactly one day type.
assert test_evaluation["operational_day_type"].notna().all()
assert len(test_evaluation) == 3_436_884


# Calculating comparable metrics for both forecasting methods.
# Calculating both model results with the reusable helper.
day_type_results = (
    calculate_metrics_by_group(
        evaluation_frame=test_evaluation,
        group_columns="operational_day_type",
        prediction_columns={
            "Final XGBoost": (
                "xgboost_prediction"
            ),
            "Weekly Seasonal Naive": (
                "weekly_naive_prediction"
            ),
        },
    )
    .rename(columns={
        "operational_day_type": "Day Type",
    })
)

# Preserving the established day-type and model order.
day_type_sort_order = {
    day_type: position
    for position, day_type in enumerate(
        day_type_order
    )
}

model_sort_order = {
    "Final XGBoost": 0,
    "Weekly Seasonal Naive": 1,
}

day_type_results[
    "_day_type_order"
] = day_type_results[
    "Day Type"
].map(day_type_sort_order)

day_type_results[
    "_model_order"
] = day_type_results[
    "Model"
].map(model_sort_order)

day_type_results = (
    day_type_results
    .sort_values(
        [
            "_day_type_order",
            "_model_order",
        ]
    )
    .drop(
        columns=[
            "_day_type_order",
            "_model_order",
        ]
    )
    .reset_index(drop=True)
)

# Checking complete test coverage for both models.
day_type_row_totals = (
    day_type_results
    .groupby(
        "Model",
        observed=True,
    )["Rows"]
    .sum()
)

assert len(day_type_results) == 10
assert set(day_type_results["Day Type"]) == set(
    day_type_order
)

assert (
    day_type_row_totals
    == len(test_evaluation)
).all()

# Preparing readable labels without changing the calculated results.
day_type_display = (
    day_type_results
    .rename(columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "wape_percent": "WAPE (%)",
        "mean_bias": "Mean Bias",
        "r_squared": "R²",
    })
    [
        [
            "Day Type",
            "Model",
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
            "R²",
        ]
    ]
    .copy()
)

# Styling the two models consistently across day types.
def style_day_type_model(row):
    if row["Model"] == "Final XGBoost":
        background = "#07346a"
        weight = "bold"
    else:
        background = "#131a2b"
        weight = "normal"

    return [
        (
            f"background-color: {background}; "
            f"font-weight: {weight}; "
            "color: #e4e6eb;"
        )
    ] * len(row)


display(
    day_type_display.style
    .hide(axis="index")
    .format({
        "Rows": "{:,.0f}",
        "MAE": "{:.3f}",
        "RMSE": "{:.3f}",
        "WAPE (%)": "{:.3f}",
        "Mean Bias": "{:+.3f}",
        "R²": "{:.4f}",
    })
    .apply(
        style_day_type_model,
        axis=1,
    )
    .set_properties(
        subset=["Day Type", "Model"],
        **{"text-align": "left"},
    )
    .set_properties(
        subset=[
            "Rows",
            "MAE",
            "RMSE",
            "WAPE (%)",
            "Mean Bias",
            "R²",
        ],
        **{"text-align": "right"},
    )
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding-bottom", "10px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#1f2937"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "8px 12px"),
                ("border-bottom", "2px solid #9ca3af"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "7px 12px"),
                ("border-bottom", "1px solid #d1d5db"),
            ],
        },
        {
            "selector": "tbody tr:nth-child(even) td",
            "props": [
                ("border-bottom", "3px solid #9ca3af"),
            ],
        },
    ])
    .set_caption(
        "Final Test Performance by Operational Day Type"
    )
)

# Preparing day-type WAPE values in the intended display order.
day_type_chart_data = (
    day_type_results
    .assign(
        **{
            "Day Type": pd.Categorical(
                day_type_results["Day Type"],
                categories=day_type_order,
                ordered=True,
            )
        }
    )
    .sort_values(["Day Type", "Model"])
)

# Checking that both models are represented for every day type.
day_type_model_counts = (
    day_type_chart_data
    .groupby(
        "Day Type",
        observed=True,
    )["Model"]
    .nunique()
)

assert (day_type_model_counts == 2).all()
assert len(day_type_chart_data) == 10

# Creating a horizontal comparison because the category labels are long.
fig, ax = plt.subplots(figsize=(11, 6.5))

sns.barplot(
    data=day_type_chart_data,
    y="Day Type",
    x="wape_percent",
    hue="Model",
    hue_order=[
        "Final XGBoost",
        "Weekly Seasonal Naive",
    ],
    order=day_type_order,
    palette={
        "Final XGBoost": "#2563EB",
        "Weekly Seasonal Naive": "#9CA3AF",
    },
    edgecolor="white",
    linewidth=0.8,
    ax=ax,
)

# Adding the exact WAPE value to every bar.
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=4,
        fontsize=10,
    )

# Leaving space for labels beyond the longest bar.
maximum_wape = day_type_chart_data[
    "wape_percent"
].max()

ax.set_xlim(0, maximum_wape * 1.16)

ax.set_title(
    "XGBoost Improves Relative Accuracy Across Every Day Type",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=24,
)

ax.text(
    0,
    1.01,
    (
        "Final held-out test set · "
        "lower WAPE indicates better performance"
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10.5,
    color="#4B5563",
)

ax.set_xlabel("WAPE (%)", fontsize=11)
ax.set_ylabel("")

ax.legend(
    title="",
    frameon=False,
    loc="upper right",
)

ax.xaxis.grid(
    False
)

ax.yaxis.grid(False)
sns.despine(ax=ax, left=True, bottom=True)

plt.tight_layout()
plt.show()

In [ ]:
# Recovering the feature names stored in the final fitted model.
final_model_feature_names = final_xgb_model.feature_names

assert final_model_feature_names is not None
assert len(final_model_feature_names) == 30
assert len(set(final_model_feature_names)) == 30

# Extracting average gain and split-frequency importance.
final_gain_scores = final_xgb_model.get_score(
    importance_type="gain",
)

final_split_scores = final_xgb_model.get_score(
    importance_type="weight",
)

# Including features that received no splits with zero importance.
final_feature_importance = pd.DataFrame({
    "Feature": final_model_feature_names,
    "Gain": [
        float(final_gain_scores.get(feature_name, 0.0))
        for feature_name in final_model_feature_names
    ],
    "Split Count": [
        int(final_split_scores.get(feature_name, 0))
        for feature_name in final_model_feature_names
    ],
})

assert len(final_feature_importance) == 30
assert final_feature_importance["Gain"].ge(0).all()
assert final_feature_importance["Split Count"].ge(0).all()
assert final_feature_importance["Gain"].sum() > 0

# Expressing gain on a normalized scale for easier comparison.
final_feature_importance["Relative Gain (%)"] = (
    100
    * final_feature_importance["Gain"]
    / final_feature_importance["Gain"].sum()
)

top_15_gain_importance = (
    final_feature_importance
    .sort_values(
        ["Gain", "Split Count"],
        ascending=[False, False],
    )
    .head(15)
    .reset_index(drop=True)
)

top_15_gain_importance.insert(
    0,
    "Rank",
    np.arange(1, len(top_15_gain_importance) + 1),
)

display(
    top_15_gain_importance.style
    .hide(axis="index")
    .format({
        "Rank": "{:.0f}",
        "Gain": "{:,.3f}",
        "Relative Gain (%)": "{:.2f}",
        "Split Count": "{:,}",
    })
    .background_gradient(
        subset=["Relative Gain (%)"],
        cmap="Blues",
    )
)

# Preparing readable feature labels and ascending order for plotting.
top_15_gain_chart = (
    top_15_gain_importance
    .copy()
    .sort_values(
        "Relative Gain (%)",
        ascending=False,
    )
)

top_15_gain_chart["Feature Label"] = (
    top_15_gain_chart["Feature"]
    .str.replace("_", " ", regex=False)
    .str.replace("pickup lag", "Pickup lag", regex=False)
    .str.replace("forecast ", "Forecast ", regex=False)
    .str.replace("is ", "", regex=False)
    .str.capitalize()
)

# Confirming that the chart represents the same 15 ranked features.
assert len(top_15_gain_chart) == 15
assert top_15_gain_chart["Relative Gain (%)"].notna().all()

fig, ax = plt.subplots(figsize=(11, 7.5))

sns.barplot(
    data=top_15_gain_chart,
    y="Feature Label",
    x="Relative Gain (%)",
    color="#2563EB",
    edgecolor="white",
    linewidth=0.8,
    ax=ax,
)

# Adding exact normalized gain scores.
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=4,
        fontsize=9.5,
    )

maximum_relative_gain = top_15_gain_chart[
    "Relative Gain (%)"
].max()

ax.set_xlim(0, maximum_relative_gain * 1.10)

ax.set_title(
    "Weekly Demand History Dominates XGBoost Gain Importance",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=24,
)

ax.text(
    0,
    1.01,
    (
        "Top 15 features from the final fitted model · "
        "scores reflect predictive usage, not causal effects"
    ),
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=10.5,
    color="#4B5563",
)

ax.set_xlabel("Relative gain score (%)", fontsize=11)
ax.set_ylabel("")

ax.xaxis.grid(
    False
)

ax.yaxis.grid(False)
sns.despine(ax=ax, left=True, bottom=True)

plt.tight_layout()
plt.show()

### What does split gain tell me about the features?

The model leans heavily on weekly pickup history. The one-week lag accounts for about 72.6% of the normalized average-gain scores, followed by the two-week lag at 13.2%. That matches the earlier baseline story. These scores describe the fitted trees' usage, not how much pickups a feature causes or an independent share of variance explained.


## What is ready, and what still needs work?

The [July replay](../website/REPLAY_UPDATE.md) takes the saved model into the following month and shows how predictions change when recent errors become available.

The model now gives me a useful offline baseline. Test RMSE is 25.06, compared with 34.92 for repeating last week, and MAE falls from 18.00 to 13.11. It still underpredicts by about 2.18 pickups per forecast case on average. The next work should address that weakness without losing the broad improvement.

I reran this notebook in a clean copy and used the saved model to reproduce the July replay. A live deployment would still need timely pickup counts and explicit rules for late inputs. The historical weather archive, event information, and daylight-saving cleaning also need to match what could have been known at issuance. I'll keep the current 263-zone artifact as the evaluation model and treat NYC-only retraining as a separate step.

Most importantly, further model changes need a new chronological evaluation. The current period has already informed the error analysis, and earlier event exploration used overlapping dates. Those results are worth keeping, but they should not become an endlessly reused test for every proposed improvement.

$$\
262 \text{ zones}
\times
6 \text{ horizons}
=
1{,}572 \text{ rows per forecast origin}
$$\


In [ ]:
import hashlib
import json
import platform
import sys

# Defining the reproducible 263-zone evaluation artifact.
evaluation_artifact_id = (
    "xgboost_263_zone_evaluation_2026-08-25"
)

evaluation_artifact_dir = (
    PROJECT_ROOT
    / "models"
    / "evaluation"
    / evaluation_artifact_id
)

evaluation_model_path = (
    evaluation_artifact_dir
    / "xgboost_model.json"
)

evaluation_artifact_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Saving only when the artifact does not already exist.
if not evaluation_model_path.exists():
    final_xgb_model.save_model(
        evaluation_model_path
    )

    model_file_action = "New model saved"

else:
    model_file_action = (
        "Existing model retained without overwrite"
    )

assert evaluation_model_path.is_file()
assert evaluation_model_path.stat().st_size > 0

# Reloading the saved artifact independently.
reloaded_evaluation_model = xgb.Booster()

reloaded_evaluation_model.load_model(
    evaluation_model_path
)

# Validating the saved model structure.
assert (
    reloaded_evaluation_model.num_boosted_rounds()
    == final_xgb_model.num_boosted_rounds()
    == 1_613
)

assert (
    reloaded_evaluation_model.feature_names
    == final_xgb_model.feature_names
)

assert len(
    reloaded_evaluation_model.feature_names
) == 30

# Generating predictions from both model objects.
current_model_raw_predictions = (
    final_xgb_model.predict(
        final_test_matrix
    )
)

reloaded_model_raw_predictions = (
    reloaded_evaluation_model.predict(
        final_test_matrix
    )
)

# Measuring serialization and repeated-GPU-inference differences.
serialization_absolute_difference = np.abs(
    current_model_raw_predictions
    - reloaded_model_raw_predictions
)

# Applying the documented nonnegative production constraint.
reloaded_test_predictions_constrained = np.clip(
    reloaded_model_raw_predictions,
    a_min=0,
    a_max=None,
)

stored_test_predictions = (
    test_evaluation[
        "xgboost_prediction"
    ].to_numpy()
)

stored_absolute_difference = np.abs(
    reloaded_test_predictions_constrained
    - stored_test_predictions
)

# Recalculating the reported test metrics.
reloaded_test_metrics = calculate_forecast_metrics(
    actual=y_test,
    prediction=reloaded_test_predictions_constrained,
)

# Accepting only operationally negligible inference differences.
assert (
    serialization_absolute_difference.max()
    < 0.01
)

assert (
    stored_absolute_difference.max()
    < 0.01
)

# Confirming that reported metrics are preserved.
for metric_name in [
    "mae",
    "rmse",
    "wape_percent",
    "mean_bias",
    "r_squared",
]:
    assert np.isclose(
        reloaded_test_metrics[metric_name],
        final_test_metrics[metric_name],
        rtol=0,
        atol=1e-4,
    ), (
        f"Reloaded metric changed: {metric_name}"
    )

# Calculating an integrity checksum.
model_sha256 = hashlib.sha256(
    evaluation_model_path.read_bytes()
).hexdigest()

model_verification_summary = pd.DataFrame({
    "Check": [
        "Original model vs reloaded model",
        "Reloaded constrained vs stored predictions",
    ],
    "Rows": [
        len(current_model_raw_predictions),
        len(stored_test_predictions),
    ],
    "Maximum Absolute Difference": [
        serialization_absolute_difference.max(),
        stored_absolute_difference.max(),
    ],
    "Mean Absolute Difference": [
        serialization_absolute_difference.mean(),
        stored_absolute_difference.mean(),
    ],
    "99.9th Percentile Difference": [
        np.quantile(
            serialization_absolute_difference,
            0.999,
        ),
        np.quantile(
            stored_absolute_difference,
            0.999,
        ),
    ],
})

display(
    model_verification_summary.style
    .hide(axis="index")
    .format({
        "Rows": "{:,}",
        "Maximum Absolute Difference": "{:.10f}",
        "Mean Absolute Difference": "{:.10f}",
        "99.9th Percentile Difference": "{:.10f}",
    })
)

print(
    "\nModel file action:",
    model_file_action,
)

print(
    "Model path:",
    evaluation_model_path,
)

print(
    "Model size (MB):",
    f"{evaluation_model_path.stat().st_size / 1024**2:.2f}",
)

print(
    "Boosting rounds:",
    f"{reloaded_evaluation_model.num_boosted_rounds():,}",
)

print(
    "Stored features:",
    len(reloaded_evaluation_model.feature_names),
)

print(
    "Reloaded MAE:",
    f"{reloaded_test_metrics['mae']:.4f}",
)

print(
    "Reloaded RMSE:",
    f"{reloaded_test_metrics['rmse']:.4f}",
)

print(
    "Reloaded WAPE (%):",
    f"{reloaded_test_metrics['wape_percent']:.4f}",
)

print(
    "Reloaded mean bias:",
    f"{reloaded_test_metrics['mean_bias']:+.4f}",
)

print(
    "Reloaded R²:",
    f"{reloaded_test_metrics['r_squared']:.5f}",
)

print(
    "SHA-256:",
    model_sha256,
)

print(
    "Model verification passed:",
    True,
)

In [ ]:
# import json
# import platform
# import sys

# # Recalculating the checksum after successful verification.
# model_sha256 = hashlib.sha256(
#     evaluation_model_path.read_bytes()
# ).hexdigest()

# artifact_created_at_utc = (
#     pd.Timestamp.now(tz="UTC").isoformat()
# )

# # Defining supporting artifact paths.
# artifact_metadata_path = (
#     evaluation_artifact_dir
#     / "metadata.json"
# )

# feature_contract_path = (
#     evaluation_artifact_dir
#     / "feature_contract.json"
# )

# test_metrics_path = (
#     evaluation_artifact_dir
#     / "test_metrics.json"
# )

# gain_importance_path = (
#     evaluation_artifact_dir
#     / "gain_importance.csv"
# )

# model_config_path = (
#     evaluation_artifact_dir
#     / "xgboost_config.json"
# )

# supporting_artifact_paths = [
#     artifact_metadata_path,
#     feature_contract_path,
#     test_metrics_path,
#     gain_importance_path,
#     model_config_path,
# ]

# # Preventing silent replacement of existing supporting artifacts.
# existing_supporting_artifacts = [
#     path
#     for path in supporting_artifact_paths
#     if path.exists()
# ]

# if existing_supporting_artifacts:
#     raise FileExistsError(
#         "Refusing to overwrite existing artifacts: "
#         f"{existing_supporting_artifacts}"
#     )

# # Documenting the model identity and intended use.
# artifact_metadata = {
#     "artifact_id": evaluation_artifact_id,
#     "artifact_type": "offline_evaluation_model",
#     "created_at_utc": artifact_created_at_utc,
#     "model_family": "XGBoost",
#     "objective": final_xgb_params["objective"],
#     "geographic_scope": {
#         "zone_count": 263,
#         "includes_ewr_location_id_1": True,
#         "approved_for_final_nyc_production": False,
#         "future_production_zone_count": 262,
#         "future_rows_per_origin": 1572,
#     },
#     "training_contract": {
#         "development_rows": 22_661_658,
#         "feature_count": 30,
#         "boosting_rounds": 1_613,
#         "test_rows": 3_436_884,
#         "test_used_for_model_selection": False,
#     },
#     "prediction_contract": {
#         "forecast_horizons": [1, 2, 3, 4, 5, 6],
#         "negative_predictions_clipped_to_zero": True,
#         "weather_availability_delay_hours": 7,
#     },
#     "model_file": evaluation_model_path.name,
#     "model_sha256": model_sha256,
#     "status": (
#         "Reproducible 263-zone evaluation artifact. "
#         "Do not label as the final 262-zone production model."
#     ),
#     "software": {
#         "python": sys.version.split()[0],
#         "xgboost": xgb.__version__,
#         "platform": platform.platform(),
#     },
# }

# # Preserving the ordered model-input contract.
# feature_contract = {
#     "feature_count": len(final_model_feature_names),
#     "ordered_feature_names": final_model_feature_names,
#     "xgboost_feature_types": (
#         final_xgb_model.feature_types
#     ),
#     "order_must_match_exactly": True,
# }

# # Preserving both original and reloaded test metrics.
# test_metric_artifact = {
#     "evaluation_scope": "all_263_zones_including_ewr",
#     "rows": 3_436_884,
#     "original_reported_metrics": {
#         key: float(value)
#         for key, value in final_test_metrics.items()
#     },
#     "reloaded_model_metrics": {
#         key: float(value)
#         for key, value in reloaded_test_metrics.items()
#     },
#     "reload_verification": {
#         "maximum_absolute_prediction_difference": float(
#             stored_absolute_difference.max()
#         ),
#         "mean_absolute_prediction_difference": float(
#             stored_absolute_difference.mean()
#         ),
#         "percentile_99_9_absolute_difference": float(
#             np.quantile(
#                 stored_absolute_difference,
#                 0.999,
#             )
#         ),
#     },
# }

# # Writing human-readable JSON contracts.
# artifact_metadata_path.write_text(
#     json.dumps(
#         artifact_metadata,
#         indent=2,
#     ),
#     encoding="utf-8",
# )

# feature_contract_path.write_text(
#     json.dumps(
#         feature_contract,
#         indent=2,
#     ),
#     encoding="utf-8",
# )

# test_metrics_path.write_text(
#     json.dumps(
#         test_metric_artifact,
#         indent=2,
#     ),
#     encoding="utf-8",
# )

# model_config_path.write_text(
#     json.dumps(
#         json.loads(
#             final_xgb_model.save_config()
#         ),
#         indent=2,
#     ),
#     encoding="utf-8",
# )

# # Saving all 30 gain-importance records, not only the top 15.
# (
#     final_feature_importance
#     .sort_values(
#         ["Gain", "Split Count"],
#         ascending=[False, False],
#     )
#     .to_csv(
#         gain_importance_path,
#         index=False,
#     )
# )

# # Verifying that every supporting artifact was written.
# for artifact_path in supporting_artifact_paths:
#     assert artifact_path.is_file()
#     assert artifact_path.stat().st_size > 0

# artifact_file_summary = pd.DataFrame({
#     "Artifact": [
#         evaluation_model_path.name,
#         *[
#             path.name
#             for path in supporting_artifact_paths
#         ],
#     ],
#     "Size (KB)": [
#         evaluation_model_path.stat().st_size / 1024,
#         *[
#             path.stat().st_size / 1024
#             for path in supporting_artifact_paths
#         ],
#     ],
# })

# display(
#     artifact_file_summary.style
#     .hide(axis="index")
#     .format({
#         "Size (KB)": "{:,.2f}",
#     })
# )

# print(
#     "\nArtifact directory:",
#     evaluation_artifact_dir,
# )

# print(
#     "Model SHA-256:",
#     model_sha256,
# )

# print(
#     "Supporting artifact validation passed:",
#     True,
# )

In [ ]:
# Defining supporting artifact paths.
artifact_metadata_path = (
    evaluation_artifact_dir
    / "metadata.json"
)

feature_contract_path = (
    evaluation_artifact_dir
    / "feature_contract.json"
)

test_metrics_path = (
    evaluation_artifact_dir
    / "test_metrics.json"
)

gain_importance_path = (
    evaluation_artifact_dir
    / "gain_importance.csv"
)

model_config_path = (
    evaluation_artifact_dir
    / "xgboost_config.json"
)

# Defining the ordered feature contract.
final_model_feature_names = (
    final_xgb_model.feature_names
)

assert final_model_feature_names is not None
assert len(final_model_feature_names) == 30

feature_contract = {
    "feature_count": 30,
    "ordered_feature_names": (
        final_model_feature_names
    ),
    "xgboost_feature_types": (
        final_xgb_model.feature_types
    ),
    "order_must_match_exactly": True,
}

# Documenting the model identity and intended use.
artifact_metadata = {
    "artifact_id": evaluation_artifact_id,
    "artifact_type": "offline_evaluation_model",
    "created_at_utc": (
        pd.Timestamp.now(
            tz="UTC"
        ).isoformat()
    ),
    "model_family": "XGBoost",
    "objective": final_xgb_params["objective"],
    "geographic_scope": {
        "zone_count": 263,
        "includes_ewr_location_id_1": True,
        "approved_for_final_nyc_production": False,
        "future_production_zone_count": 262,
        "future_rows_per_origin": 1_572,
    },
    "training_contract": {
        "development_rows": 22_661_658,
        "feature_count": 30,
        "boosting_rounds": 1_613,
        "test_rows": 3_436_884,
        "test_used_for_model_selection": False,
    },
    "prediction_contract": {
        "forecast_horizons": [
            1, 2, 3, 4, 5, 6
        ],
        "negative_predictions_clipped_to_zero": True,
        "weather_availability_delay_hours": 7,
    },
    "model_file": evaluation_model_path.name,
    "model_sha256": model_sha256,
    "status": (
        "Reproducible 263-zone evaluation artifact. "
        "Do not label as the final 262-zone production model."
    ),
    "software": {
        "python": sys.version.split()[0],
        "xgboost": xgb.__version__,
        "platform": platform.platform(),
    },
}

# Preserving the original and reloaded evaluation metrics.
test_metric_artifact = {
    "evaluation_scope": (
        "all_263_zones_including_ewr"
    ),
    "rows": 3_436_884,
    "original_reported_metrics": {
        metric_name: float(metric_value)
        for metric_name, metric_value
        in final_test_metrics.items()
    },
    "reloaded_model_metrics": {
        metric_name: float(metric_value)
        for metric_name, metric_value
        in reloaded_test_metrics.items()
    },
    "reload_verification": {
        "maximum_absolute_prediction_difference": float(
            stored_absolute_difference.max()
        ),
        "mean_absolute_prediction_difference": float(
            stored_absolute_difference.mean()
        ),
        "percentile_99_9_absolute_difference": float(
            np.quantile(
                stored_absolute_difference,
                0.999,
            )
        ),
    },
}

current_model_config = json.loads(
    final_xgb_model.save_config()
)

artifact_actions = {}

# Creating or validating the metadata.
if not artifact_metadata_path.exists():
    artifact_metadata_path.write_text(
        json.dumps(
            artifact_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )
    artifact_actions["metadata.json"] = "Created"

else:
    saved_metadata = json.loads(
        artifact_metadata_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        saved_metadata["artifact_id"]
        == evaluation_artifact_id
    )
    assert (
        saved_metadata["model_sha256"]
        == model_sha256
    )
    assert (
        saved_metadata[
            "geographic_scope"
        ]["zone_count"]
        == 263
    )

    artifact_actions["metadata.json"] = (
        "Existing file validated"
    )

# Creating or validating the feature contract.
if not feature_contract_path.exists():
    feature_contract_path.write_text(
        json.dumps(
            feature_contract,
            indent=2,
        ),
        encoding="utf-8",
    )
    artifact_actions[
        "feature_contract.json"
    ] = "Created"

else:
    saved_feature_contract = json.loads(
        feature_contract_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        saved_feature_contract
        == feature_contract
    )

    artifact_actions[
        "feature_contract.json"
    ] = "Existing file validated"

# Creating or validating the test metrics.
if not test_metrics_path.exists():
    test_metrics_path.write_text(
        json.dumps(
            test_metric_artifact,
            indent=2,
        ),
        encoding="utf-8",
    )
    artifact_actions[
        "test_metrics.json"
    ] = "Created"

else:
    saved_test_metrics = json.loads(
        test_metrics_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        saved_test_metrics["rows"]
        == 3_436_884
    )

    for metric_name in [
        "mae",
        "rmse",
        "wape_percent",
        "mean_bias",
        "r_squared",
    ]:
        assert np.isclose(
            saved_test_metrics[
                "reloaded_model_metrics"
            ][metric_name],
            reloaded_test_metrics[
                metric_name
            ],
            rtol=0,
            atol=1e-4,
        )

    artifact_actions[
        "test_metrics.json"
    ] = "Existing file validated"

# Creating or validating the XGBoost configuration.
if not model_config_path.exists():
    model_config_path.write_text(
        json.dumps(
            current_model_config,
            indent=2,
        ),
        encoding="utf-8",
    )
    artifact_actions[
        "xgboost_config.json"
    ] = "Created"

else:
    saved_model_config = json.loads(
        model_config_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        saved_model_config
        == current_model_config
    )

    artifact_actions[
        "xgboost_config.json"
    ] = "Existing file validated"

# Preparing the complete 30-feature gain table.
gain_importance_artifact = (
    final_feature_importance
    .sort_values(
        ["Gain", "Split Count"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

# Creating or validating gain importance.
if not gain_importance_path.exists():
    gain_importance_artifact.to_csv(
        gain_importance_path,
        index=False,
    )
    artifact_actions[
        "gain_importance.csv"
    ] = "Created"

else:
    saved_gain_importance = pd.read_csv(
        gain_importance_path
    )

    pd.testing.assert_frame_equal(
        saved_gain_importance,
        gain_importance_artifact,
        check_dtype=False,
        rtol=1e-10,
        atol=1e-10,
    )

    artifact_actions[
        "gain_importance.csv"
    ] = "Existing file validated"

# Confirming that the complete bundle exists.
artifact_paths = [
    evaluation_model_path,
    artifact_metadata_path,
    feature_contract_path,
    test_metrics_path,
    gain_importance_path,
    model_config_path,
]

for artifact_path in artifact_paths:
    assert artifact_path.is_file()
    assert artifact_path.stat().st_size > 0

artifact_file_summary = pd.DataFrame({
    "Artifact": [
        artifact_path.name
        for artifact_path in artifact_paths
    ],
    "Action": [
        (
            model_file_action
            if artifact_path
            == evaluation_model_path
            else artifact_actions[
                artifact_path.name
            ]
        )
        for artifact_path in artifact_paths
    ],
    "Size (KB)": [
        artifact_path.stat().st_size
        / 1024
        for artifact_path in artifact_paths
    ],
})

display(
    artifact_file_summary.style
    .hide(axis="index")
    .format({
        "Size (KB)": "{:,.2f}",
    })
)

print(
    "\nArtifact directory:",
    evaluation_artifact_dir,
)

print(
    "Model SHA-256:",
    model_sha256,
)

print(
    "Complete artifact bundle validated:",
    True,
)

In [ ]:
# Measuring notebook and system memory before cleanup.
cleanup_process = psutil.Process()

process_ram_before_gb = (
    cleanup_process.memory_info().rss
    / 1024**3
)

available_ram_before_gb = (
    psutil.virtual_memory().available
    / 1024**3
)

objects_to_release = []
estimated_tabular_array_gb = 0.0

# Identifying large Pandas and NumPy objects.
for object_name, object_value in list(
    globals().items()
):
    if object_name.startswith("_"):
        continue

    object_size_bytes = 0

    if isinstance(
        object_value,
        pd.DataFrame,
    ):
        object_size_bytes = int(
            object_value.memory_usage(
                index=True,
                deep=True,
            ).sum()
        )

    elif isinstance(
        object_value,
        pd.Series,
    ):
        object_size_bytes = int(
            object_value.memory_usage(
                index=True,
                deep=True,
            )
        )

    elif isinstance(
        object_value,
        np.ndarray,
    ):
        object_size_bytes = int(
            object_value.nbytes
        )

    # Releasing tabular or array objects of at least 25 MB.
    if object_size_bytes >= 25 * 1024**2:
        objects_to_release.append(
            object_name
        )

        estimated_tabular_array_gb += (
            object_size_bytes
            / 1024**3
        )

# Identifying every in-memory XGBoost matrix and model.
for object_name, object_value in list(
    globals().items()
):
    if object_name.startswith("_"):
        continue

    if isinstance(
        object_value,
        (xgb.DMatrix, xgb.Booster),
    ):
        if object_name not in objects_to_release:
            objects_to_release.append(
                object_name
            )

# Removing selected objects from the notebook namespace.
for object_name in objects_to_release:
    globals().pop(
        object_name,
        None,
    )

# Removing loop references that could keep the last object alive.
del object_name
del object_value

garbage_collected_objects = gc.collect()

process_ram_after_gb = (
    cleanup_process.memory_info().rss
    / 1024**3
)

available_ram_after_gb = (
    psutil.virtual_memory().available
    / 1024**3
)

process_ram_reduction_gb = (
    process_ram_before_gb
    - process_ram_after_gb
)

cleanup_summary = pd.DataFrame({
    "Measure": [
        "Notebook process RAM before cleanup",
        "Notebook process RAM after cleanup",
        "Process RAM reduction",
        "System available RAM before cleanup",
        "System available RAM after cleanup",
        "Estimated large Pandas/NumPy data released",
        "Released notebook objects",
        "Garbage-collected Python objects",
    ],
    "Value": [
        f"{process_ram_before_gb:.2f} GB",
        f"{process_ram_after_gb:.2f} GB",
        f"{process_ram_reduction_gb:.2f} GB",
        f"{available_ram_before_gb:.2f} GB",
        f"{available_ram_after_gb:.2f} GB",
        f"{estimated_tabular_array_gb:.2f} GB",
        f"{len(objects_to_release):,}",
        f"{garbage_collected_objects:,}",
    ],
})

display(
    cleanup_summary.style
    .hide(axis="index")
)

print("\nReleased objects:")

for released_object_name in sorted(
    objects_to_release
):
    print(
        "-",
        released_object_name,
    )

# Confirming that cleanup did not affect saved artifacts.
assert evaluation_model_path.is_file()
assert artifact_metadata_path.is_file()
assert feature_contract_path.is_file()
assert test_metrics_path.is_file()
assert gain_importance_path.is_file()
assert model_config_path.is_file()

print(
    "\nSaved artifact bundle still exists:",
    True,
)

print(
    "Memory cleanup completed:",
    True,
)